In [1]:
# Import necessary modules
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# Load dataset
df = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df.head()
 # pd.read_excel("filename.xlsx") 

,S/N,Longitude,Latitude,As,Cd,Cr,Cu,Hg,Ni,Pb,Zn,HI_Total_Adult,HI_Total_Child,ILCR _Total_Adult,ILCR _Total_Child
0,Mean S1,-1.240194,5.133018,10.833333,3.458333,96.723333,83.033333,3.158333,52.183333,56.280000,33.275000,1.961270,1.464479,0.001484,0.000181
1,Mean S2,-1.228260,5.166977,10.650000,3.413333,94.923333,83.336667,2.693333,51.650000,52.650000,31.923333,1.894783,1.412806,0.001457,0.000178
2,Mean S3,-1.240289,5.131882,10.816667,3.591667,96.556667,609.885000,3.483333,21.831667,49.035000,30.645000,2.462855,1.605323,0.001442,0.000122
3,Mean S4,-1.241133,5.131173,10.900000,3.471667,98.186667,2.353500,2.556667,0.126000,41.366667,30.645000,1.762027,1.334865,0.001437,0.000081
4,Mean S5,-1.241214,5.131136,7.100000,3.538333,97.605000,77.596667,3.463333,17.067000,35.810000,26.206667,1.632638,1.224794,0.001445,0.000106


In [2]:
# python

# """
# =============================================================================
# Statistical Analysis Pipeline — Cape Coast Landfill Heavy Metal Study
# =============================================================================
# Sections:
#   1. Data Loading & Preprocessing
#   2. Descriptive Statistics (mean ± SD, CV, min/max, percentiles)
#   3. Normality Tests (Shapiro-Wilk, Kolmogorov-Smirnov)
#   4. Non-Parametric Comparisons (Kruskal-Wallis, Mann-Whitney U)
#   5. Correlation Matrices (Pearson & Spearman)
#   6. Publication-quality figures saved to /outputs/
# =============================================================================
# """

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, kstest, kruskal, mannwhitneyu,
                         pearsonr, spearmanr, norm)
from itertools import combinations
from pathlib import Path
import textwrap

# ── Output directory ────────────────────────────────────────────────────────
# OUT = Path("/mnt/user-data/outputs")
# OUT = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
# OUT = Path('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
# OUT.mkdir(parents=True, exist_ok=True
# from pathlib import Path

OUT = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        10,
    "axes.titlesize":   11,
    "axes.labelsize":   10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "figure.dpi":       150,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
})

PALETTE   = "#2E4057"          # deep navy  — primary bars / markers
ACCENT    = "#E84855"          # crimson    — significance flags
NEUTRAL   = "#A8DADC"          # muted teal — secondary fills
LIGHT_BG  = "#F7F9FB"

METALS    = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]
RISK_VARS = ["HI_Total_Adult", "HI_Total_Child",
             "ILCR _Total_Adult", "ILCR _Total_Child"]
RISK_LABELS = {
    "HI_Total_Adult":    "HI Adult",
    "HI_Total_Child":    "HI Child",
    "ILCR _Total_Adult": "ILCR Adult",
    "ILCR _Total_Child": "ILCR Child",
}

# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("  CAPE COAST LANDFILL — STATISTICAL ANALYSIS PIPELINE")
print("=" * 70)

df_raw = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df_raw.columns = df_raw.columns.str.strip()

# Separate landfill sites from residential background
df_landfill    = df_raw[~df_raw["S/N"].str.contains("Residential", na=False)].copy()
df_residential = df_raw[ df_raw["S/N"].str.contains("Residential", na=False)].copy()

# Clean site labels
df_landfill["Site"] = df_landfill["S/N"].str.replace("Mean ", "", regex=False).str.strip()
df_residential["Site"] = "Residential"

print(f"\n  Landfill sites   : {len(df_landfill)}")
print(f"  Background sites : {len(df_residential)}")
print(f"  Metals analysed  : {METALS}")
print(f"  Risk variables   : {[RISK_LABELS[r] for r in RISK_VARS]}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 2. DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────────────────────
print("─" * 70)
print("  SECTION 2 — DESCRIPTIVE STATISTICS")
print("─" * 70)

def descriptive_table(df, cols, label="Landfill Sites"):
    rows = []
    for col in cols:
        s = df[col].dropna()
        rows.append({
            "Variable":  col,
            "n":         len(s),
            "Mean":      s.mean(),
            "SD":        s.std(ddof=1),
            "CV (%)":    s.std(ddof=1) / s.mean() * 100 if s.mean() != 0 else np.nan,
            "Min":       s.min(),
            "P25":       s.quantile(0.25),
            "Median":    s.median(),
            "P75":       s.quantile(0.75),
            "Max":       s.max(),
        })
    tbl = pd.DataFrame(rows).set_index("Variable")
    print(f"\n  [{label}]")
    print(tbl.round(4).to_string())
    return tbl

desc_metals = descriptive_table(df_landfill, METALS, "Metals — Landfill Sites")
desc_risk   = descriptive_table(df_landfill, ["HI_Total_Adult","HI_Total_Child",
                                               "ILCR _Total_Adult","ILCR _Total_Child"],
                                 "Risk Variables — Landfill Sites")

# ── Figure 1: Descriptive statistics panel (mean ± SD + CV bar) ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=LIGHT_BG)
fig.suptitle("Figure 1 — Descriptive Statistics: Heavy Metals at Cape Coast Landfill",
             fontsize=12, fontweight="bold", y=1.01)

# (a) Mean ± SD per metal
ax = axes[0]
means = desc_metals["Mean"]
sds   = desc_metals["SD"]
bars  = ax.bar(means.index, means.values, yerr=sds.values,
               color=PALETTE, alpha=0.85, capsize=5,
               error_kw={"elinewidth": 1.5, "ecolor": ACCENT})
ax.set_title("(a) Mean Concentration ± SD (mg kg⁻¹)", fontweight="bold")
ax.set_ylabel("Concentration (mg kg⁻¹)")
ax.set_xlabel("Heavy Metal")
ax.set_facecolor(LIGHT_BG)

# Annotate Cu as outlier
cu_idx = list(means.index).index("Cu")
ax.annotate("*Outlier\n(S3 hotspot)",
            xy=(cu_idx, means["Cu"]),
            xytext=(cu_idx + 0.8, means["Cu"] * 1.05),
            fontsize=8, color=ACCENT,
            arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1.2))

# (b) Coefficient of Variation
ax2 = axes[1]
cvs = desc_metals["CV (%)"].sort_values(ascending=False)
colors_cv = [ACCENT if c > 50 else PALETTE for c in cvs.values]
ax2.barh(cvs.index, cvs.values, color=colors_cv, alpha=0.85)
ax2.axvline(50, color=ACCENT, linestyle="--", linewidth=1.2,
            label="CV = 50% threshold")
ax2.set_title("(b) Coefficient of Variation (%)", fontweight="bold")
ax2.set_xlabel("CV (%)")
ax2.set_facecolor(LIGHT_BG)
ax2.legend(fontsize=8)

# Annotate bars
for bar, val in zip(ax2.patches, cvs.values):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
             f"{val:.1f}%", va="center", fontsize=8)

plt.tight_layout()
fig.savefig(OUT / "Fig1_Descriptive_Statistics.png")
plt.close()
print("\n  → Figure 1 saved: Fig1_Descriptive_Statistics.png")


# ─────────────────────────────────────────────────────────────────────────────
# 3. NORMALITY TESTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("  SECTION 3 — NORMALITY TESTS (Shapiro-Wilk & Kolmogorov-Smirnov)")
print("─" * 70)

ALL_VARS = METALS + ["HI_Total_Adult", "HI_Total_Child",
                     "ILCR _Total_Adult", "ILCR _Total_Child"]

normality_rows = []
for var in ALL_VARS:
    data = df_landfill[var].dropna().values
    if len(data) < 3:
        continue

    sw_stat, sw_p = shapiro(data)

    # KS test against fitted normal distribution
    mu, sigma = data.mean(), data.std(ddof=1)
    ks_stat, ks_p = kstest(data, "norm", args=(mu, sigma))

    sw_normal = "Normal" if sw_p > 0.05 else "Non-normal"
    ks_normal = "Normal" if ks_p > 0.05 else "Non-normal"

    normality_rows.append({
        "Variable":      var,
        "SW Statistic":  round(sw_stat, 4),
        "SW p-value":    round(sw_p, 4),
        "SW Decision":   sw_normal,
        "KS Statistic":  round(ks_stat, 4),
        "KS p-value":    round(ks_p, 4),
        "KS Decision":   ks_normal,
    })

norm_df = pd.DataFrame(normality_rows).set_index("Variable")
print("\n  Shapiro-Wilk & KS Normality Results (α = 0.05):")
print(norm_df.to_string())

# ── Figure 2: Normality visual — Q-Q plots for all metals ────────────────────
n_metals = len(METALS)
ncols = 4
nrows = int(np.ceil(n_metals / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 6), facecolor=LIGHT_BG)
axes = axes.flatten()
fig.suptitle("Figure 2 — Q-Q Plots: Normality Assessment of Heavy Metals",
             fontsize=12, fontweight="bold", y=1.01)

for i, metal in enumerate(METALS):
    ax = axes[i]
    data = df_landfill[metal].dropna().values
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist="norm")
    ax.plot(osm, osr, "o", color=PALETTE, markersize=6, alpha=0.8, label="Observed")
    ax.plot(osm, slope * np.array(osm) + intercept,
            "--", color=ACCENT, linewidth=1.5, label="Expected")
    ax.set_title(f"{metal}", fontweight="bold")
    ax.set_xlabel("Theoretical Quantiles", fontsize=8)
    ax.set_ylabel("Sample Quantiles", fontsize=8)
    ax.set_facecolor(LIGHT_BG)

    # Embed SW p-value
    sw_row = norm_df.loc[metal] if metal in norm_df.index else None
    if sw_row is not None:
        flag = "✓ Normal" if sw_row["SW Decision"] == "Normal" else "✗ Non-normal"
        color = "green" if sw_row["SW Decision"] == "Normal" else ACCENT
        ax.text(0.05, 0.92, f"SW p={sw_row['SW p-value']:.3f}\n{flag}",
                transform=ax.transAxes, fontsize=7.5, color=color,
                verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(OUT / "Fig2_QQ_Plots_Normality.png")
plt.close()
print("\n  → Figure 2 saved: Fig2_QQ_Plots_Normality.png")

# ── Figure 3: Normality summary heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=LIGHT_BG)
fig.suptitle("Figure 3 — Normality Test Summary (p-values)",
             fontsize=12, fontweight="bold")

pval_data = norm_df[["SW p-value", "KS p-value"]].T
pval_data.columns = [c.replace("ILCR _Total_", "ILCR\n").replace("HI_Total_", "HI\n") 
                     for c in pval_data.columns]

sns.heatmap(pval_data.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn", vmin=0, vmax=0.1,
            linewidths=0.5, ax=ax,
            annot_kws={"size": 9})
ax.set_title("Green = p > 0.05 (normal) | Red = p ≤ 0.05 (non-normal)",
             fontsize=9, style="italic")
ax.set_xlabel("Variable")
ax.set_ylabel("Test")

plt.tight_layout()
fig.savefig(OUT / "Fig3_Normality_Heatmap.png")
plt.close()
print("  → Figure 3 saved: Fig3_Normality_Heatmap.png")


# ─────────────────────────────────────────────────────────────────────────────
# 4. NON-PARAMETRIC COMPARISONS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("  SECTION 4 — NON-PARAMETRIC COMPARISONS")
print("─" * 70)

# ── 4a. Kruskal-Wallis: are metal concentrations significantly different ──────
#        across all 12 landfill sites?
print("\n  4a. Kruskal-Wallis Test — Metal concentrations across all sites")
print("  (H₀: concentrations are equal across all sampling sites)\n")

kw_rows = []
for metal in METALS:
    groups = [grp[metal].dropna().values
              for _, grp in df_landfill.groupby("Site")
              if len(grp[metal].dropna()) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = kruskal(*groups)
    decision = "Reject H₀ *" if p_val < 0.05 else "Fail to reject H₀"
    kw_rows.append({
        "Metal":       metal,
        "H-statistic": round(h_stat, 4),
        "p-value":     round(p_val, 4),
        "Decision (α=0.05)": decision,
    })

kw_df = pd.DataFrame(kw_rows).set_index("Metal")
print(kw_df.to_string())

# ── 4b. Kruskal-Wallis: HI and ILCR across sites ─────────────────────────────
print("\n  4b. Kruskal-Wallis Test — Risk variables across all sites\n")
kw_risk_rows = []
for var in ["HI_Total_Adult", "HI_Total_Child", "ILCR _Total_Adult", "ILCR _Total_Child"]:
    groups = [grp[var].dropna().values
              for _, grp in df_landfill.groupby("Site")
              if len(grp[var].dropna()) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = kruskal(*groups)
    kw_risk_rows.append({
        "Risk Variable": RISK_LABELS.get(var, var),
        "H-statistic":   round(h_stat, 4),
        "p-value":       round(p_val, 4),
        "Decision":      "Reject H₀ *" if p_val < 0.05 else "Fail to reject H₀",
    })
kw_risk_df = pd.DataFrame(kw_risk_rows).set_index("Risk Variable")
print(kw_risk_df.to_string())

# ── 4c. Mann-Whitney U: Landfill vs. Residential for each metal ───────────────
print("\n  4c. Mann-Whitney U Test — Landfill vs. Residential Background")
print("  (H₀: distributions are equal between landfill and residential)\n")

mw_rows = []
for metal in METALS:
    lf_vals = df_landfill[metal].dropna().values
    res_vals = df_residential[metal].dropna().values
    if len(res_vals) == 0 or len(lf_vals) == 0:
        continue
    u_stat, p_val = mannwhitneyu(lf_vals, res_vals, alternative="two-sided")
    median_lf  = np.median(lf_vals)
    median_res = np.median(res_vals)
    fold_change = median_lf / median_res if median_res != 0 else np.nan
    mw_rows.append({
        "Metal":              metal,
        "Median Landfill":    round(median_lf, 3),
        "Median Residential": round(median_res, 3),
        "Fold Change (LF/Res)": round(fold_change, 2),
        "U-statistic":        round(u_stat, 2),
        "p-value":            round(p_val, 4),
        "Significant":        "Yes *" if p_val < 0.05 else "No",
    })

mw_df = pd.DataFrame(mw_rows).set_index("Metal")
print(mw_df.to_string())

# ── Figure 4: Kruskal-Wallis H-statistics + p-value flags ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=LIGHT_BG)
fig.suptitle("Figure 4 — Kruskal-Wallis Test Results",
             fontsize=12, fontweight="bold")

# (a) H-statistics
ax = axes[0]
h_vals  = kw_df["H-statistic"]
sig     = kw_df["p-value"] < 0.05
bar_colors = [ACCENT if s else NEUTRAL for s in sig]
bars = ax.bar(h_vals.index, h_vals.values, color=bar_colors, alpha=0.9)
ax.set_title("(a) H-statistic per Metal\n(Red = p < 0.05 significant)", fontweight="bold")
ax.set_ylabel("Kruskal-Wallis H-statistic")
ax.set_xlabel("Heavy Metal")
ax.set_facecolor(LIGHT_BG)
for bar, p in zip(bars, kw_df["p-value"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"p={p:.3f}", ha="center", fontsize=8,
            color=ACCENT if p < 0.05 else PALETTE)

# (b) Mann-Whitney fold change
ax2 = axes[1]
fcs = mw_df["Fold Change (LF/Res)"]
sig_mw = mw_df["p-value"] < 0.05
colors_mw = [ACCENT if s else NEUTRAL for s in sig_mw]
ax2.bar(fcs.index, fcs.values, color=colors_mw, alpha=0.9)
ax2.axhline(1, color="black", linestyle="--", linewidth=1, label="Fold Change = 1")
ax2.set_title("(b) Mann-Whitney: Landfill / Residential\nMedian Fold Change", fontweight="bold")
ax2.set_ylabel("Fold Change (Landfill / Residential)")
ax2.set_xlabel("Heavy Metal")
ax2.set_facecolor(LIGHT_BG)
ax2.legend(fontsize=8)
for bar, p in zip(ax2.patches, mw_df["p-value"]):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5,
             "*" if p < 0.05 else "", ha="center", fontsize=12, color=ACCENT)

plt.tight_layout()
fig.savefig(OUT / "Fig4_NonParametric_Tests.png")
plt.close()
print("\n  → Figure 4 saved: Fig4_NonParametric_Tests.png")

# ── Figure 5: Mann-Whitney pairwise — metals across select site pairs ─────────
print("\n  4d. Mann-Whitney Pairwise — High-risk sites (S1–S3) vs. Low-risk (S8–S11)")
sites_high = ["S1", "S2", "S3"]
sites_low  = ["S8", "S9", "S10", "S11"]

high_df = df_landfill[df_landfill["Site"].isin(sites_high)]
low_df  = df_landfill[df_landfill["Site"].isin(sites_low)]

pairwise_rows = []
for metal in METALS:
    h_vals_data = high_df[metal].dropna().values
    l_vals_data = low_df[metal].dropna().values
    if len(h_vals_data) == 0 or len(l_vals_data) == 0:
        continue
    u_stat, p_val = mannwhitneyu(h_vals_data, l_vals_data, alternative="two-sided")
    pairwise_rows.append({
        "Metal":           metal,
        "Mean High (S1–S3)": round(h_vals_data.mean(), 3),
        "Mean Low (S8–S11)": round(l_vals_data.mean(), 3),
        "U-statistic":     round(u_stat, 2),
        "p-value":         round(p_val, 4),
        "Significant":     "Yes *" if p_val < 0.05 else "No",
    })
pw_df = pd.DataFrame(pairwise_rows).set_index("Metal")
print(pw_df.to_string())

# ── Figure 6: Box plots for metal concentrations by site group ────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor=LIGHT_BG)
axes = axes.flatten()
fig.suptitle("Figure 5 — Heavy Metal Concentrations Across Landfill Sites\n(Box Plots)",
             fontsize=12, fontweight="bold")

for i, metal in enumerate(METALS):
    ax = axes[i]
    site_data   = [df_landfill[df_landfill["Site"] == s][metal].dropna().values
                   for s in df_landfill["Site"].unique()]
    site_labels = df_landfill["Site"].unique().tolist()

    bp = ax.boxplot(site_data, patch_artist=True, notch=False,
                    medianprops={"color": ACCENT, "linewidth": 2},
                    whiskerprops={"linewidth": 1.2},
                    capprops={"linewidth": 1.2})
    for patch in bp["boxes"]:
        patch.set_facecolor(PALETTE)
        patch.set_alpha(0.6)

    # Overlay residential background as dashed line
    res_val = df_residential[metal].dropna().values
    if len(res_val) > 0:
        ax.axhline(res_val.mean(), color="green", linestyle="--",
                   linewidth=1.2, label="Residential background")

    ax.set_title(metal, fontweight="bold")
    ax.set_xticks(range(1, len(site_labels) + 1))
    ax.set_xticklabels(site_labels, rotation=55, fontsize=7)
    ax.set_ylabel("mg kg⁻¹", fontsize=8)
    ax.set_facecolor(LIGHT_BG)

    # KW p-value annotation
    kw_p = kw_df.loc[metal, "p-value"] if metal in kw_df.index else None
    if kw_p is not None:
        ax.set_xlabel(f"KW p = {kw_p:.3f}" +
                      (" *" if kw_p < 0.05 else ""), fontsize=8,
                      color=ACCENT if kw_p < 0.05 else "gray")

green_line = mpatches.Patch(color="green", label="Residential background mean")
fig.legend(handles=[green_line], loc="lower right", fontsize=9)
plt.tight_layout()
fig.savefig(OUT / "Fig5_Boxplots_by_Site.png")
plt.close()
print("  → Figure 5 saved: Fig5_Boxplots_by_Site.png")


# ─────────────────────────────────────────────────────────────────────────────
# 5. CORRELATION MATRICES (Pearson & Spearman)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("  SECTION 5 — CORRELATION MATRICES (Pearson & Spearman)")
print("─" * 70)

corr_vars = METALS + ["HI_Total_Adult", "HI_Total_Child",
                       "ILCR _Total_Adult", "ILCR _Total_Child"]
corr_labels = METALS + ["HI Adult", "HI Child", "ILCR Adult", "ILCR Child"]
data_corr = df_landfill[corr_vars].dropna()

# ── Compute matrices ──────────────────────────────────────────────────────────
n_vars = len(corr_vars)
pearson_r  = np.zeros((n_vars, n_vars))
pearson_p  = np.zeros((n_vars, n_vars))
spearman_r = np.zeros((n_vars, n_vars))
spearman_p = np.zeros((n_vars, n_vars))

for i, v1 in enumerate(corr_vars):
    for j, v2 in enumerate(corr_vars):
        x = data_corr[v1].values
        y = data_corr[v2].values
        pr, pp  = pearsonr(x, y)
        sr, sp  = spearmanr(x, y)
        pearson_r[i, j]  = pr
        pearson_p[i, j]  = pp
        spearman_r[i, j] = sr
        spearman_p[i, j] = sp

pearson_df  = pd.DataFrame(pearson_r,  index=corr_labels, columns=corr_labels)
spearman_df = pd.DataFrame(spearman_r, index=corr_labels, columns=corr_labels)
pearson_p_df  = pd.DataFrame(pearson_p,  index=corr_labels, columns=corr_labels)
spearman_p_df = pd.DataFrame(spearman_p, index=corr_labels, columns=corr_labels)

print("\n  Pearson Correlation Matrix (r):")
print(pearson_df.round(3).to_string())
print("\n  Spearman Correlation Matrix (ρ):")
print(spearman_df.round(3).to_string())

# Significant pairs (|r| > 0.6 and p < 0.05)
print("\n  Notable Correlations (|Pearson r| ≥ 0.6 and p < 0.05):")
for i, v1 in enumerate(corr_labels):
    for j, v2 in enumerate(corr_labels):
        if i >= j:
            continue
        if abs(pearson_r[i, j]) >= 0.6 and pearson_p[i, j] < 0.05:
            print(f"    {v1:12s} ↔ {v2:12s}  r = {pearson_r[i,j]:+.3f}  "
                  f"p = {pearson_p[i,j]:.4f}")

# ── Figure 6: Pearson correlation heatmap ────────────────────────────────────
def correlation_heatmap(r_matrix, p_matrix, labels, title, fname, cmap="RdBu_r"):
    fig, ax = plt.subplots(figsize=(11, 9), facecolor=LIGHT_BG)
    fig.suptitle(title, fontsize=12, fontweight="bold")

    mask = np.triu(np.ones_like(r_matrix, dtype=bool), k=1)
    r_lower = np.where(mask, np.nan, r_matrix)

    im = ax.imshow(r_lower, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Correlation coefficient")

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(labels)):
            if mask[i, j] or np.isnan(r_lower[i, j]):
                continue
            r_val = r_lower[i, j]
            p_val = p_matrix[i, j]
            sig_star = "***" if p_val < 0.001 else ("**" if p_val < 0.01
                       else ("*" if p_val < 0.05 else ""))
            text_color = "white" if abs(r_val) > 0.6 else "black"
            ax.text(j, i, f"{r_val:.2f}\n{sig_star}",
                    ha="center", va="center",
                    fontsize=7.5, color=text_color, fontweight="bold")

    sig_note = mpatches.Patch(color="none",
        label="Significance: * p<0.05  ** p<0.01  *** p<0.001")
    ax.legend(handles=[sig_note], loc="upper right",
              bbox_to_anchor=(1.0, 1.10), fontsize=8, frameon=False)
    plt.tight_layout()
    fig.savefig(OUT / fname)
    plt.close()
    print(f"  → {fname} saved.")

correlation_heatmap(pearson_r, pearson_p_df.values, corr_labels,
                    "Figure 6 — Pearson Correlation Matrix\n(Lower Triangle | * p<0.05)",
                    "Fig6_Pearson_Correlation.png")

correlation_heatmap(spearman_r, spearman_p_df.values, corr_labels,
                    "Figure 7 — Spearman Correlation Matrix\n(Lower Triangle | * p<0.05)",
                    "Fig7_Spearman_Correlation.png", cmap="PuOr")

# ── Figure 8: Pearson vs Spearman scatter comparison ─────────────────────────
pairs   = list(combinations(range(n_vars), 2))
pr_vals = [pearson_r[i, j]  for i, j in pairs]
sr_vals = [spearman_r[i, j] for i, j in pairs]
pp_vals = [pearson_p[i, j]  for i, j in pairs]

fig, ax = plt.subplots(figsize=(7, 6), facecolor=LIGHT_BG)
fig.suptitle("Figure 8 — Pearson vs. Spearman: Agreement Between Methods",
             fontsize=12, fontweight="bold")

sig_mask   = np.array(pp_vals) < 0.05
not_sig    = ~sig_mask

ax.scatter(np.array(pr_vals)[not_sig], np.array(sr_vals)[not_sig],
           color=NEUTRAL, alpha=0.7, s=40, label="p ≥ 0.05")
ax.scatter(np.array(pr_vals)[sig_mask], np.array(sr_vals)[sig_mask],
           color=ACCENT, alpha=0.85, s=55, label="p < 0.05", zorder=5)

# Reference line
lims = [-1.05, 1.05]
ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5, label="Perfect agreement")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Pearson r", fontsize=10)
ax.set_ylabel("Spearman ρ", fontsize=10)
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT_BG)

# Pearson-Spearman agreement stat
r_agree, p_agree = pearsonr(pr_vals, sr_vals)
ax.text(0.05, 0.95,
        f"Pearson–Spearman agreement:\nr = {r_agree:.3f}, p = {p_agree:.4f}",
        transform=ax.transAxes, fontsize=9,
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8))

plt.tight_layout()
fig.savefig(OUT / "Fig8_Pearson_vs_Spearman.png")
plt.close()
print("  → Figure 8 saved: Fig8_Pearson_vs_Spearman.png")


# ─────────────────────────────────────────────────────────────────────────────
# 6. EXPORT RESULTS TO EXCEL
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("  SECTION 6 — EXPORTING RESULTS TO EXCEL")
print("─" * 70)

out_xlsx = OUT / "Statistical_Analysis_Results.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    desc_metals.round(4).to_excel(writer, sheet_name="Descriptive_Metals")
    desc_risk.round(6).to_excel(writer, sheet_name="Descriptive_Risk")
    norm_df.to_excel(writer, sheet_name="Normality_Tests")
    kw_df.to_excel(writer, sheet_name="KruskalWallis_Metals")
    kw_risk_df.to_excel(writer, sheet_name="KruskalWallis_Risk")
    mw_df.to_excel(writer, sheet_name="MannWhitney_LF_vs_Res")
    pw_df.to_excel(writer, sheet_name="MannWhitney_HighvsLow")
    pearson_df.round(4).to_excel(writer, sheet_name="Pearson_r")
    pearson_p_df.round(4).to_excel(writer, sheet_name="Pearson_p")
    spearman_df.round(4).to_excel(writer, sheet_name="Spearman_rho")
    spearman_p_df.round(4).to_excel(writer, sheet_name="Spearman_p")

print(f"\n  → Results workbook saved: Statistical_Analysis_Results.xlsx")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
# print("\n" + "=" * 70)
# print("  PIPELINE COMPLETE")
# print("=" * 70)
# print("""
#   Outputs generated:
#   ┌──────────────────────────────────────────────────────────┐
#   │ Fig1_Descriptive_Statistics.png  — Mean±SD + CV bars     │
#   │ Fig2_QQ_Plots_Normality.png      — Q-Q plots all metals  │
#   │ Fig3_Normality_Heatmap.png       — SW & KS p-value grid  │
#   │ Fig4_NonParametric_Tests.png     — KW + Mann-Whitney      │
#   │ Fig5_Boxplots_by_Site.png        — Box plots per site     │
#   │ Fig6_Pearson_Correlation.png     — Pearson heatmap        │
#   │ Fig7_Spearman_Correlation.png    — Spearman heatmap       │
#   │ Fig8_Pearson_vs_Spearman.png     — Method agreement plot  │
#   │ Statistical_Analysis_Results.xlsx — All tables (11 tabs) │
#   └──────────────────────────────────────────────────────────┘
# """)

  CAPE COAST LANDFILL — STATISTICAL ANALYSIS PIPELINE

  Landfill sites   : 12
  Background sites : 1
  Metals analysed  : ['As', 'Cd', 'Cr', 'Cu', 'Hg', 'Ni', 'Pb', 'Zn']
  Risk variables   : ['HI Adult', 'HI Child', 'ILCR Adult', 'ILCR Child']

──────────────────────────────────────────────────────────────────────
  SECTION 2 — DESCRIPTIVE STATISTICS
──────────────────────────────────────────────────────────────────────

  [Metals — Landfill Sites]
           n      Mean        SD    CV (%)      Min      P25   Median      P75       Max
Variable                                                                                
As        12    7.0222    2.8700   40.8698   4.5200   4.7667   5.2042  10.6917   10.9000
Cd        12    3.6317    0.6396   17.6128   2.9067   3.4296   3.4842   3.5592    5.5567
Cr        12   90.8696    5.5204    6.0750  84.2167  85.6218  89.5183  96.5983   98.1867
Cu        12  117.0913  156.7702  133.8872   2.3535  77.7029  78.5025  80.5346  609.8850
Hg        1

In [3]:
# python

# """
# =============================================================================
# Ecological Indices Pipeline — Cape Coast Landfill Heavy Metal Study
# =============================================================================
# Indices calculated and appended to the original dataframe:

#   1. Enrichment Factor          (EF)      — per metal
#   2. Geo-accumulation Index     (Igeo)    — per metal
#   3. Contamination Factor       (CF)      — per metal
#   4. Pollution Load Index       (PLI)     — per site (combined all metals)
#   5. Individual Ecological Risk (Eᵣⁱ)    — per metal
#   6. Total Ecological Risk Index(ERI)     — per site (combined all metals)

# Background values: World Average Soil (Taylor & McLennan 1995;
#                    Kabata-Pendias 2011; Rudnick & Gao 2003)
# Reference element for EF: Fe (world average crustal Fe used)
# =============================================================================
# """

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

# ── Output directory ─────────────────────────────────────────────────────────
# OUT = Path("/mnt/user-data/outputs")
# OUT.mkdir(parents=True, exist_ok=True)
OUT = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

PALETTE  = "#2E4057"   # deep navy
ACCENT   = "#E84855"   # crimson
LIGHT_BG = "#F7F9FB"
GREENS   = ["#d4edda", "#82c996", "#2e8b57", "#1a5c38"]  # risk gradient (low→high)

METALS = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]

# =============================================================================
# REFERENCE VALUES
# =============================================================================
# ── World Average Soil Background (mg kg⁻¹) ──────────────────────────────────
# Sources:
#   As  — Kabata-Pendias (2011): 6.0  mg/kg
#   Cd  — Kabata-Pendias (2011): 0.41 mg/kg
#   Cr  — Kabata-Pendias (2011): 54.0 mg/kg
#   Cu  — Kabata-Pendias (2011): 28.0 mg/kg
#   Hg  — Kabata-Pendias (2011): 0.06 mg/kg
#   Ni  — Kabata-Pendias (2011): 29.0 mg/kg
#   Pb  — Kabata-Pendias (2011): 27.0 mg/kg
#   Zn  — Kabata-Pendias (2011): 70.0 mg/kg
#   Fe  — used as EF reference element (Taylor & McLennan 1995): 43200 mg/kg
# ─────────────────────────────────────────────────────────────────────────────
BACKGROUND = {
    "As": 6.0,
    "Cd": 0.41,
    "Cr": 54.0,
    "Cu": 28.0,
    "Hg": 0.06,
    "Ni": 29.0,
    "Pb": 27.0,
    "Zn": 70.0,
}
BG_FE = 43_200.0   # world average Fe (mg/kg) — EF reference element

# ── Toxic Response Factors (Tᵣⁱ) — Hakanson (1980) ───────────────────────────
# Note: Hg toxicity factor = 40 (highest among common metals)
TOXIC_FACTOR = {
    "As": 10,
    "Cd": 30,
    "Cr":  2,
    "Cu":  5,
    "Hg": 40,
    "Ni":  5,
    "Pb":  5,
    "Zn":  1,
}

# =============================================================================
# CLASSIFICATION FUNCTIONS
# =============================================================================

def classify_ef(ef):
    """Enrichment Factor classification (Sutherland 2000)."""
    if ef < 1:       return "No enrichment"
    elif ef < 2:     return "Slight enrichment"
    elif ef < 5:     return "Moderate enrichment"
    elif ef < 20:    return "Significant enrichment"
    elif ef < 40:    return "Very high enrichment"
    else:            return "Extremely high enrichment"

def classify_igeo(igeo):
    """Geo-accumulation Index classification (Müller 1969)."""
    if igeo < 0:     return "Class 0: Uncontaminated"
    elif igeo < 1:   return "Class 1: Uncontaminated–Moderate"
    elif igeo < 2:   return "Class 2: Moderately contaminated"
    elif igeo < 3:   return "Class 3: Moderate–Heavy"
    elif igeo < 4:   return "Class 4: Heavily contaminated"
    elif igeo < 5:   return "Class 5: Heavy–Extreme"
    else:            return "Class 6: Extremely contaminated"

def classify_cf(cf):
    """Contamination Factor classification (Hakanson 1980)."""
    if cf < 1:       return "Low"
    elif cf < 3:     return "Moderate"
    elif cf < 6:     return "Considerable"
    else:            return "Very high"

def classify_eri(er):
    """Individual Ecological Risk classification (Hakanson 1980)."""
    if er < 40:      return "Low"
    elif er < 80:    return "Moderate"
    elif er < 160:   return "Considerable"
    elif er < 320:   return "High"
    else:            return "Very high"

def classify_total_eri(eri):
    """Total ERI classification (Hakanson 1980)."""
    if eri < 150:    return "Low"
    elif eri < 300:  return "Moderate"
    elif eri < 600:  return "Considerable"
    else:            return "Very high"

def classify_pli(pli):
    """PLI classification (Tomlinson et al. 1980)."""
    if pli <= 1:     return "Baseline — no pollution"
    elif pli <= 2:   return "Moderately polluted"
    elif pli <= 3:   return "Progressively deteriorating"
    else:            return "Strongly deteriorating"

# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("=" * 70)
print("  ECOLOGICAL INDICES PIPELINE — CAPE COAST LANDFILL STUDY")
print("=" * 70)

# df_raw = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
# df_raw.columns = df_raw.columns.str.strip()
df = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df.columns = df.columns.str.strip()
df["Site"] = df["S/N"].str.replace("Mean ", "", regex=False).str.strip()

print(f"\n  Loaded {len(df)} sites: {df['Site'].tolist()}")
print(f"  Metals analysed : {METALS}")
print(f"\n  World Average Background Concentrations (mg kg⁻¹):")
for m, v in BACKGROUND.items():
    print(f"    {m:4s} : {v:>8.2f}   |   Toxic Factor (Tᵣⁱ) = {TOXIC_FACTOR[m]}")
print(f"    Fe   : {BG_FE:>8.1f}   (EF reference element)")

# =============================================================================
# 2. CALCULATE INDICES
# =============================================================================
print("\n" + "─" * 70)
print("  CALCULATING INDICES ...")
print("─" * 70)

# ── 2a. Enrichment Factor (EF) ────────────────────────────────────────────────
# EF = (Cᵢ / C_Fe)_sample  ÷  (Bᵢ / B_Fe)_background
# Requires Fe column. If absent, we note it and use residential as proxy.
# Here we use world average Fe as the background normaliser since Fe is
# not in the dataset columns.
print("\n  EF: Using world average crustal Fe as denominator reference.")
print("  Formula: EF = (Cᵢ_sample / C_Fe_bg) ÷ (Bᵢ_bg / B_Fe_bg)")
print("         = (Cᵢ_sample / Bᵢ_bg) × (B_Fe_bg / C_Fe_bg)")
print("  Since dataset has no Fe column, EF reduces to CF × (B_Fe_bg / BG_Fe)")
print("  — equivalent to normalising CF by the Fe crustal ratio (= 1.0 at bg).")
print("  Note: For publication, measuring Fe in samples is strongly recommended.")

for metal in METALS:
    bg    = BACKGROUND[metal]
    # EF = (C_metal/C_Fe_sample) / (B_metal/B_Fe_bg)
    # Without sample Fe, we approximate:  C_Fe_sample ≈ BG_FE (conservative)
    # This gives EF ≈ CF, but the formula and column are correctly labelled.
    df[f"EF_{metal}"]       = (df[metal] / BG_FE) / (bg / BG_FE)
    df[f"EF_{metal}_Class"] = df[f"EF_{metal}"].apply(classify_ef)

# ── 2b. Geo-accumulation Index (Igeo) ─────────────────────────────────────────
# Igeo = log₂( Cᵢ / (1.5 × Bᵢ) )
# Factor 1.5 accounts for lithogenic variation (Müller 1969)
for metal in METALS:
    bg = BACKGROUND[metal]
    df[f"Igeo_{metal}"]       = np.log2(df[metal] / (1.5 * bg))
    df[f"Igeo_{metal}_Class"] = df[f"Igeo_{metal}"].apply(classify_igeo)

# ── 2c. Contamination Factor (CF) ─────────────────────────────────────────────
# CF = Cᵢ_sample / Bᵢ_background
for metal in METALS:
    bg = BACKGROUND[metal]
    df[f"CF_{metal}"]       = df[metal] / bg
    df[f"CF_{metal}_Class"] = df[f"CF_{metal}"].apply(classify_cf)

# ── 2d. Pollution Load Index (PLI) ────────────────────────────────────────────
# PLI = (CF₁ × CF₂ × … × CFₙ)^(1/n)
# Geometric mean of all CFs per site
cf_cols = [f"CF_{m}" for m in METALS]
df["PLI"]       = df[cf_cols].prod(axis=1) ** (1 / len(METALS))
df["PLI_Class"] = df["PLI"].apply(classify_pli)

# ── 2e. Individual Ecological Risk Index (Eᵣⁱ) ────────────────────────────────
# Eᵣⁱ = Tᵣⁱ × CF  =  Tᵣⁱ × (Cᵢ / Bᵢ)
for metal in METALS:
    tr = TOXIC_FACTOR[metal]
    df[f"Er_{metal}"]       = tr * df[f"CF_{metal}"]
    df[f"Er_{metal}_Class"] = df[f"Er_{metal}"].apply(classify_eri)

# ── 2f. Total Ecological Risk Index (ERI) ─────────────────────────────────────
# ERI = Σ Eᵣⁱ  (sum of all individual risk indices per site)
er_cols = [f"Er_{m}" for m in METALS]
df["ERI"]       = df[er_cols].sum(axis=1)
df["ERI_Class"] = df["ERI"].apply(classify_total_eri)

print("\n  ✓ All indices calculated successfully.")

# =============================================================================
# 3. PRINT SUMMARY TABLES
# =============================================================================
print("\n" + "─" * 70)
print("  INDEX SUMMARY TABLES")
print("─" * 70)

index_groups = {
    "Enrichment Factor (EF)":            [f"EF_{m}"   for m in METALS],
    "Geo-accumulation Index (Igeo)":     [f"Igeo_{m}" for m in METALS],
    "Contamination Factor (CF)":         [f"CF_{m}"   for m in METALS],
    "Individual Eco Risk (Eᵣⁱ)":        [f"Er_{m}"   for m in METALS],
}

for title, cols in index_groups.items():
    print(f"\n  [{title}]")
    tbl = df[["Site"] + cols].set_index("Site")
    tbl.columns = METALS
    print(tbl.round(3).to_string())

print("\n  [PLI and ERI per Site]")
summary_cols = ["Site", "PLI", "PLI_Class", "ERI", "ERI_Class"]
print(df[summary_cols].set_index("Site").to_string())

# =============================================================================
# 4. FIGURES
# =============================================================================
print("\n" + "─" * 70)
print("  GENERATING FIGURES ...")
print("─" * 70)

sites       = df["Site"].tolist()
n_sites     = len(sites)
site_labels = [s.replace("Mean ", "") for s in sites]

# ── Colour maps for risk levels ───────────────────────────────────────────────
EF_LEVELS   = ["No enrichment", "Slight enrichment", "Moderate enrichment",
               "Significant enrichment", "Very high enrichment",
               "Extremely high enrichment"]
CF_LEVELS   = ["Low", "Moderate", "Considerable", "Very high"]
ER_LEVELS   = ["Low", "Moderate", "Considerable", "High", "Very high"]
ERI_LEVELS  = ["Low", "Moderate", "Considerable", "Very high"]
PLI_LEVELS  = ["Baseline — no pollution", "Moderately polluted",
               "Progressively deteriorating", "Strongly deteriorating"]

RISK_COLORS = {
    "No enrichment": "#2e8b57",      "Slight enrichment":       "#82c996",
    "Moderate enrichment": "#f0e442","Significant enrichment":  "#f5a623",
    "Very high enrichment": "#E84855","Extremely high enrichment":"#8B0000",
    "Low":             "#2e8b57",    "Moderate":                "#f0e442",
    "Considerable":    "#f5a623",    "High":                    "#E84855",
    "Very high":       "#8B0000",
    "Baseline — no pollution": "#2e8b57",
    "Moderately polluted":     "#f0e442",
    "Progressively deteriorating": "#f5a623",
    "Strongly deteriorating":  "#E84855",
    "Class 0: Uncontaminated":         "#2e8b57",
    "Class 1: Uncontaminated–Moderate":"#82c996",
    "Class 2: Moderately contaminated":"#f0e442",
    "Class 3: Moderate–Heavy":         "#f5a623",
    "Class 4: Heavily contaminated":   "#E84855",
    "Class 5: Heavy–Extreme":          "#c0392b",
    "Class 6: Extremely contaminated": "#8B0000",
}

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — EF heatmap (sites × metals)
# ─────────────────────────────────────────────────────────────────────────────
ef_matrix = df[[f"EF_{m}" for m in METALS]].values
ef_df     = pd.DataFrame(ef_matrix, index=site_labels, columns=METALS)

fig, ax = plt.subplots(figsize=(12, 7), facecolor=LIGHT_BG)
fig.suptitle("Figure 1 — Enrichment Factor (EF) Heatmap\n"
             "EF < 1: no enrichment | 1–2: slight | 2–5: moderate | "
             "5–20: significant | >20: very high",
             fontsize=11, fontweight="bold")

im = ax.imshow(ef_df.values, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=20)
plt.colorbar(im, ax=ax, label="EF value", shrink=0.8)
ax.set_xticks(range(len(METALS)))
ax.set_yticks(range(n_sites))
ax.set_xticklabels(METALS, fontsize=10, fontweight="bold")
ax.set_yticklabels(site_labels, fontsize=9)
ax.set_xlabel("Heavy Metal")
ax.set_ylabel("Sampling Site")

for i in range(n_sites):
    for j, metal in enumerate(METALS):
        val = ef_df.iloc[i, j]
        text_col = "white" if val > 10 else "black"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                fontsize=8, color=text_col, fontweight="bold")

# Threshold lines
for x in np.arange(-0.5, len(METALS), 1):
    ax.axvline(x, color="white", linewidth=0.5)
for y in np.arange(-0.5, n_sites, 1):
    ax.axhline(y, color="white", linewidth=0.5)

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig1_EF_Heatmap.png")
plt.close()
print("  → EcoIdx_Fig1_EF_Heatmap.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — Igeo heatmap + class annotation
# ─────────────────────────────────────────────────────────────────────────────
igeo_matrix = df[[f"Igeo_{m}" for m in METALS]].values
igeo_df     = pd.DataFrame(igeo_matrix, index=site_labels, columns=METALS)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=LIGHT_BG)
fig.suptitle("Figure 2 — Geo-accumulation Index (Igeo)\n"
             "Class 0: <0 uncontaminated | 1: 0–1 | 2: 1–2 | "
             "3: 2–3 | 4: 3–4 | 5: 4–5 | 6: >5 extremely contaminated",
             fontsize=11, fontweight="bold")

# (a) Value heatmap
ax = axes[0]
im = ax.imshow(igeo_df.values, cmap="RdYlGn_r", aspect="auto", vmin=-2, vmax=6)
plt.colorbar(im, ax=ax, label="Igeo value", shrink=0.8)
ax.set_title("(a) Igeo Values", fontweight="bold")
ax.set_xticks(range(len(METALS)));    ax.set_xticklabels(METALS, fontweight="bold")
ax.set_yticks(range(n_sites));        ax.set_yticklabels(site_labels)
for i in range(n_sites):
    for j in range(len(METALS)):
        val = igeo_df.iloc[i, j]
        tc  = "white" if val > 3 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=tc)

# (b) Class number heatmap
ax2 = axes[1]
class_matrix = df[[f"Igeo_{m}_Class" for m in METALS]].applymap(
    lambda x: int(x.split(":")[0].replace("Class ", ""))
).values
im2 = ax2.imshow(class_matrix, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=6)
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8, label="Igeo Class (0–6)")
cbar2.set_ticks(range(7))
ax2.set_title("(b) Igeo Class", fontweight="bold")
ax2.set_xticks(range(len(METALS)));    ax2.set_xticklabels(METALS, fontweight="bold")
ax2.set_yticks(range(n_sites));        ax2.set_yticklabels(site_labels)
for i in range(n_sites):
    for j in range(len(METALS)):
        ax2.text(j, i, str(class_matrix[i, j]),
                 ha="center", va="center", fontsize=10,
                 color="white" if class_matrix[i, j] > 3 else "black",
                 fontweight="bold")

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig2_Igeo_Heatmap.png")
plt.close()
print("  → EcoIdx_Fig2_Igeo_Heatmap.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3 — CF grouped bar chart (all metals, all sites)
# ─────────────────────────────────────────────────────────────────────────────
cf_df = df[[f"CF_{m}" for m in METALS]].copy()
cf_df.index = site_labels
cf_df.columns = METALS

fig, ax = plt.subplots(figsize=(14, 6), facecolor=LIGHT_BG)
fig.suptitle("Figure 3 — Contamination Factor (CF) by Site and Metal\n"
             "CF < 1: Low | 1–3: Moderate | 3–6: Considerable | >6: Very high",
             fontsize=11, fontweight="bold")

x      = np.arange(len(METALS))
width  = 0.9 / n_sites
colors = plt.cm.tab20(np.linspace(0, 1, n_sites))

for i, (site, row) in enumerate(cf_df.iterrows()):
    offset = (i - n_sites / 2 + 0.5) * width
    bars   = ax.bar(x + offset, row.values, width * 0.9,
                    label=site, color=colors[i], alpha=0.85)

# Threshold lines
for thresh, label, style in [(1, "CF=1 Low", ":"),
                              (3, "CF=3 Considerable", "--"),
                              (6, "CF=6 Very High", "-.")]:
    ax.axhline(thresh, color="black", linestyle=style, linewidth=1.0,
               alpha=0.6, label=label)

ax.set_xticks(x)
ax.set_xticklabels(METALS, fontsize=10, fontweight="bold")
ax.set_ylabel("Contamination Factor (CF)")
ax.set_xlabel("Heavy Metal")
ax.set_facecolor(LIGHT_BG)
ax.legend(loc="upper right", fontsize=7, ncol=3, framealpha=0.8)

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig3_CF_BarChart.png")
plt.close()
print("  → EcoIdx_Fig3_CF_BarChart.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4 — PLI per site with risk zone bands
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5), facecolor=LIGHT_BG)
fig.suptitle("Figure 4 — Pollution Load Index (PLI) per Sampling Site\n"
             "PLI ≤ 1: Baseline | 1–2: Moderate | 2–3: Progressive | >3: Strong deterioration",
             fontsize=11, fontweight="bold")

pli_vals = df["PLI"].values
bar_cols  = []
for v in pli_vals:
    if v <= 1:   bar_cols.append("#2e8b57")
    elif v <= 2: bar_cols.append("#f0e442")
    elif v <= 3: bar_cols.append("#f5a623")
    else:        bar_cols.append("#E84855")

bars = ax.bar(site_labels, pli_vals, color=bar_cols, alpha=0.9,
              edgecolor="white", linewidth=0.8)

# Risk zone bands
ax.axhspan(0,   1,  alpha=0.07, color="#2e8b57", zorder=0)
ax.axhspan(1,   2,  alpha=0.07, color="#f0e442", zorder=0)
ax.axhspan(2,   3,  alpha=0.07, color="#f5a623", zorder=0)
ax.axhspan(3,  20,  alpha=0.07, color="#E84855", zorder=0)

# Threshold lines
for y, lbl in [(1, "Baseline (1)"), (2, "Moderate (2)"), (3, "Progressive (3)")]:
    ax.axhline(y, color="grey", linestyle="--", linewidth=1.0, alpha=0.8)
    ax.text(n_sites - 0.5, y + 0.05, lbl, fontsize=8, color="grey", ha="right")

# Value labels on bars
for bar, val, cls in zip(bars, pli_vals, df["PLI_Class"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", fontsize=8, fontweight="bold")

ax.set_ylabel("PLI")
ax.set_xlabel("Sampling Site")
ax.set_xticklabels(site_labels, rotation=45, ha="right")
ax.set_facecolor(LIGHT_BG)
ax.set_ylim(0, max(pli_vals) * 1.25)

legend_patches = [
    mpatches.Patch(color="#2e8b57", label="Baseline (≤1)"),
    mpatches.Patch(color="#f0e442", label="Moderate (1–2)"),
    mpatches.Patch(color="#f5a623", label="Progressive (2–3)"),
    mpatches.Patch(color="#E84855", label="Strong (>3)"),
]
ax.legend(handles=legend_patches, fontsize=8, loc="upper right")
plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig4_PLI_BarChart.png")
plt.close()
print("  → EcoIdx_Fig4_PLI_BarChart.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 5 — Individual Ecological Risk (Eᵣⁱ) stacked bar
# ─────────────────────────────────────────────────────────────────────────────
# er_matrix = df[[f"Er_{m}" for m in METALS]].copy()
# er_matrix.index = site_labels
er_matrix = df[[f"Er_{m}" for m in METALS]].copy()
er_matrix.columns = METALS
er_matrix.index = site_labels

metal_colors = plt.cm.Set2(np.linspace(0, 1, len(METALS)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=LIGHT_BG)
fig.suptitle("Figure 5 — Individual Ecological Risk Index (Eᵣⁱ) per Metal and Site",
             fontsize=11, fontweight="bold")

# (a) Stacked bar — contribution of each metal to total per site
ax = axes[0]
bottom = np.zeros(n_sites)
for j, metal in enumerate(METALS):
    vals = er_matrix[metal].values
    ax.bar(site_labels, vals, bottom=bottom,
           label=metal, color=metal_colors[j], alpha=0.9)
    bottom += vals

ax.axhline(40,  color="green",  linestyle="--", linewidth=1.2,
           label="Low/Moderate threshold (40)")
ax.axhline(80,  color="orange", linestyle="--", linewidth=1.2,
           label="Moderate/Considerable threshold (80)")
ax.axhline(160, color="red",    linestyle="--", linewidth=1.2,
           label="Considerable/High threshold (160)")
ax.set_title("(a) Stacked Eᵣⁱ Contributions", fontweight="bold")
ax.set_ylabel("Eᵣⁱ")
ax.set_xlabel("Site")
ax.set_xticklabels(site_labels, rotation=45, ha="right")
ax.set_facecolor(LIGHT_BG)
ax.legend(fontsize=7, loc="upper right", ncol=2)

# (b) Heatmap of individual Eᵣⁱ values
ax2 = axes[1]
im = ax2.imshow(er_matrix.T.values, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax2, label="Eᵣⁱ value", shrink=0.8)
ax2.set_title("(b) Eᵣⁱ Heatmap (metals × sites)", fontweight="bold")
ax2.set_xticks(range(n_sites));     ax2.set_xticklabels(site_labels, rotation=45, ha="right", fontsize=8)
ax2.set_yticks(range(len(METALS))); ax2.set_yticklabels(METALS, fontweight="bold")
for i, metal in enumerate(METALS):
    for j in range(n_sites):
        val = er_matrix.iloc[j, i]
        tc  = "white" if val > 80 else "black"
        ax2.text(j, i, f"{val:.1f}", ha="center", va="center",
                 fontsize=7.5, color=tc)

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig5_Er_Individual.png")
plt.close()
print("  → EcoIdx_Fig5_Er_Individual.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 6 — Total ERI per site with classification
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5), facecolor=LIGHT_BG)
fig.suptitle("Figure 6 — Total Ecological Risk Index (ERI) per Site\n"
             "ERI < 150: Low | 150–300: Moderate | 300–600: Considerable | >600: Very High",
             fontsize=11, fontweight="bold")

eri_vals  = df["ERI"].values
eri_cols  = []
for v in eri_vals:
    if v < 150:   eri_cols.append("#2e8b57")
    elif v < 300: eri_cols.append("#f0e442")
    elif v < 600: eri_cols.append("#f5a623")
    else:         eri_cols.append("#E84855")

bars = ax.bar(site_labels, eri_vals, color=eri_cols, alpha=0.9,
              edgecolor="white", linewidth=0.8)

ax.axhspan(0,   150, alpha=0.06, color="#2e8b57", zorder=0)
ax.axhspan(150, 300, alpha=0.06, color="#f0e442", zorder=0)
ax.axhspan(300, 600, alpha=0.06, color="#f5a623", zorder=0)
ax.axhspan(600, max(eri_vals) * 1.3 + 10, alpha=0.06, color="#E84855", zorder=0)

for y, lbl in [(150, "Low/Moderate (150)"),
               (300, "Moderate/Considerable (300)"),
               (600, "Considerable/Very High (600)")]:
    ax.axhline(y, color="grey", linestyle="--", linewidth=1.0, alpha=0.8)
    ax.text(n_sites - 0.5, y + 5, lbl, fontsize=8, color="grey", ha="right")

for bar, val, cls in zip(bars, eri_vals, df["ERI_Class"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            f"{val:.1f}\n({cls})", ha="center", fontsize=7.5, fontweight="bold",
            multialignment="center")

ax.set_ylabel("Total ERI")
ax.set_xlabel("Sampling Site")
ax.set_xticklabels(site_labels, rotation=45, ha="right")
ax.set_facecolor(LIGHT_BG)
ax.set_ylim(0, max(eri_vals) * 1.40)

legend_patches = [
    mpatches.Patch(color="#2e8b57", label="Low (<150)"),
    mpatches.Patch(color="#f0e442", label="Moderate (150–300)"),
    mpatches.Patch(color="#f5a623", label="Considerable (300–600)"),
    mpatches.Patch(color="#E84855", label="Very High (>600)"),
]
ax.legend(handles=legend_patches, fontsize=8, loc="upper left")
plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig6_ERI_Total.png")
plt.close()
print("  → EcoIdx_Fig6_ERI_Total.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 7 — Combined radar chart: all indices per site (mean across metals)
# ─────────────────────────────────────────────────────────────────────────────
# Normalise each index to 0–1 scale for radar comparison
radar_df = pd.DataFrame({
    "Site":   site_labels,
    "EF":     df[[f"EF_{m}"   for m in METALS]].mean(axis=1).values,
    "Igeo":   df[[f"Igeo_{m}" for m in METALS]].mean(axis=1).values,
    "CF":     df[[f"CF_{m}"   for m in METALS]].mean(axis=1).values,
    "PLI":    df["PLI"].values,
    "ER_mean":df[[f"Er_{m}"   for m in METALS]].mean(axis=1).values,
    "ERI":    df["ERI"].values,
}).set_index("Site")

# Normalise 0–1
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min())
categories = radar_norm.columns.tolist()
N          = len(categories)
angles     = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles    += angles[:1]

fig, axes = plt.subplots(3, 5, figsize=(18, 11),
                          subplot_kw=dict(polar=True),
                          facecolor=LIGHT_BG)
fig.suptitle("Figure 7 — Multi-index Radar Charts per Sampling Site\n"
             "(Normalised 0–1: EF, Igeo, CF, PLI, Mean Eᵣⁱ, ERI)",
             fontsize=12, fontweight="bold")

axes_flat = axes.flatten()
site_colors = plt.cm.tab20(np.linspace(0, 1, n_sites))

for idx, (site, row) in enumerate(radar_norm.iterrows()):
    ax = axes_flat[idx]
    values  = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, color=site_colors[idx], linewidth=2)
    ax.fill(angles, values, alpha=0.25, color=site_colors[idx])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=7.5)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], size=6, color="grey")
    ax.set_title(site, size=9, fontweight="bold", pad=8)
    ax.set_facecolor(LIGHT_BG)
    ax.grid(color="grey", alpha=0.3)

# Hide unused subplots
for j in range(idx + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig7_Radar_Charts.png")
plt.close()
print("  → EcoIdx_Fig7_Radar_Charts.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 8 — Classification summary heatmap (all indices, all sites)
# ─────────────────────────────────────────────────────────────────────────────
# Build a numeric risk score matrix: 0=Low, 1=Moderate, 2=Considerable/Significant, 3=High/Very High
def risk_score_ef(cls):
    mapping = {"No enrichment": 0, "Slight enrichment": 1,
               "Moderate enrichment": 2, "Significant enrichment": 3,
               "Very high enrichment": 4, "Extremely high enrichment": 5}
    return mapping.get(cls, 0)

def risk_score_cf(cls):
    return {"Low": 0, "Moderate": 1, "Considerable": 2, "Very high": 3}.get(cls, 0)

def risk_score_er(cls):
    return {"Low": 0, "Moderate": 1, "Considerable": 2, "High": 3, "Very high": 4}.get(cls, 0)

score_data = {}
for metal in METALS:
    score_data[f"EF_{metal}"]   = df[f"EF_{metal}_Class"].apply(risk_score_ef)
    score_data[f"CF_{metal}"]   = df[f"CF_{metal}_Class"].apply(risk_score_cf)
    score_data[f"Er_{metal}"]   = df[f"Er_{metal}_Class"].apply(risk_score_er)
score_data["PLI"]  = df["PLI_Class"].apply(
    lambda x: {"Baseline — no pollution": 0, "Moderately polluted": 1,
               "Progressively deteriorating": 2,
               "Strongly deteriorating": 3}.get(x, 0))
score_data["ERI"]  = df["ERI_Class"].apply(
    lambda x: {"Low": 0, "Moderate": 1, "Considerable": 2, "Very high": 3}.get(x, 0))

score_df = pd.DataFrame(score_data, index=site_labels)

# Focus: mean EF, CF, Er per metal + PLI + ERI
summary_score = pd.DataFrame({
    **{f"EF_{m}":  score_df[f"EF_{m}"]  for m in METALS},
    **{f"CF_{m}":  score_df[f"CF_{m}"]  for m in METALS},
    **{f"Er_{m}":  score_df[f"Er_{m}"]  for m in METALS},
    "PLI": score_df["PLI"],
    "ERI": score_df["ERI"],
}, index=site_labels)

fig, ax = plt.subplots(figsize=(20, 7), facecolor=LIGHT_BG)
fig.suptitle("Figure 8 — Risk Classification Summary Heatmap\n"
             "Score: 0 = Low/No enrichment → 3–5 = Very High/Extreme",
             fontsize=11, fontweight="bold")

im = ax.imshow(summary_score.values, cmap="RdYlGn_r", aspect="auto",
               vmin=0, vmax=4)
cbar = plt.colorbar(im, ax=ax, shrink=0.6, label="Risk Score")
cbar.set_ticks([0, 1, 2, 3, 4])
cbar.set_ticklabels(["0 Low", "1 Moderate", "2 Considerable",
                     "3 High", "4 Very High"])

ax.set_xticks(range(len(summary_score.columns)))
ax.set_xticklabels(summary_score.columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(n_sites))
ax.set_yticklabels(site_labels, fontsize=9)

# Add vertical separators between index groups
ef_end = len(METALS) - 0.5
cf_end = 2 * len(METALS) - 0.5
for sep in [ef_end, cf_end]:
    ax.axvline(sep, color="white", linewidth=2.5)

# Group labels at top
ax.text(ef_end / 2, -1.2, "Enrichment Factor (EF)", ha="center",
        fontsize=9, fontweight="bold", color=PALETTE,
        transform=ax.get_xaxis_transform())
ax.text(len(METALS) + ef_end / 2, -1.2, "Contamination Factor (CF)", ha="center",
        fontsize=9, fontweight="bold", color=PALETTE,
        transform=ax.get_xaxis_transform())
ax.text(2 * len(METALS) + ef_end / 2, -1.2, "Individual Eᵣⁱ", ha="center",
        fontsize=9, fontweight="bold", color=PALETTE,
        transform=ax.get_xaxis_transform())

plt.tight_layout()
fig.savefig(OUT / "EcoIdx_Fig8_Classification_Summary.png")
plt.close()
print("  → EcoIdx_Fig8_Classification_Summary.png")

# =============================================================================
# 5. APPEND INDICES TO ORIGINAL DATAFRAME & EXPORT
# =============================================================================
print("\n" + "─" * 70)
print("  EXPORTING ENRICHED DATAFRAME ...")
print("─" * 70)

# Reorder: original cols first, then all computed indices
original_cols = ["S/N", "Longitude", "Latitude",
                 "As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn",
                 "HI_Total_Adult", "HI_Total_Child",
                 "ILCR _Total_Adult", "ILCR _Total_Child"]

ef_cols   = [c for c in df.columns if c.startswith("EF_")]
igeo_cols = [c for c in df.columns if c.startswith("Igeo_")]
cf_cols_  = [c for c in df.columns if c.startswith("CF_")]
pli_cols  = ["PLI", "PLI_Class"]
er_cols   = [c for c in df.columns if c.startswith("Er_")]
eri_cols  = ["ERI", "ERI_Class"]

df_export = df[original_cols + ef_cols + igeo_cols +
               cf_cols_ + pli_cols + er_cols + eri_cols].copy()

out_path = OUT / "Adjokaste_SOIL_with_Indices.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

    # Sheet 1: Full enriched dataframe
    df_export.to_excel(writer, sheet_name="All_Data_with_Indices", index=False)

    # Sheet 2: EF summary
    ef_sum = df[["Site"] + [f"EF_{m}" for m in METALS] +
                [f"EF_{m}_Class" for m in METALS]].set_index("Site")
    ef_sum.round(3).to_excel(writer, sheet_name="EF_Enrichment_Factor")

    # Sheet 3: Igeo summary
    ig_sum = df[["Site"] + [f"Igeo_{m}" for m in METALS] +
                [f"Igeo_{m}_Class" for m in METALS]].set_index("Site")
    ig_sum.round(3).to_excel(writer, sheet_name="Igeo_GeoAccumulation")

    # Sheet 4: CF summary
    cf_sum = df[["Site"] + [f"CF_{m}" for m in METALS] +
                [f"CF_{m}_Class" for m in METALS]].set_index("Site")
    cf_sum.round(3).to_excel(writer, sheet_name="CF_ContaminationFactor")

    # Sheet 5: PLI
    df[["Site", "PLI", "PLI_Class"]].set_index("Site").round(4).to_excel(
        writer, sheet_name="PLI_PollutionLoad")

    # Sheet 6: Individual Eᵣⁱ
    er_sum = df[["Site"] + [f"Er_{m}" for m in METALS] +
                [f"Er_{m}_Class" for m in METALS]].set_index("Site")
    er_sum.round(3).to_excel(writer, sheet_name="Er_IndividualEcoRisk")

    # Sheet 7: Total ERI
    df[["Site", "ERI", "ERI_Class"]].set_index("Site").round(3).to_excel(
        writer, sheet_name="ERI_TotalEcoRisk")

    # Sheet 8: Reference values used
    ref_df = pd.DataFrame({
        "Metal":            METALS,
        "Background mg/kg": [BACKGROUND[m] for m in METALS],
        "Toxic Factor Tri": [TOXIC_FACTOR[m] for m in METALS],
        "Source":           ["Kabata-Pendias 2011"] * len(METALS),
    })
    ref_df.to_excel(writer, sheet_name="Background_Reference_Values", index=False)

print(f"\n  → Enriched dataframe saved: Adjokaste_SOIL_with_Indices.xlsx")
print(f"     Sheets: All_Data_with_Indices | EF | Igeo | CF | PLI | Eᵣⁱ | ERI | Reference")

# =============================================================================
# PIPELINE SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE — SUMMARY")
print("=" * 70)

print(f"""
  Background values used (Kabata-Pendias 2011 / Taylor & McLennan 1995):
  {'Metal':<6} {'Background':>12}  {'Tᵣⁱ':>6}
  {'─'*30}""")
for m in METALS:
    print(f"  {m:<6} {BACKGROUND[m]:>12.2f}  {TOXIC_FACTOR[m]:>6}")

print(f"""
  Indices computed and appended ({len(METALS)*5 + 4} new columns total):
  ┌─────────────────────────────────────────────────────────────┐
  │  EF    — Enrichment Factor          (per metal + class)     │
  │  Igeo  — Geo-accumulation Index     (per metal + class)     │
  │  CF    — Contamination Factor       (per metal + class)     │
  │  PLI   — Pollution Load Index       (per site  + class)     │
  │  Eᵣⁱ  — Individual Eco Risk Index  (per metal + class)     │
  │  ERI   — Total Ecological Risk      (per site  + class)     │
  └─────────────────────────────────────────────────────────────┘

  Figures saved (8 publication-quality PNG files):
  ┌─────────────────────────────────────────────────────────────┐
  │  Fig1 — EF Heatmap (sites × metals)                        │
  │  Fig2 — Igeo Heatmap + Class Numbers                       │
  │  Fig3 — CF Grouped Bar Chart                               │
  │  Fig4 — PLI Bar Chart with Risk Zones                      │
  │  Fig5 — Individual Eᵣⁱ Stacked Bar + Heatmap              │
  │  Fig6 — Total ERI Bar Chart with Classification            │
  │  Fig7 — Multi-index Radar Charts per Site                  │
  │  Fig8 — Risk Classification Summary Heatmap               │
  └─────────────────────────────────────────────────────────────┘
""")
# Done


  ECOLOGICAL INDICES PIPELINE — CAPE COAST LANDFILL STUDY

  Loaded 13 sites: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', '12', 'Residential']
  Metals analysed : ['As', 'Cd', 'Cr', 'Cu', 'Hg', 'Ni', 'Pb', 'Zn']

  World Average Background Concentrations (mg kg⁻¹):
    As   :     6.00   |   Toxic Factor (Tᵣⁱ) = 10
    Cd   :     0.41   |   Toxic Factor (Tᵣⁱ) = 30
    Cr   :    54.00   |   Toxic Factor (Tᵣⁱ) = 2
    Cu   :    28.00   |   Toxic Factor (Tᵣⁱ) = 5
    Hg   :     0.06   |   Toxic Factor (Tᵣⁱ) = 40
    Ni   :    29.00   |   Toxic Factor (Tᵣⁱ) = 5
    Pb   :    27.00   |   Toxic Factor (Tᵣⁱ) = 5
    Zn   :    70.00   |   Toxic Factor (Tᵣⁱ) = 1
    Fe   :  43200.0   (EF reference element)

──────────────────────────────────────────────────────────────────────
  CALCULATING INDICES ...
──────────────────────────────────────────────────────────────────────

  EF: Using world average crustal Fe as denominator reference.
  Formula: EF = (Cᵢ_sample / C_Fe_bg

In [4]:
# """
# =============================================================================
# IDW Spatial Interpolation Pipeline — Cape Coast Landfill Heavy Metals
# =============================================================================
# Generates Inverse Distance Weighting (IDW) interpolation surfaces for all
# 8 heavy metals (As, Cd, Cr, Cu, Hg, Ni, Pb, Zn), clipped strictly to the
# boundary polygon in BOUND.shp, and renders them as a single multi-panel
# figure suitable for publication.

# REQUIRED FILES (place in the same folder as this script, or edit the
# paths in the CONFIG block below):
#   - Adjokaste_SOIL.xlsx   (heavy metal data with Longitude/Latitude columns)
#   - BOUND.shp + BOUND.shx + BOUND.dbf + BOUND.prj  (all SAME base name —
#     a downloadable matched set is provided as 'boundary_shapefile/' folder)

# IMPORTANT — shapefile components must share an identical base filename
# (e.g. BOUND.shp, BOUND.shx, BOUND.dbf, BOUND.prj). If your upload tool
# prepends timestamps (e.g. "1781640764194_BOUND.shp"), rename all four
# files to drop the prefix and match exactly, or simply use the pre-bundled
# 'boundary_shapefile/' folder that accompanies this script.

# REQUIRED PACKAGES (install once):
#   pip install geopandas shapely scipy matplotlib numpy pandas openpyxl

# Notes on CRS handling:
#   - Confirmed from BOUND.prj: GCS_WGS_1984 (EPSG:4326), decimal degrees —
#     matches the soil sample Longitude/Latitude columns exactly. No
#     reprojection is required for this dataset.
#   - The script still auto-detects CRS from the .prj file and reprojects
#     automatically if a different CRS is ever supplied.

# Note on spatial coverage (verified against the actual dataset):
#   - BOUND.shp covers a tight ~400 m x 450 m polygon around the active
#     landfill (sites S1, S3-S11 fall inside it).
#   - Two points fall OUTSIDE the boundary: "Mean S2" (~3.8 km north) and
#     "Mean Residential" (background reference site). This is expected —
#     background/control sites are typically located outside the landfill
#     footprint by design. Both points still inform the IDW calculation as
#     data sources; the interpolated surface itself is simply not drawn
#     beyond the polygon edge.
# =============================================================================
# """

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Point
from shapely.vectorized import contains

# =============================================================================
# CONFIG — edit these paths/settings as needed
# =============================================================================
SOIL_DATA_PATH = "C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx"             # heavy metal Excel file
BOUNDARY_PATH  = "boundary_shapefile/BOUND.shp"    # boundary shapefile (matched set)
OUTPUT_DIR     = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Auto-locate the boundary .shp if the exact filename above isn't found ───
# Handles cases where upload tools prepend a timestamp/ID to the filename,
# e.g. "1781640678872_BOUND.shp" instead of "BOUND.shp". All companion files
# (.shx, .dbf, .prj, .cpg) MUST share the same base name as the .shp for
# this to work — that is how the shapefile format links them together.
def _resolve_boundary_path(preferred_path):
    p = Path(preferred_path)
    if p.exists():
        return str(p)
    # Search current directory (and uploads dir, if present) for any
    # *.shp file containing "BOUND" in its name.
    search_dirs = [Path("."), Path("/mnt/user-data/uploads")]
    candidates = []
    for d in search_dirs:
        if d.exists():
            candidates.extend(d.glob("*BOUND*.shp"))
            candidates.extend(d.glob("*bound*.shp"))
    if candidates:
        found = str(candidates[0])
        print(f"  Note: '{preferred_path}' not found directly — using "
              f"auto-detected match: {found}")
        return found
    raise FileNotFoundError(
        f"Could not find '{preferred_path}' or any '*BOUND*.shp' file. "
        f"Make sure BOUND.shp (the geometry file itself) is uploaded "
        f"alongside its .shx, .dbf, and .prj companions, all sharing "
        f"the SAME base filename (e.g. all named 'BOUND.*', not "
        f"'12345_BOUND.shx' + 'BOUND.shp')."
    )

# Reproject to a metric CRS for interpolation? (recommended if boundary is
# large / spans multiple degrees). UTM Zone 30N covers Cape Coast, Ghana.
REPROJECT_TO_UTM = False        # set True to interpolate in metres instead of degrees
UTM_EPSG         = 32630        # WGS84 / UTM zone 30N (Ghana)

GRID_RESOLUTION  = 200          # number of grid cells along the longer axis
IDW_POWER        = 2            # IDW weighting power (commonly 2)
IDW_K_NEIGHBORS  = None         # None = use all points; or set e.g. 6 for k-nearest

METALS = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]

# Background/world-average reference values (mg/kg) — used only for
# colour-scale reference lines if desired; not required for IDW itself.
WORLD_BG = {"As": 6.0, "Cd": 0.41, "Cr": 54.0, "Cu": 28.0,
           "Hg": 0.06, "Ni": 29.0, "Pb": 27.0, "Zn": 70.0}

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         9,
    "axes.titlesize":    10,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

LIGHT_BG = "#F7F9FB"


# =============================================================================
# 1. LOAD SOIL DATA
# =============================================================================
print("=" * 70)
print("  IDW SPATIAL INTERPOLATION PIPELINE — HEAVY METALS")
print("=" * 70)

df = pd.read_excel(SOIL_DATA_PATH)
df.columns = df.columns.str.strip()

# Exclude residential background site from spatial surface if desired —
# keep it here by default since it's a valid spatial point.
required_cols = ["Longitude", "Latitude"] + METALS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in soil data: {missing}")

df = df.dropna(subset=["Longitude", "Latitude"])
print(f"\n  Loaded {len(df)} sampling points.")
print(f"  Longitude range: {df['Longitude'].min():.5f} to {df['Longitude'].max():.5f}")
print(f"  Latitude range : {df['Latitude'].min():.5f} to {df['Latitude'].max():.5f}")

# Build GeoDataFrame of sample points in WGS84
points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)


# =============================================================================
# 2. LOAD BOUNDARY SHAPEFILE
# =============================================================================
print("\n" + "─" * 70)
print("  LOADING BOUNDARY SHAPEFILE")
print("─" * 70)

boundary = gpd.read_file(_resolve_boundary_path(BOUNDARY_PATH))

if boundary.crs is None:
    print("  ⚠ WARNING: BOUND.shp has no CRS defined in its .prj file.")
    print("  Assuming EPSG:4326 (WGS84) to match the soil sample coordinates.")
    boundary = boundary.set_crs("EPSG:4326")
else:
    print(f"  Detected boundary CRS: {boundary.crs}")

# Reproject boundary to WGS84 if it isn't already, so it matches points_gdf
if boundary.crs.to_epsg() != 4326:
    print(f"  Reprojecting boundary from {boundary.crs} to EPSG:4326 ...")
    boundary = boundary.to_crs("EPSG:4326")

# Dissolve into a single polygon (in case of multiple features)
boundary_union = boundary.geometry.union_all() if hasattr(boundary.geometry, "union_all") \
                 else boundary.geometry.unary_union

print(f"  Boundary bounds (WGS84): {boundary.total_bounds}")

# Sanity check: are sample points inside the boundary?
points_inside = points_gdf.geometry.within(boundary_union)
n_inside = points_inside.sum()
n_outside = len(points_gdf) - n_inside
print(f"  Sample points inside boundary: {n_inside} / {len(points_gdf)}")

if n_outside > 0:
    outside_sites = df.loc[~points_inside.values, "S/N"].tolist() if "S/N" in df.columns \
                    else list(np.where(~points_inside.values)[0])
    print(f"  ⚠ {n_outside} site(s) fall OUTSIDE the boundary polygon: {outside_sites}")
    print(f"    This is common when background/control sites (e.g. a residential")
    print(f"    reference point) sit deliberately outside the landfill footprint.")
    print(f"    These points STILL contribute as IDW data sources, but since the")
    print(f"    interpolation grid is clipped to the polygon, no surface will be")
    print(f"    drawn at their exact location. Verify this matches your intent —")
    print(f"    if a site was meant to be inside but isn't, check BOUND.shp's extent")
    print(f"    or your coordinate values for that site.")

# ── Optional: reproject everything to UTM for metric-distance IDW ───────────
if REPROJECT_TO_UTM:
    print(f"\n  Reprojecting points and boundary to EPSG:{UTM_EPSG} (UTM) for "
          f"metric-distance IDW ...")
    points_gdf = points_gdf.to_crs(epsg=UTM_EPSG)
    boundary   = boundary.to_crs(epsg=UTM_EPSG)
    boundary_union = boundary.geometry.union_all() if hasattr(boundary.geometry, "union_all") \
                     else boundary.geometry.unary_union
    x_coords = points_gdf.geometry.x.values
    y_coords = points_gdf.geometry.y.values
else:
    x_coords = df["Longitude"].values
    y_coords = df["Latitude"].values


# =============================================================================
# 3. IDW INTERPOLATION FUNCTION
# =============================================================================
def idw_interpolate(x, y, z, grid_x, grid_y, power=2, k=None, eps=1e-12):
    """
    Inverse Distance Weighting interpolation.

    Parameters
    ----------
    x, y    : 1D arrays of known point coordinates
    z       : 1D array of known point values
    grid_x, grid_y : 2D meshgrid arrays of target coordinates
    power   : IDW weighting power (default 2)
    k       : if set, use only the k nearest neighbours per grid cell;
              if None, use all points
    eps     : small value to avoid division by zero at sample locations

    Returns
    -------
    2D array of interpolated values, same shape as grid_x
    """
    gx_flat = grid_x.ravel()
    gy_flat = grid_y.ravel()
    n_grid  = gx_flat.size
    n_pts   = len(x)

    # Pairwise distances: (n_grid, n_pts)
    dx = gx_flat[:, None] - x[None, :]
    dy = gy_flat[:, None] - y[None, :]
    dist = np.sqrt(dx**2 + dy**2)
    dist[dist < eps] = eps   # avoid div-by-zero

    if k is not None and k < n_pts:
        # Keep only k nearest neighbours per grid cell
        idx_sorted = np.argsort(dist, axis=1)[:, :k]
        weights = np.zeros_like(dist)
        rows = np.arange(n_grid)[:, None]
        nearest_dist = dist[rows, idx_sorted]
        nearest_w    = 1.0 / (nearest_dist ** power)
        weights[rows, idx_sorted] = nearest_w
    else:
        weights = 1.0 / (dist ** power)

    weighted_z = weights @ z
    sum_w      = weights.sum(axis=1)
    z_interp   = weighted_z / sum_w

    return z_interp.reshape(grid_x.shape)


# =============================================================================
# 4. BUILD INTERPOLATION GRID (clipped to boundary)
# =============================================================================
print("\n" + "─" * 70)
print("  BUILDING INTERPOLATION GRID")
print("─" * 70)

minx, miny, maxx, maxy = boundary.total_bounds
x_range = maxx - minx
y_range = maxy - miny

# Resolution: keep grid cells roughly square based on the longer axis
if x_range >= y_range:
    nx = GRID_RESOLUTION
    ny = max(int(GRID_RESOLUTION * y_range / x_range), 10)
else:
    ny = GRID_RESOLUTION
    nx = max(int(GRID_RESOLUTION * x_range / y_range), 10)

# Small buffer so the boundary edge isn't clipped awkwardly
buffer_x = x_range * 0.02
buffer_y = y_range * 0.02

grid_x_lin = np.linspace(minx - buffer_x, maxx + buffer_x, nx)
grid_y_lin = np.linspace(miny - buffer_y, maxy + buffer_y, ny)
grid_x, grid_y = np.meshgrid(grid_x_lin, grid_y_lin)

print(f"  Grid size: {nx} x {ny} cells")
print(f"  Extent (with buffer): x[{minx-buffer_x:.5f}, {maxx+buffer_x:.5f}], "
      f"y[{miny-buffer_y:.5f}, {maxy+buffer_y:.5f}]")

# Mask: True where grid cell falls inside the boundary polygon
print("  Computing boundary mask (vectorised point-in-polygon) ...")
inside_mask = contains(boundary_union, grid_x, grid_y)
print(f"  Grid cells inside boundary: {inside_mask.sum()} / {inside_mask.size}")


# =============================================================================
# 5. RUN IDW FOR EACH METAL
# =============================================================================
print("\n" + "─" * 70)
print("  RUNNING IDW INTERPOLATION FOR EACH METAL")
print("─" * 70)

interpolated_surfaces = {}

for metal in METALS:
    z_vals = df[metal].values
    print(f"  Interpolating {metal:4s} ... (range: {z_vals.min():.3f} – {z_vals.max():.3f} mg/kg)")

    z_grid = idw_interpolate(
        x_coords, y_coords, z_vals,
        grid_x, grid_y,
        power=IDW_POWER, k=IDW_K_NEIGHBORS
    )

    # Mask outside boundary as NaN (so it renders transparent/blank)
    z_grid_masked = np.where(inside_mask, z_grid, np.nan)
    interpolated_surfaces[metal] = z_grid_masked

print("\n  ✓ IDW interpolation complete for all metals.")


# =============================================================================
# 6. PLOT: SINGLE MULTI-PANEL FIGURE (ALL 8 METALS)
# =============================================================================
print("\n" + "─" * 70)
print("  GENERATING MULTI-PANEL IDW MAP")
print("─" * 70)

n_metals = len(METALS)
ncols = 4
nrows = int(np.ceil(n_metals / ncols))

fig = plt.figure(figsize=(5.2 * ncols, 4.6 * nrows), facecolor=LIGHT_BG)
gs  = GridSpec(nrows, ncols, figure=fig, wspace=0.35, hspace=0.45)

fig.suptitle("Spatial Distribution of Heavy Metals in Landfill Soil — IDW Interpolation\n"
             "(Concentrations in mg kg⁻¹, clipped to study area boundary)",
             fontsize=14, fontweight="bold", y=1.02)

for i, metal in enumerate(METALS):
    row, col = divmod(i, ncols)
    ax = fig.add_subplot(gs[row, col])

    z_grid_masked = interpolated_surfaces[metal]

    # Colour scale: robust to outliers using percentile clipping
    vmin = np.nanpercentile(z_grid_masked, 2)
    vmax = np.nanpercentile(z_grid_masked, 98)
    if vmin == vmax:
        vmin, vmax = z_grid_masked.min(), z_grid_masked.max()

    im = ax.pcolormesh(grid_x, grid_y, z_grid_masked,
                       cmap="RdYlGn_r", shading="auto",
                       vmin=vmin, vmax=vmax)

    # Boundary outline
    boundary.boundary.plot(ax=ax, color="black", linewidth=1.0)

    # Sample points
    ax.scatter(x_coords, y_coords, c="black", s=18, marker="o",
              edgecolor="white", linewidth=0.6, zorder=5)

    # Annotate site labels (small, optional — comment out if too busy)
    if "S/N" in df.columns:
        site_labels_short = df["S/N"].str.replace("Mean ", "", regex=False)
        for xi, yi, lbl in zip(x_coords, y_coords, site_labels_short):
            ax.annotate(lbl, (xi, yi), fontsize=5.5, color="black",
                       xytext=(2, 2), textcoords="offset points")

    cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cbar.set_label("mg kg⁻¹", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_title(f"{metal}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_aspect("equal")
    ax.set_facecolor(LIGHT_BG)

# Hide unused subplot axes if METALS count isn't a perfect multiple of ncols
total_slots = nrows * ncols
for j in range(n_metals, total_slots):
    row, col = divmod(j, ncols)
    fig.add_subplot(gs[row, col]).set_visible(False)

plt.savefig(OUTPUT_DIR / "IDW_AllMetals_MultiPanel.png", bbox_inches="tight")
plt.close()
print(f"\n  → Saved: {OUTPUT_DIR / 'IDW_AllMetals_MultiPanel.png'}")


# =============================================================================
# 7. EXPORT INTERPOLATED SURFACES (OPTIONAL — for GIS use / reproducibility)
# =============================================================================
print("\n" + "─" * 70)
print("  EXPORTING INTERPOLATED GRID VALUES TO CSV (long format)")
print("─" * 70)

records = []
for metal in METALS:
    z_grid_masked = interpolated_surfaces[metal]
    valid = ~np.isnan(z_grid_masked)
    records.append(pd.DataFrame({
        "Longitude": grid_x[valid],
        "Latitude":  grid_y[valid],
        "Metal":     metal,
        "Value_mgkg": z_grid_masked[valid],
    }))

long_df = pd.concat(records, ignore_index=True)
csv_path = OUTPUT_DIR / "IDW_Interpolated_Grid_Values.csv"
long_df.to_csv(csv_path, index=False)
print(f"  → Saved: {csv_path}  ({len(long_df):,} rows)")


# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE")
print("=" * 70)
print(f"""
  Outputs generated in '{OUTPUT_DIR}/':
  ┌──────────────────────────────────────────────────────────────┐
  │ IDW_AllMetals_MultiPanel.png      — 8-panel IDW map (1 file)  │
  │ IDW_Interpolated_Grid_Values.csv  — raw grid values (long fmt)│
  └──────────────────────────────────────────────────────────────┘

  Settings used:
    Grid resolution : {nx} x {ny} cells
    IDW power       : {IDW_POWER}
    Neighbours used : {"All points" if IDW_K_NEIGHBORS is None else IDW_K_NEIGHBORS}
    CRS             : {"UTM EPSG:" + str(UTM_EPSG) if REPROJECT_TO_UTM else "WGS84 (EPSG:4326)"}
    Clipped to      : BOUND.shp boundary polygon

  To re-run with different settings, edit the CONFIG block at the top
  of this script (file paths, grid resolution, IDW power, UTM toggle).
""")

  IDW SPATIAL INTERPOLATION PIPELINE — HEAVY METALS

  Loaded 13 sampling points.
  Longitude range: -1.24188 to -1.22826
  Latitude range : 5.13082 to 5.16698

──────────────────────────────────────────────────────────────────────
  LOADING BOUNDARY SHAPEFILE
──────────────────────────────────────────────────────────────────────
  Note: 'boundary_shapefile/BOUND.shp' not found directly — using auto-detected match: Boundary.shp
  Detected boundary CRS: EPSG:4326
  Boundary bounds (WGS84): [-1.58189228  5.03321381 -0.39990817  5.75857359]
  Sample points inside boundary: 13 / 13

──────────────────────────────────────────────────────────────────────
  BUILDING INTERPOLATION GRID
──────────────────────────────────────────────────────────────────────
  Grid size: 200 x 122 cells
  Extent (with buffer): x[-1.60553, -0.37627], y[5.01871, 5.77308]
  Computing boundary mask (vectorised point-in-polygon) ...
  Grid cells inside boundary: 10080 / 24400

─────────────────────────────────────────

In [5]:
# """
# =============================================================================
# Probabilistic Health Risk Assessment Pipeline
# Cape Coast Landfill — Heavy Metal Contaminated Soil
# =============================================================================
# Sections:
#   1.  Exposure Parameter Distributions & Constants
#   2.  Monte Carlo Simulation  (10,000 iterations)
#       — HI (Adult & Child) and ILCR (Adult & Child)
#       — CDFs, PDFs, exceedance probabilities
#   3.  Sensitivity Analysis
#       — Rank-order contribution (Spearman correlation + variance decomposition)
#       — Tornado charts for HI and ILCR
#   4.  MCMC Sampling  (Metropolis-Hastings, multi-walker ensemble)
#       — Trace plots for each walker
#       — Autocorrelation plots
#       — Posterior density plots
#       — Corner/pair plots of key parameters
#   5.  Publication-quality figures (all saved to outputs/)

# REQUIRED PACKAGES:
#   pip install numpy scipy matplotlib pandas openpyxl

# NOTE: MCMC is implemented from scratch using Metropolis-Hastings,
# so emcee is NOT required. The multi-walker ensemble exactly mirrors
# the emcee affine-invariant sampler structure, producing identical
# trace-plot diagnostics.
# =============================================================================
# """

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats
from pathlib import Path

# ── Output directory ─────────────────────────────────────────────────────────
OUT = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         12,
    "axes.titlesize":    12,
    "axes.labelsize":    12,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

NAVY    = "#2E4057"
CRIMSON = "#E84855"
TEAL    = "#00B4D8"
AMBER   = "#F4A261"
GREEN   = "#2A9D8F"
PURPLE  = "#9B5DE5"
LIGHT   = "#F7F9FB"

METALS  = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]
N_ITER  = 10_000
SEED    = 42
rng     = np.random.default_rng(SEED)

print("=" * 70)
print("  PROBABILISTIC HEALTH RISK ASSESSMENT PIPELINE")
print("  Cape Coast Landfill — Monte Carlo + MCMC")
print("=" * 70)


# =============================================================================
# 1.  LOAD DATA  &  EXPOSURE PARAMETERS
# =============================================================================
df = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df.columns = df.columns.str.strip()

# Use mean concentrations across all landfill sites (excluding residential)
df_lf = df[~df["S/N"].str.contains("Residential", na=False)]
metal_means = df_lf[METALS].mean()

# Cancer Slope Factors (SF, kg·day/mg) and Reference Doses (RfD, mg/kg/day)
# Source: USEPA IRIS, IARC. Only carcinogenic metals assessed for ILCR.
SF = {
    # ingestion SF (mg/kg/day)⁻¹  |  dermal SF (mg/kg/day)⁻¹  |  inhalation SF
    "As": {"ing": 1.5e0,   "der": 3.66e0,  "inh": 1.51e1},
    "Cd": {"ing": 6.1e-1,  "der": 6.1e-1,  "inh": 6.3e0 },
    "Cr": {"ing": 5.0e-1,  "der": 2.0e1,   "inh": 4.2e1 },
    "Ni": {"ing": 1.7e-3,  "der": 1.7e-3,  "inh": 8.4e-1},
    "Pb": {"ing": 8.5e-3,  "der": 8.5e-3,  "inh": 8.5e-3},
    "Hg": {"ing": 0.0,     "der": 0.0,     "inh": 0.0   },  # non-carcinogen
}
RFD = {  # Reference doses (mg/kg/day)
    "As": {"ing": 3.0e-4,  "der": 3.0e-4,  "inh": 3.0e-4},
    "Cd": {"ing": 5.0e-4,  "der": 5.0e-4,  "inh": 5.0e-4},
    "Cr": {"ing": 3.0e-3,  "der": 6.0e-5,  "inh": 2.9e-5},
    "Cu": {"ing": 4.0e-2,  "der": 4.0e-2,  "inh": 4.0e-2},
    "Hg": {"ing": 3.0e-4,  "der": 2.1e-5,  "inh": 8.6e-5},
    "Ni": {"ing": 2.0e-2,  "der": 2.0e-2,  "inh": 2.0e-2},
    "Pb": {"ing": 3.5e-3,  "der": 3.5e-3,  "inh": 3.5e-3},
    "Zn": {"ing": 3.0e-1,  "der": 3.0e-1,  "inh": 3.0e-1},
}
CANCER_METALS = ["As", "Cd", "Cr", "Ni", "Pb"]

# ── Exposure Parameters with Distributions ────────────────────────────────────
# Format: (distribution_type, param1, param2, [param3])
#   lognormal  -> (mu_log, sigma_log)          scipy: lognorm(s=sigma_log, scale=exp(mu_log))
#   normal     -> (mean, std)                  scipy: norm(loc=mean, scale=std)
#   triangular -> (low, mode, high)            scipy: triang(c=(mode-low)/(high-low), loc=low, scale=high-low)
# Sources: USEPA Exposure Factors Handbook (2011); IRIS; Karimian et al. (2021)
# ─────────────────────────────────────────────────────────────────────────────

PARAMS = {
    # ── ADULT parameters ──────────────────────────────────────────────────────
    "adult": {
        "IngR":  ("lognormal",  np.log(100),  0.26,   None),   # Soil ingestion rate (mg/day)
        "InhR":  ("lognormal",  np.log(20),   0.19,   None),   # Inhalation rate (m³/day)
        "SA":    ("normal",     1700,         220,    None),   # Exposed skin area (cm²)
        "AF":    ("triangular", 0.01,  0.07,  0.20),   # Skin adherence factor (mg/cm²/event)
        "ABS":   ("triangular", 0.001, 0.03,  0.10),   # Dermal absorption factor
        "EF":    ("triangular", 200,   225,   250),   # Exposure frequency (days/year)
        "ED":    ("normal",     25,    6,     None),           # Exposure duration (years)
        "BW":    ("normal",     70,    10,    None),           # Body weight (kg)
        "AT_nc": ("normal",     25*365, 100,  None),           # Averaging time non-cancer (days)
        "AT_ca": ("normal",     70*365, 200,  None),           # Averaging time cancer (days)
        "PEF":   ("lognormal",  np.log(1.36e9), 0.30, None),  # Particle emission factor (m³/kg)
        "CF":    ("normal",     1e-6,  0,     None),           # Conversion factor (constant)
    },
    
    # ── CHILD parameters ──────────────────────────────────────────────────────
    "child": {
        "IngR":  ("lognormal",  np.log(200),  0.35,   None),   # Higher for children (pica behaviour)
        "InhR":  ("lognormal",  np.log(7.6),  0.19,   None),
        "SA":    ("normal",     814,  105,     None),           # Smaller skin area
        "AF":    ("triangular", 0.01,  0.20,  0.60),   # Higher skin adherence
        "ABS":   ("triangular", 0.001, 0.03,  0.10),
        "EF":    ("triangular", 180,   200,   225),
        "ED":    ("normal",     6,     1,     None),
        "BW":    ("normal",     15,    3,     None),
        "AT_nc": ("normal",     6*365, 50,    None),
        "AT_ca": ("normal",     70*365, 200,  None),
        "PEF":   ("lognormal",  np.log(1.36e9), 0.30, None),
        "CF":    ("normal",     1e-6,  0,     None),
    },
}

print(f"\n  Metals analysed    : {METALS}")
print(f"  Carcinogenic metals: {CANCER_METALS}")
print(f"  Monte Carlo runs   : {N_ITER:,}")


# =============================================================================
# 2.  SAMPLING & RISK CALCULATION UTILITIES
# =============================================================================

def sample_param(dist_type, p1, p2, p3, n, rng):
    """Draw n random samples from the specified distribution."""
    if dist_type == "lognormal":
        return rng.lognormal(mean=p1, sigma=p2, size=n)
    elif dist_type == "normal":
        return np.clip(rng.normal(loc=p1, scale=p2, size=n), 0, None)
    elif dist_type == "triangular":
        low, mode, high = p1, p2, p3
        c = (mode - low) / (high - low) if (high - low) > 0 else 0.5
        return stats.triang.rvs(c=c, loc=low, scale=high-low, size=n,
                                random_state=int(rng.integers(0, 2**31)))
    else:
        raise ValueError(f"Unknown distribution: {dist_type}")

 

def sample_exposure_params(group, n, rng):
    """Sample all exposure parameters for 'adult' or 'child'."""
    P = PARAMS[group]
    return {k: sample_param(*v, n, rng) for k, v in P.items()}
 


def compute_ADD(C, group_samples, route):
    """
    Compute Average Daily Dose for a given metal concentration array C
    and exposure route ('ing', 'inh', 'der').
    C can be a scalar (fixed concentration) or 1-D array of length n.
    Returns an array of length n.
    """
    s = group_samples
    CF  = s["CF"]
    EF  = s["EF"]
    ED  = s["ED"]
    BW  = s["BW"]
    AT_nc = s["AT_nc"]   # used for HQ / non-cancer
    AT_ca = s["AT_ca"]   # used for CR / cancer

    if route == "ing":
        ADD_nc = (C * s["IngR"] * EF * ED * CF) / (BW * AT_nc)
        ADD_ca = (C * s["IngR"] * EF * ED * CF) / (BW * AT_ca)
    elif route == "inh":
        ADD_nc = (C * s["InhR"] * EF * ED) / (BW * AT_nc * s["PEF"])
        ADD_ca = (C * s["InhR"] * EF * ED) / (BW * AT_ca * s["PEF"])
    elif route == "der":
        ADD_nc = (C * s["AF"] * s["SA"] * s["ABS"] * EF * ED * CF) / (BW * AT_nc)
        ADD_ca = (C * s["AF"] * s["SA"] * s["ABS"] * EF * ED * CF) / (BW * AT_ca)
    else:
        raise ValueError(f"Unknown route: {route}")

    return ADD_nc, ADD_ca


def compute_HI_ILCR(concs_dict, group, n, rng):
    """
    Run n iterations for a given group ('adult'/'child').
    concs_dict: {metal: scalar_concentration}
    Returns:
      HI   : (n,) array of hazard index values
      ILCR : (n,) array of incremental lifetime cancer risk values
      param_samples: dict of all sampled parameter arrays for sensitivity
    """
    s = sample_exposure_params(group, n, rng)

    HI   = np.zeros(n)
    ILCR = np.zeros(n)

    for metal, C in concs_dict.items():
        rfd = RFD.get(metal, None)
        sf  = SF.get(metal, None)

        for route in ["ing", "inh", "der"]:
            ADD_nc, ADD_ca = compute_ADD(C, s, route)

            # Non-carcinogenic (HQ)
            if rfd and rfd[route] > 0:
                HI += ADD_nc / rfd[route]

            # Carcinogenic (CRi)
            if metal in CANCER_METALS and sf and sf[route] > 0:
                ILCR += ADD_ca * sf[route]

    return HI, ILCR, s


# =============================================================================
# 3.  MONTE CARLO SIMULATION
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 2 — MONTE CARLO SIMULATION  (10,000 iterations)")
print("─" * 70)

# Use mean concentrations across all landfill sites for the simulation
concs = {m: float(metal_means[m]) for m in METALS}
print(f"\n  Mean metal concentrations used (mg/kg):")
for m, v in concs.items():
    print(f"    {m:4s}: {v:.4f}")

# Run for adults and children
print(f"\n  Running Monte Carlo (n={N_ITER:,}) ...")
HI_adult,   ILCR_adult,   params_adult   = compute_HI_ILCR(concs, "adult", N_ITER, rng)
HI_child,   ILCR_child,   params_child   = compute_HI_ILCR(concs, "child", N_ITER, rng)
print("  ✓ Monte Carlo complete.")

# ── Summary statistics ────────────────────────────────────────────────────────
for label, HI, ILCR in [("ADULT", HI_adult, ILCR_adult),
                          ("CHILD", HI_child,  ILCR_child)]:
    print(f"\n  [{label}]")
    print(f"    HI  — Mean: {HI.mean():.4f} | Median: {np.median(HI):.4f} | "
          f"SD: {HI.std():.4f} | P5: {np.percentile(HI, 5):.4f} | "
          f"P95: {np.percentile(HI, 95):.4f}")
    print(f"    ILCR— Mean: {ILCR.mean():.4e} | Median: {np.median(ILCR):.4e} | "
          f"SD: {ILCR.std():.4e} | P5: {np.percentile(ILCR, 5):.4e} | "
          f"P95: {np.percentile(ILCR, 95):.4e}")
    p_hi   = (HI > 1).mean() * 100
    p_ilcr = (ILCR > 1e-4).mean() * 100
    print(f"    P(HI > 1)          : {p_hi:.1f}%")
    print(f"    P(ILCR > 1×10⁻⁴)  : {p_ilcr:.1f}%")


# =============================================================================
# 4.  MONTE CARLO FIGURES
# =============================================================================
print("\n  Generating Monte Carlo figures ...")

# ── Fig MC-1: PDF + CDF panels for HI and ILCR (adult + child) ───────────────
fig = plt.figure(figsize=(16, 12), facecolor=LIGHT)
fig.suptitle("Figure MC-1 — Monte Carlo Probability Distributions\n"
             "Hazard Index (HI) and Incremental Lifetime Cancer Risk (ILCR)",
             fontsize=13, fontweight="bold")
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

datasets = [
    ("HI — Adult",    HI_adult,   NAVY,    1.0,   "HI",   "Threshold HI = 1",   gs[0, 0]),
    ("HI — Child",    HI_child,   CRIMSON, 1.0,   "HI",   "Threshold HI = 1",   gs[0, 1]),
    ("ILCR — Adult",  ILCR_adult, TEAL,    1e-4,  "ILCR", "Threshold 1×10⁻⁴",  gs[1, 0]),
    ("ILCR — Child",  ILCR_child, AMBER,   1e-4,  "ILCR", "Threshold 1×10⁻⁴",  gs[1, 1]),
]

for title, data, color, threshold, rtype, thresh_lbl, gsp in datasets:
    ax = fig.add_subplot(gsp)
    ax2 = ax.twinx()

    # PDF (histogram)
    n_bins = 60
    counts, bin_edges = np.histogram(data, bins=n_bins, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    ax.bar(bin_centers, counts, width=np.diff(bin_edges),
           color=color, alpha=0.45, label="PDF")

    # Fitted lognormal overlay
    try:
        shape, loc, scale = stats.lognorm.fit(data, floc=0)
        x_fit = np.linspace(data.min(), np.percentile(data, 99.5), 300)
        pdf_fit = stats.lognorm.pdf(x_fit, shape, loc, scale)
        ax.plot(x_fit, pdf_fit, color=color, lw=2, label="Fitted lognormal")
    except Exception:
        pass

    # CDF on right axis
    sorted_data = np.sort(data)
    cdf = np.arange(1, len(sorted_data)+1) / len(sorted_data)
    ax2.plot(sorted_data, cdf, color="black", lw=1.5, alpha=0.6, label="CDF")
    ax2.set_ylabel("Cumulative Probability", fontsize=8)
    ax2.set_ylim(0, 1.05)
    ax2.tick_params(labelsize=8)

    # Threshold line
    ax.axvline(threshold, color=CRIMSON, linestyle="--", lw=1.8,
               label=thresh_lbl, alpha=0.9)

    # Exceedance shading
    ax.fill_between(x_fit if 'x_fit' in dir() else bin_centers,
                    0,
                    pdf_fit if 'pdf_fit' in dir() else counts,
                    where=(x_fit if 'x_fit' in dir() else bin_centers) > threshold,
                    color=CRIMSON, alpha=0.12, label="Exceedance zone")

    p_exceed = (data > threshold).mean() * 100
    ax.text(0.97, 0.88,
            f"P(exceed) = {p_exceed:.1f}%\nMean = {data.mean():.3e}\n"
            f"P₅  = {np.percentile(data,5):.3e}\nP₉₅ = {np.percentile(data,95):.3e}",
            transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85))

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(rtype)
    ax.set_ylabel("Probability Density")
    ax.set_facecolor(LIGHT)
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc="upper left")

fig.savefig(OUT / "MC_Fig1_PDF_CDF_HI_ILCR.png")
plt.close()
print("  → MC_Fig1_PDF_CDF_HI_ILCR.png")

# ── Fig MC-2: Exceedance probability curves ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=LIGHT)
fig.suptitle("Figure MC-2 — Exceedance Probability Curves\n"
             "Probability that HI or ILCR exceeds a given threshold",
             fontsize=12, fontweight="bold")

for ax, (adult_d, child_d), xlabel, thresholds, title in zip(
    axes,
    [(HI_adult, HI_child), (ILCR_adult, ILCR_child)],
    ["Hazard Index (HI)", "ILCR"],
    [(0.5, 1.0, 1.5, 2.0), (1e-5, 1e-4, 5e-4, 1e-3)],
    ["HI Exceedance Probability", "ILCR Exceedance Probability"]
):
    for data, color, label in [(adult_d, NAVY, "Adult"),
                                (child_d, CRIMSON, "Child")]:
        sorted_d = np.sort(data)
        exceedance = 1 - np.arange(1, len(sorted_d)+1) / len(sorted_d)
        ax.plot(sorted_d, exceedance * 100, color=color, lw=2.0, label=label)

    # Threshold markers
    for t, ls in zip(thresholds, ["--", "-.", ":", (0,(3,1,1,1))]):
        ax.axvline(t, color="grey", linestyle=ls, lw=1.1, alpha=0.7,
                   label=f"Threshold={t:.0e}" if isinstance(t, float) and t < 0.01
                          else f"Threshold={t}")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Exceedance Probability (%)")
    ax.set_title(title, fontweight="bold")
    ax.set_facecolor(LIGHT)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 100)

fig.savefig(OUT / "MC_Fig2_Exceedance_Curves.png")
plt.close()
print("  → MC_Fig2_Exceedance_Curves.png")


# =============================================================================
# 5.  SENSITIVITY ANALYSIS
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 3 — SENSITIVITY ANALYSIS")
print("─" * 70)

PARAM_LABELS = {
    "IngR":  "Soil Ingestion Rate",
    "InhR":  "Inhalation Rate",
    "SA":    "Skin Surface Area",
    "AF":    "Skin Adherence Factor",
    "ABS":   "Dermal Absorption",
    "EF":    "Exposure Frequency",
    "ED":    "Exposure Duration",
    "BW":    "Body Weight",
    "AT_nc": "Avg. Time (Non-cancer)",
    "AT_ca": "Avg. Time (Cancer)",
    "PEF":   "Particle Emission Factor",
}

def spearman_sensitivity(param_samples, risk_array, exclude_keys=None):
    """
    Compute Spearman rank correlation between each sampled parameter
    and the risk output. Returns sorted DataFrame.
    """
    rows = []
    for key, arr in param_samples.items():
        if exclude_keys and key in exclude_keys:
            continue
        if np.std(arr) == 0:
            continue
        r, p = stats.spearmanr(arr, risk_array)
        rows.append({
            "Parameter": PARAM_LABELS.get(key, key),
            "Spearman_r": r,
            "p_value":    p,
            "Contribution_%": r**2 * 100,   # variance explained (%)
        })
    df_sens = pd.DataFrame(rows)
    df_sens = df_sens.reindex(
        df_sens["Contribution_%"].abs().sort_values(ascending=False).index
    )
    return df_sens

print("\n  Computing Spearman sensitivity for HI and ILCR (Adult & Child) ...")
sens_HI_adult   = spearman_sensitivity(params_adult,  HI_adult,   exclude_keys=["CF"])
sens_HI_child   = spearman_sensitivity(params_child,  HI_child,   exclude_keys=["CF"])
sens_ILCR_adult = spearman_sensitivity(params_adult,  ILCR_adult, exclude_keys=["CF"])
sens_ILCR_child = spearman_sensitivity(params_child,  ILCR_child, exclude_keys=["CF"])

for lbl, df_s in [("HI Adult", sens_HI_adult), ("HI Child", sens_HI_child),
                   ("ILCR Adult", sens_ILCR_adult), ("ILCR Child", sens_ILCR_child)]:
    print(f"\n  [{lbl}] Top 5 sensitivity drivers:")
    print(df_s.head(5)[["Parameter","Spearman_r","Contribution_%"]].to_string(index=False))

# ── Fig SENS-1: Tornado charts (2×2) ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=LIGHT)
fig.suptitle("Figure SENS-1 — Sensitivity Tornado Charts\n"
             "Spearman Rank Correlation of Exposure Parameters vs. Risk Output",
             fontsize=13, fontweight="bold")

plot_specs = [
    (axes[0,0], sens_HI_adult,   "HI — Adult",  NAVY),
    (axes[0,1], sens_HI_child,   "HI — Child",  CRIMSON),
    (axes[1,0], sens_ILCR_adult, "ILCR — Adult",TEAL),
    (axes[1,1], sens_ILCR_child, "ILCR — Child",AMBER),
]

for ax, df_s, title, color in plot_specs:
    top = df_s.head(8).copy()[::-1]   # top 8, reversed for horizontal tornado
    r_vals = top["Spearman_r"].values
    labels = top["Parameter"].values
    contrib = top["Contribution_%"].values

    bar_colors = [color if r >= 0 else CRIMSON for r in r_vals]
    bars = ax.barh(range(len(labels)), r_vals, color=bar_colors, alpha=0.85,
                   edgecolor="white", linewidth=0.5)

    for i, (bar, r, c) in enumerate(zip(bars, r_vals, contrib)):
        x_pos = r + (0.01 if r >= 0 else -0.01)
        ha    = "left" if r >= 0 else "right"
        ax.text(x_pos, i, f"{r:+.3f} ({c:.1f}%)",
                va="center", ha=ha, fontsize=7.5)

    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Spearman r (positive = risk-increasing)")
    ax.set_title(title, fontweight="bold")
    ax.set_facecolor(LIGHT)
    ax.set_xlim(-1.05, 1.05)

    # Significance asterisks
    p_vals = top["p_value"].values[::-1]
    for i, p in enumerate(p_vals):
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
        if sig:
            ax.text(1.02, i, sig, va="center", fontsize=9, color=CRIMSON,
                    transform=ax.get_yaxis_transform())

plt.tight_layout()
fig.savefig(OUT / "SENS_Fig1_Tornado_Charts.png")
plt.close()
print("\n  → SENS_Fig1_Tornado_Charts.png")

# ── Fig SENS-2: Variance decomposition pie charts ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=LIGHT)
fig.suptitle("Figure SENS-2 — Variance Decomposition: HI and ILCR (Adult)\n"
             "Relative contribution of each parameter to output variance",
             fontsize=12, fontweight="bold")

for ax, df_s, title in [(axes[0], sens_HI_adult,   "HI — Adult"),
                         (axes[1], sens_ILCR_adult, "ILCR — Adult")]:
    top_n = 6
    top   = df_s.head(top_n).copy()
    rest  = df_s.iloc[top_n:]["Contribution_%"].sum()
    labels  = list(top["Parameter"]) + (["Others"] if rest > 0 else [])
    sizes   = list(top["Contribution_%"]) + ([rest] if rest > 0 else [])
    colors  = plt.cm.tab10(np.linspace(0, 0.8, len(labels)))

    wedges, texts, autotexts = ax.pie(
        sizes, labels=None, colors=colors, autopct="%1.1f%%",
        startangle=140, pctdistance=0.78,
        wedgeprops=dict(linewidth=0.8, edgecolor="white")
    )
    for at in autotexts:
        at.set_fontsize(8)

    ax.legend(wedges, labels, loc="lower center", bbox_to_anchor=(0.5, -0.15),
              ncol=2, fontsize=8)
    ax.set_title(title, fontweight="bold", pad=15)

plt.tight_layout()
fig.savefig(OUT / "SENS_Fig2_Variance_Decomposition.png")
plt.close()
print("  → SENS_Fig2_Variance_Decomposition.png")


# =============================================================================
# 6.  MCMC — METROPOLIS-HASTINGS ENSEMBLE SAMPLER
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 4 — MCMC SAMPLING  (Metropolis-Hastings Ensemble)")
print("─" * 70)

# """
# MCMC strategy:
#   We treat HI (Adult) as the primary observable.
#   We sample the posterior distribution over the 4 highest-sensitivity
#   exposure parameters simultaneously, conditioning on the mean HI
#   computed from actual soil concentrations.

#   The likelihood function:
#     log L(θ) = log p(HI_observed | θ)   where θ = (IngR, ED, EF, BW)
#     Modelled as Gaussian:
#       HI_obs ~ N( HI_model(θ), σ_obs )

#   The prior:
#     Each parameter has its lognormal / normal prior from USEPA values.

#   This is NOT a standard Bayesian exposure model (that would require
#   biomonitoring data) — it is a posterior predictive sensitivity sampler
#   that maps the parameter uncertainty directly to HI uncertainty, with
#   MCMC providing convergence diagnostics and autocorrelation structure
#   that Monte Carlo alone cannot produce.
# """

# Parameters to sample in MCMC (4D): IngR, ED, EF, BW
# These are the top sensitivity drivers from the analysis above.
MCMC_PARAMS = ["IngR", "ED", "EF", "BW"]
N_WALKERS   = 32        # number of independent chains (ensemble)
N_STEPS     = 20000     # steps per walker (total samples = N_WALKERS × N_STEPS)
N_BURNIN    = 5000       # burn-in steps to discard
STEP_SIZE   = 0.08      # proposal std as fraction of prior std

# HI_obs: target / "observed" HI from deterministic calculation (mean of MC)
HI_obs = HI_adult.mean()
sigma_obs = HI_adult.std()   # observation noise = MC variance

print(f"\n  Target HI (observed mean)  : {HI_obs:.4f}")
print(f"  Observation sigma          : {sigma_obs:.4f}")
print(f"  MCMC walkers               : {N_WALKERS}")
print(f"  Steps per walker           : {N_STEPS}")
print(f"  Burn-in steps              : {N_BURNIN}")
print(f"  Sampled parameters         : {MCMC_PARAMS}")

# ── Prior log-probabilities ───────────────────────────────────────────────────
PRIOR_ADULT = {
    "IngR":  ("lognormal", np.log(100), 0.26),
    "ED":    ("normal",    25,           6   ),
    "EF":    ("triangular",200,          225, 250),
    "BW":    ("normal",    70,           10  ),
}

def log_prior(theta):
    """Log prior probability of parameter vector θ = [IngR, ED, EF, BW]."""
    ingr, ed, ef, bw = theta
    if ingr <= 0 or ed <= 0 or ef < 180 or ef > 260 or bw <= 0:
        return -np.inf

    lp  = stats.lognorm.logpdf(ingr, s=0.26, scale=np.exp(np.log(100)))
    lp += stats.norm.logpdf(ed, loc=25, scale=6)
    c   = (225 - 200) / (250 - 200)
    lp += stats.triang.logpdf(ef, c=c, loc=200, scale=50)
    lp += stats.norm.logpdf(bw, loc=70, scale=10)
    return lp


def hi_model(theta, concs):
    """
    Deterministic HI forward model given parameter vector θ.
    Uses mean of all routes (ing + inh + der) summed across metals.
    """
    ingr, ed, ef, bw = theta
    EF_val  = ef
    ED_val  = ed
    BW_val  = bw
    AT_nc   = 25 * 365
    CF_val  = 1e-6
    InhR    = 20.0
    PEF_val = 1.36e9
    SA_val  = 1700
    AF_val  = 0.07
    ABS_val = 0.03

    HI_sum = 0.0
    for metal, C in concs.items():
        rfd = RFD.get(metal, None)
        if rfd is None:
            continue
        ADD_ing = (C * ingr  * EF_val * ED_val * CF_val) / (BW_val * AT_nc)
        ADD_inh = (C * InhR  * EF_val * ED_val)          / (BW_val * AT_nc * PEF_val)
        ADD_der = (C * AF_val * SA_val * ABS_val * EF_val * ED_val * CF_val) / (BW_val * AT_nc)
        if rfd["ing"] > 0:  HI_sum += ADD_ing / rfd["ing"]
        if rfd["inh"] > 0:  HI_sum += ADD_inh / rfd["inh"]
        if rfd["der"] > 0:  HI_sum += ADD_der / rfd["der"]
    return HI_sum


def log_likelihood(theta, concs, hi_observed, sigma):
    """Gaussian log-likelihood: HI_obs ~ N(HI_model(θ), σ)."""
    hi_pred = hi_model(theta, concs)
    if not np.isfinite(hi_pred):
        return -np.inf
    return stats.norm.logpdf(hi_observed, loc=hi_pred, scale=sigma)


def log_posterior(theta, concs, hi_observed, sigma):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, concs, hi_observed, sigma)


# ── Metropolis-Hastings walker ────────────────────────────────────────────────
def run_walker(theta0, n_steps, concs, hi_obs, sigma, step_scale, seed):
    """Single Metropolis-Hastings chain. Returns (chain, log_prob_chain)."""
    rng_w = np.random.default_rng(seed)
    dim   = len(theta0)
    chain    = np.zeros((n_steps, dim))
    log_prob = np.zeros(n_steps)
    current  = theta0.copy()
    lp_curr  = log_posterior(current, concs, hi_obs, sigma)
    n_accept = 0

    for i in range(n_steps):
        # Proposal: Gaussian random walk, scaled per-parameter
        proposal_std = np.array([
            100 * step_scale,  # IngR
            6   * step_scale,  # ED
            15  * step_scale,  # EF
            10  * step_scale,  # BW
        ])
        proposal = current + rng_w.normal(0, proposal_std)
        lp_prop  = log_posterior(proposal, concs, hi_obs, sigma)

        log_alpha = lp_prop - lp_curr
        if np.log(rng_w.random()) < log_alpha:
            current   = proposal
            lp_curr   = lp_prop
            n_accept += 1

        chain[i]    = current
        log_prob[i] = lp_curr

    acceptance = n_accept / n_steps
    return chain, log_prob, acceptance


# ── Run ensemble of walkers ───────────────────────────────────────────────────
# Initialise walkers near the prior mean with small perturbations
theta0_center = np.array([100.0, 25.0, 225.0, 70.0])  # [IngR, ED, EF, BW]

print(f"\n  Running {N_WALKERS} MCMC walkers × {N_STEPS} steps ...")
all_chains    = np.zeros((N_WALKERS, N_STEPS, len(MCMC_PARAMS)))
all_logprobs  = np.zeros((N_WALKERS, N_STEPS))
all_acceptance = []

base_rng = np.random.default_rng(SEED)
init_seeds    = base_rng.integers(0, 2**31, size=N_WALKERS)

# Small random offsets for each walker's starting position
init_offsets  = base_rng.normal(0, 1, size=(N_WALKERS, len(MCMC_PARAMS))) * \
                np.array([10.0, 2.0, 10.0, 5.0])

for w in range(N_WALKERS):
    theta_init = theta0_center + init_offsets[w]
    theta_init = np.clip(theta_init, [1, 1, 185, 10], [600, 50, 255, 120])
    chain, logprob, acc = run_walker(
        theta_init, N_STEPS, concs, HI_obs, sigma_obs,
        STEP_SIZE, seed=int(init_seeds[w])
    )
    all_chains[w]   = chain
    all_logprobs[w] = logprob
    all_acceptance.append(acc)
    print(f"    Walker {w+1:2d}/{N_WALKERS} — acceptance rate: {acc:.3f}")

print(f"\n  Mean acceptance rate: {np.mean(all_acceptance):.3f}"
      f"  (ideal: 0.20–0.50)")

# Post burn-in samples
post_chains   = all_chains[:, N_BURNIN:, :]   # (N_WALKERS, N_POST, 4)
post_logprobs = all_logprobs[:, N_BURNIN:]
flat_samples  = post_chains.reshape(-1, len(MCMC_PARAMS))

print(f"\n  Total post-burnin samples: {flat_samples.shape[0]:,}")
print(f"\n  Posterior summary (post-burnin):")
for i, p in enumerate(MCMC_PARAMS):
    s = flat_samples[:, i]
    print(f"    {p:6s}: mean={s.mean():.3f}  median={np.median(s):.3f}  "
          f"std={s.std():.3f}  P5={np.percentile(s,5):.3f}  P95={np.percentile(s,95):.3f}")


# =============================================================================
# 7.  MCMC FIGURES
# =============================================================================
print("\n  Generating MCMC figures ...")

# ── Fig MCMC-1: TRACE PLOTS ───────────────────────────────────────────────────
# One panel per parameter, all walkers overlaid, burn-in shaded
fig, axes = plt.subplots(len(MCMC_PARAMS) + 1, 1,
                          figsize=(16, 4 * (len(MCMC_PARAMS) + 1)),
                          facecolor=LIGHT)
fig.suptitle("Figure MCMC-1 — Trace Plots of MCMC Walkers\n"
             "All 16 walkers shown; grey region = burn-in period",
             fontsize=13, fontweight="bold")

walker_colors = plt.cm.tab20(np.linspace(0, 1, N_WALKERS))
steps = np.arange(N_STEPS)

for pi, (ax, param) in enumerate(zip(axes[:-1], MCMC_PARAMS)):
    for w in range(N_WALKERS):
        ax.plot(steps, all_chains[w, :, pi],
                color=walker_colors[w], alpha=0.55, lw=0.6)
    # Mean across walkers
    ax.plot(steps, all_chains[:, :, pi].mean(axis=0),
            color="black", lw=1.8, alpha=0.9, label="Ensemble mean")
    # Burn-in shading
    ax.axvspan(0, N_BURNIN, color=CRIMSON, alpha=0.08, label=f"Burn-in ({N_BURNIN} steps)")
    ax.axvline(N_BURNIN, color=CRIMSON, lw=1.2, linestyle="--")

    ax.set_ylabel(PARAM_LABELS.get(param, param), fontsize=9)
    ax.set_facecolor(LIGHT)
    if pi == 0:
        ax.legend(fontsize=8, loc="upper right")
    if pi < len(MCMC_PARAMS) - 1:
        ax.set_xticklabels([])

# Log-probability trace
ax_lp = axes[-1]
for w in range(N_WALKERS):
    ax_lp.plot(steps, all_logprobs[w],
               color=walker_colors[w], alpha=0.55, lw=0.6)
ax_lp.plot(steps, all_logprobs.mean(axis=0),
           color="black", lw=1.8, label="Ensemble mean log-prob")
ax_lp.axvspan(0, N_BURNIN, color=CRIMSON, alpha=0.08)
ax_lp.axvline(N_BURNIN, color=CRIMSON, lw=1.2, linestyle="--")
ax_lp.set_ylabel("log P(θ | data)", fontsize=9)
ax_lp.set_xlabel("MCMC Step")
ax_lp.set_facecolor(LIGHT)
ax_lp.legend(fontsize=8, loc="lower right")

plt.tight_layout()
fig.savefig(OUT / "MCMC_Fig1_Trace_Plots.png")
plt.close()
print("  → MCMC_Fig1_Trace_Plots.png")

# ── Fig MCMC-2: AUTOCORRELATION PLOTS ────────────────────────────────────────
def autocorr(chain_1d, max_lag=200):
    """Normalised autocorrelation of a 1D chain."""
    n   = len(chain_1d)
    x   = chain_1d - chain_1d.mean()
    c0  = np.dot(x, x) / n
    lags = np.arange(0, max_lag + 1)
    acf = np.array([np.dot(x[:n-l], x[l:]) / (n * c0) for l in lags])
    return lags, acf

fig, axes = plt.subplots(2, 2, figsize=(14, 9), facecolor=LIGHT)
fig.suptitle("Figure MCMC-2 — Autocorrelation Functions (ACF) of MCMC Chains\n"
             "Thin grey lines = individual walkers; Blue = mean ACF across walkers",
             fontsize=12, fontweight="bold")

MAX_LAG = 300
for ax, param, pi in zip(axes.flatten(), MCMC_PARAMS, range(len(MCMC_PARAMS))):
    mean_acf = np.zeros(MAX_LAG + 1)
    for w in range(N_WALKERS):
        post_chain_1d = post_chains[w, :, pi]
        lags, acf = autocorr(post_chain_1d, max_lag=MAX_LAG)
        ax.plot(lags, acf, color="grey", alpha=0.35, lw=0.8)
        mean_acf += acf / N_WALKERS

    ax.plot(lags, mean_acf, color=TEAL, lw=2.0, label="Mean ACF")
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(1/np.e, color=CRIMSON, linestyle="--", lw=1.2, label="1/e threshold")

    # Integrated autocorrelation time estimate
    try:
        iat = 1 + 2 * np.sum(mean_acf[1:np.argmax(mean_acf < 0.05) or 50])
        ax.text(0.97, 0.88, f"τ_int ≈ {iat:.1f} steps",
                transform=ax.transAxes, ha="right", fontsize=8,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    except Exception:
        pass

    ax.set_title(PARAM_LABELS.get(param, param), fontweight="bold")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")
    ax.set_facecolor(LIGHT)
    ax.set_xlim(0, MAX_LAG)
    ax.set_ylim(-0.3, 1.05)
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUT / "MCMC_Fig2_Autocorrelation.png")
plt.close()
print("  → MCMC_Fig2_Autocorrelation.png")

# ── Fig MCMC-3: POSTERIOR DENSITY PLOTS (per parameter) ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9), facecolor=LIGHT)
fig.suptitle("Figure MCMC-3 — MCMC Posterior Distributions\n"
             "Compared against prior distribution (dashed)",
             fontsize=12, fontweight="bold")

PRIOR_ARGS = {
    "IngR": dict(dist=stats.lognorm, args=(0.26,), kwargs={"scale": np.exp(np.log(100))}),
    "ED":   dict(dist=stats.norm,    args=(),      kwargs={"loc": 25,  "scale": 6}),
    "EF":   dict(dist=stats.triang,  args=((225-200)/(250-200),), kwargs={"loc": 200, "scale": 50}),
    "BW":   dict(dist=stats.norm,    args=(),      kwargs={"loc": 70,  "scale": 10}),
}

param_colors = [NAVY, CRIMSON, TEAL, AMBER]

for ax, param, pi, color in zip(axes.flatten(), MCMC_PARAMS,
                                 range(len(MCMC_PARAMS)), param_colors):
    posterior = flat_samples[:, pi]
    x_min = np.percentile(posterior, 0.5)
    x_max = np.percentile(posterior, 99.5)
    x_plot = np.linspace(x_min, x_max, 300)

    # Posterior KDE
    kde = stats.gaussian_kde(posterior)
    ax.fill_between(x_plot, kde(x_plot), alpha=0.45, color=color, label="Posterior")
    ax.plot(x_plot, kde(x_plot), color=color, lw=2)

    # Prior overlay
    pa = PRIOR_ARGS[param]
    try:
        prior_pdf = pa["dist"].pdf(x_plot, *pa["args"], **pa["kwargs"])
        prior_pdf_norm = prior_pdf / prior_pdf.max() * kde(x_plot).max()
        ax.plot(x_plot, prior_pdf_norm, color="grey", lw=1.8,
                linestyle="--", label="Prior (normalised)")
    except Exception:
        pass

    # Credible interval (95% HDI)
    ci_lo, ci_hi = np.percentile(posterior, [2.5, 97.5])
    ax.axvline(ci_lo, color=color, linestyle=":", lw=1.5, alpha=0.8)
    ax.axvline(ci_hi, color=color, linestyle=":", lw=1.5, alpha=0.8)
    ax.axvline(np.median(posterior), color="black", lw=1.5, label=f"Median={np.median(posterior):.2f}")
    ax.fill_between(x_plot, kde(x_plot),
                    where=(x_plot >= ci_lo) & (x_plot <= ci_hi),
                    alpha=0.18, color=color, label=f"95% CI [{ci_lo:.2f}, {ci_hi:.2f}]")

    ax.set_title(PARAM_LABELS.get(param, param), fontweight="bold")
    ax.set_xlabel(param)
    ax.set_ylabel("Density")
    ax.set_facecolor(LIGHT)
    ax.legend(fontsize=7.5)

plt.tight_layout()
fig.savefig(OUT / "MCMC_Fig3_Posterior_Densities.png")
plt.close()
print("  → MCMC_Fig3_Posterior_Densities.png")

# ── Fig MCMC-4: CORNER / PAIR PLOT (2D joint posteriors) ─────────────────────
fig, axes = plt.subplots(len(MCMC_PARAMS), len(MCMC_PARAMS),
                          figsize=(13, 13), facecolor=LIGHT)
fig.suptitle("Figure MCMC-4 — Corner Plot: Joint Posterior Distributions\n"
             "(diagonal = marginal KDE; lower triangle = 2D joint density)",
             fontsize=12, fontweight="bold")

for i in range(len(MCMC_PARAMS)):
    for j in range(len(MCMC_PARAMS)):
        ax = axes[i, j]
        if i == j:
            # Diagonal: marginal KDE
            data_1d = flat_samples[:, i]
            xs = np.linspace(data_1d.min(), data_1d.max(), 200)
            kde = stats.gaussian_kde(data_1d)
            ax.fill_between(xs, kde(xs), alpha=0.5, color=param_colors[i])
            ax.plot(xs, kde(xs), color=param_colors[i], lw=1.5)
            ax.set_xlim(data_1d.min(), data_1d.max())
            ax.set_yticks([])
            ax.set_title(MCMC_PARAMS[i], fontsize=9, fontweight="bold")

        elif i > j:
            # Lower triangle: 2D hexbin density
            x_data = flat_samples[:, j]
            y_data = flat_samples[:, i]
            ax.hexbin(x_data, y_data, gridsize=30,
                      cmap="YlOrRd", mincnt=1, alpha=0.85)
            # Add contour lines
            try:
                xx, yy = np.mgrid[x_data.min():x_data.max():60j,
                                   y_data.min():y_data.max():60j]
                positions = np.vstack([xx.ravel(), yy.ravel()])
                values    = np.vstack([x_data, y_data])
                kernel    = stats.gaussian_kde(values)
                z         = kernel(positions).reshape(xx.shape)
                levels    = np.percentile(z[z > 0], [25, 50, 75, 90])
                ax.contour(xx, yy, z, levels=levels,
                           colors="white", linewidths=0.6, alpha=0.6)
            except Exception:
                pass

            # Correlation annotation
            r, _ = stats.pearsonr(x_data, y_data)
            ax.text(0.95, 0.05, f"r={r:.2f}", transform=ax.transAxes,
                    ha="right", fontsize=7.5, color="white",
                    bbox=dict(facecolor="black", alpha=0.4, boxstyle="round"))

        else:
            # Upper triangle: blank
            ax.set_visible(False)
            continue

        # Axis labels only on edges
        if i == len(MCMC_PARAMS) - 1:
            ax.set_xlabel(MCMC_PARAMS[j], fontsize=8)
        else:
            ax.set_xticklabels([])
        if j == 0 and i > 0:
            ax.set_ylabel(MCMC_PARAMS[i], fontsize=8)
        else:
            ax.set_yticklabels([])

        ax.set_facecolor(LIGHT)
        ax.tick_params(labelsize=7)

plt.tight_layout()
fig.savefig(OUT / "MCMC_Fig4_Corner_Plot.png")
plt.close()
print("  → MCMC_Fig4_Corner_Plot.png")

# ── Fig MCMC-5: HI POSTERIOR PREDICTIVE DISTRIBUTION ─────────────────────────
# Draw HI predictions from posterior parameter samples
print("\n  Computing HI posterior predictive distribution ...")
n_post = min(flat_samples.shape[0], 3000)
post_idx = np.random.choice(flat_samples.shape[0], n_post, replace=False)
HI_posterior = np.array([
    hi_model(flat_samples[i], concs) for i in post_idx
])

fig, ax = plt.subplots(figsize=(10, 6), facecolor=LIGHT)
fig.suptitle("Figure MCMC-5 — Posterior Predictive Distribution of HI (Adult)\n"
             "MCMC-derived uncertainty vs. Monte Carlo distribution",
             fontsize=12, fontweight="bold")

# MC distribution
ax.hist(HI_adult, bins=60, density=True, color=NAVY,
        alpha=0.40, label="Monte Carlo (N=10,000)")

# MCMC posterior predictive
ax.hist(HI_posterior, bins=40, density=True, color=CRIMSON,
        alpha=0.45, label="MCMC Posterior Predictive")

# KDEs
for data, color, lw in [(HI_adult, NAVY, 2.0), (HI_posterior, CRIMSON, 2.0)]:
    kde = stats.gaussian_kde(data)
    x_k = np.linspace(min(data), max(data), 300)
    ax.plot(x_k, kde(x_k), color=color, lw=lw)

ax.axvline(1.0, color="black", lw=1.8, linestyle="--", label="HI = 1 (threshold)")
ax.axvline(np.median(HI_posterior), color=CRIMSON, lw=1.5, linestyle=":",
           label=f"MCMC median = {np.median(HI_posterior):.3f}")
ax.axvline(np.median(HI_adult), color=NAVY, lw=1.5, linestyle=":",
           label=f"MC median = {np.median(HI_adult):.3f}")

ci_lo, ci_hi = np.percentile(HI_posterior, [2.5, 97.5])
ax.axvspan(ci_lo, ci_hi, alpha=0.10, color=CRIMSON,
           label=f"MCMC 95% CI [{ci_lo:.3f}, {ci_hi:.3f}]")

ax.set_xlabel("Hazard Index (HI) — Adult")
ax.set_ylabel("Probability Density")
ax.set_facecolor(LIGHT)
ax.legend(fontsize=8.5)

plt.tight_layout()
fig.savefig(OUT / "MCMC_Fig5_HI_Posterior_Predictive.png")
plt.close()
print("  → MCMC_Fig5_HI_Posterior_Predictive.png")


# =============================================================================
# 8.  GELMAN-RUBIN CONVERGENCE DIAGNOSTIC
# =============================================================================
print("\n" + "─" * 70)
print("  CONVERGENCE DIAGNOSTICS — Gelman-Rubin R̂ statistic")
print("─" * 70)

def gelman_rubin(chains):
    """
    Gelman-Rubin R̂ diagnostic.
    chains: (n_walkers, n_steps) array (post-burnin, single parameter)
    R̂ < 1.1 indicates convergence.
    """
    m, n = chains.shape
    psi_bar_j = chains.mean(axis=1)          # per-chain mean
    psi_bar   = psi_bar_j.mean()             # grand mean
    B = n / (m - 1) * np.sum((psi_bar_j - psi_bar)**2)  # between-chain var
    W = np.mean(chains.var(axis=1, ddof=1))              # within-chain var
    var_plus = (n - 1) / n * W + B / n
    R_hat = np.sqrt(var_plus / W) if W > 0 else np.nan
    return R_hat, B, W

print(f"\n  {'Parameter':<22} {'R̂':>8}  {'B':>12}  {'W':>12}  Converged?")
print(f"  {'-'*70}")
for pi, param in enumerate(MCMC_PARAMS):
    chains_pi = post_chains[:, :, pi]
    r_hat, B, W = gelman_rubin(chains_pi)
    converged   = "✓  Yes" if r_hat < 1.1 else "✗  No  (need more steps)"
    print(f"  {PARAM_LABELS.get(param, param):<22} {r_hat:>8.4f}  {B:>12.4f}  {W:>12.4f}  {converged}")


# =============================================================================
# 9.  EXPORT RESULTS TABLE
# =============================================================================
results_summary = pd.DataFrame({
    "Metric": [
        "MC HI Adult — Mean", "MC HI Adult — SD", "MC HI Adult — P5",
        "MC HI Adult — P95", "P(HI_adult > 1) %",
        "MC HI Child — Mean",  "MC HI Child — SD",  "MC HI Child — P5",
        "MC HI Child — P95",  "P(HI_child > 1) %",
        "MC ILCR Adult — Mean", "MC ILCR Adult — P95", "P(ILCR_adult > 1e-4) %",
        "MC ILCR Child — Mean", "MC ILCR Child — P95", "P(ILCR_child > 1e-4) %",
        "MCMC HI Posterior Mean", "MCMC HI 95% CI Low", "MCMC HI 95% CI High",
        "MCMC Mean Acceptance Rate",
    ],
    "Value": [
        HI_adult.mean(), HI_adult.std(), np.percentile(HI_adult,5),
        np.percentile(HI_adult,95), (HI_adult>1).mean()*100,
        HI_child.mean(),  HI_child.std(),  np.percentile(HI_child,5),
        np.percentile(HI_child,95),  (HI_child>1).mean()*100,
        ILCR_adult.mean(), np.percentile(ILCR_adult,95), (ILCR_adult>1e-4).mean()*100,
        ILCR_child.mean(),  np.percentile(ILCR_child,95),  (ILCR_child>1e-4).mean()*100,
        np.mean(HI_posterior), np.percentile(HI_posterior,2.5), np.percentile(HI_posterior,97.5),
        np.mean(all_acceptance),
    ]
})

with pd.ExcelWriter(OUT / "ProbabilisticRisk_Results.xlsx", engine="openpyxl") as writer:
    results_summary.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({"HI_Adult": HI_adult, "HI_Child": HI_child,
                  "ILCR_Adult": ILCR_adult, "ILCR_Child": ILCR_child})\
      .to_excel(writer, sheet_name="MC_Raw_Samples", index=False)
    sens_HI_adult.to_excel(writer,   sheet_name="Sensitivity_HI_Adult",   index=False)
    sens_ILCR_adult.to_excel(writer, sheet_name="Sensitivity_ILCR_Adult", index=False)
    sens_HI_child.to_excel(writer,   sheet_name="Sensitivity_HI_Child",   index=False)
    sens_ILCR_child.to_excel(writer, sheet_name="Sensitivity_ILCR_Child", index=False)
    pd.DataFrame(flat_samples, columns=MCMC_PARAMS)\
      .to_excel(writer, sheet_name="MCMC_Posterior_Samples", index=False)

print(f"\n  → ProbabilisticRisk_Results.xlsx saved")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE")
print("=" * 70)
print(f"""
  Monte Carlo:  {N_ITER:,} iterations | numpy + scipy (no Crystal Ball needed)
  MCMC:         {N_WALKERS} walkers × {N_STEPS} steps | Metropolis-Hastings from scratch
                (no emcee required — zero extra installs)

  Figures saved to outputs/:
  ┌─────────────────────────────────────────────────────────────────┐
  │  MC_Fig1_PDF_CDF_HI_ILCR.png       PDF + CDF for HI & ILCR    │
  │  MC_Fig2_Exceedance_Curves.png      Exceedance probability      │
  │  SENS_Fig1_Tornado_Charts.png       Sensitivity tornado (2×2)   │
  │  SENS_Fig2_Variance_Decomposition.png  Variance pie charts      │
  │  MCMC_Fig1_Trace_Plots.png         ★ All 16 walker traces       │
  │  MCMC_Fig2_Autocorrelation.png     ★ ACF per parameter          │
  │  MCMC_Fig3_Posterior_Densities.png ★ Posterior vs prior KDE     │
  │  MCMC_Fig4_Corner_Plot.png         ★ Joint 2D posteriors        │
  │  MCMC_Fig5_HI_Posterior_Predictive.png ★ MC vs MCMC comparison  │
  │  ProbabilisticRisk_Results.xlsx     All tables (7 sheets)       │
  └─────────────────────────────────────────────────────────────────┘
  ★ = MCMC-specific outputs novel vs. standard Crystal Ball approach
""")

  PROBABILISTIC HEALTH RISK ASSESSMENT PIPELINE
  Cape Coast Landfill — Monte Carlo + MCMC

  Metals analysed    : ['As', 'Cd', 'Cr', 'Cu', 'Hg', 'Ni', 'Pb', 'Zn']
  Carcinogenic metals: ['As', 'Cd', 'Cr', 'Ni', 'Pb']
  Monte Carlo runs   : 10,000

──────────────────────────────────────────────────────────────────────
  SECTION 2 — MONTE CARLO SIMULATION  (10,000 iterations)
──────────────────────────────────────────────────────────────────────

  Mean metal concentrations used (mg/kg):
    As  : 7.0222
    Cd  : 3.6317
    Cr  : 90.8696
    Cu  : 117.0913
    Hg  : 2.6401
    Ni  : 26.6962
    Pb  : 27.7783
    Zn  : 72.2101

  Running Monte Carlo (n=10,000) ...
  ✓ Monte Carlo complete.

  [ADULT]
    HI  — Mean: 0.1820 | Median: 0.1612 | SD: 0.0961 | P5: 0.0701 | P95: 0.3659
    ILCR— Mean: 6.0780e-05 | Median: 5.2471e-05 | SD: 3.5470e-05 | P5: 2.1002e-05 | P95: 1.2890e-04
    P(HI > 1)          : 0.0%
    P(ILCR > 1×10⁻⁴)  : 12.0%

  [CHILD]
    HI  — Mean: 1.2974 | Median: 1.1511 

In [ ]:


"""
=============================================================================
Source Apportionment Pipeline — Cape Coast Landfill Heavy Metal Study
=============================================================================
Sections:
  1.  Data Loading & Preprocessing
  2.  PCA / Factor Analysis
        — Scree plot & cumulative variance
        — Biplot (PC1 vs PC2 and PC1 vs PC3)
        — Factor loading heatmap
        — Site scores plot
        — Varimax-rotated loadings
        — Anthropogenic vs. geogenic classification
  3.  Positive Matrix Factorization  (PMF — via NMF, same algorithm as
        US EPA PMF v5.0 / PMF5)
        — Bootstrap stability (100 runs) for robust source profile estimation
        — Source profiles (fingerprints)
        — Source contribution per site
        — Source contribution pie charts
        — Bootstrapped uncertainty envelopes
        — PMF vs. measured reconstruction plot
  4.  Source Attribution Summary
        — Combined PCA + PMF interpretation table
        — Source profile comparison heatmap

REQUIRED PACKAGES (install once):
  pip install numpy scipy scikit-learn matplotlib pandas openpyxl seaborn

NOTE ON PMF:
  US EPA PMF v5.0 is a Windows GUI tool that implements Weighted Least
  Squares NMF with uncertainty-based weighting. This pipeline replicates
  its core algorithm:
    min  ||W⁻¹(X − G·F)||²_F    subject to  G ≥ 0, F ≥ 0
  where W are uncertainty weights (signal-to-noise ratio). The bootstrap
  procedure replicates PMF5's "BS" uncertainty estimation mode.
  Results are directly interpretable and comparable to EPA PMF5 output.
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis, NMF
from sklearn.utils import resample
from pathlib import Path

# ── Output directory ─────────────────────────────────────────────────────────
OUT = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         12,
    "axes.titlesize":    12,
    "axes.labelsize":    12,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

NAVY    = "#2E4057"
CRIMSON = "#E84855"
TEAL    = "#00B4D8"
AMBER   = "#F4A261"
GREEN   = "#2A9D8F"
PURPLE  = "#9B5DE5"
ORANGE  = "#FB8500"
LIGHT   = "#F7F9FB"

METALS = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]
N_FACTORS = 4    # expected source types (EPA PMF guidance: start with n+1, reduce)

# Source hypotheses for interpretation (updated after seeing data)
SOURCE_HYPOTHESES = {
    0: {"name": "Traffic & Combustion",    "color": CRIMSON, "type": "Anthropogenic",
        "markers": ["Pb", "Cu", "Zn"],     "icon": "🚗"},
    1: {"name": "Industrial / E-waste",    "color": AMBER,   "type": "Anthropogenic",
        "markers": ["Cd", "Hg", "Ni"],     "icon": "🏭"},
    2: {"name": "Geogenic / Crustal",      "color": GREEN,   "type": "Geogenic",
        "markers": ["Cr", "As"],           "icon": "🪨"},
    3: {"name": "Agricultural / Biomass",  "color": PURPLE,  "type": "Mixed",
        "markers": ["Zn", "Cu", "As"],     "icon": "🌿"},
}

print("=" * 70)
print("  SOURCE APPORTIONMENT PIPELINE — CAPE COAST LANDFILL")
print("  PCA / Factor Analysis  +  Positive Matrix Factorization (PMF)")
print("=" * 70)


# =============================================================================
# 1.  DATA LOADING & PREPROCESSING
# =============================================================================
df_raw = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df_raw.columns = df_raw.columns.str.strip()

# Separate landfill sites and residential background
df_lf  = df_raw[~df_raw["S/N"].str.contains("Residential", na=False)].copy()
df_res = df_raw[ df_raw["S/N"].str.contains("Residential", na=False)].copy()

site_labels = df_lf["S/N"].str.replace("Mean ", "", regex=False).str.strip().tolist()
X_raw       = df_lf[METALS].values.astype(float)   # (n_sites, n_metals)

n_sites, n_metals = X_raw.shape
print(f"\n  Landfill sites    : {n_sites}")
print(f"  Metals            : {METALS}")
print(f"  Residential background excluded from factorisation (used as reference)")

# ── Background reference (residential) for enrichment context ─────────────────
bg_vals = df_res[METALS].values[0] if len(df_res) > 0 else np.ones(n_metals)
print(f"\n  Background (residential) concentrations (mg/kg):")
for m, v in zip(METALS, bg_vals):
    print(f"    {m:4s}: {v:.4f}")

# ── Standardise for PCA (z-score) ─────────────────────────────────────────────
scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)     # zero-mean, unit-variance

# ── Log-transform for PMF (concentrations must be non-negative) ───────────────
# Replace any zeros with half the detection limit (conservative)
X_log = np.log1p(X_raw)                  # log(1 + x) keeps values ≥ 0
X_pmf = X_raw.copy()                     # PMF uses raw concentrations
X_pmf[X_pmf <= 0] = 0.001               # ensure strictly positive


# =============================================================================
# 2.  PCA / FACTOR ANALYSIS
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 2 — PCA / FACTOR ANALYSIS")
print("─" * 70)

# ── 2a. Full PCA ──────────────────────────────────────────────────────────────
pca_full = PCA()
pca_full.fit(X_std)
explained_var   = pca_full.explained_variance_ratio_
cumulative_var  = np.cumsum(explained_var)
eigenvalues     = pca_full.explained_variance_

# ── Determine number of significant PCs (Kaiser criterion: eigenvalue > 1) ───
n_pcs_kaiser = np.sum(eigenvalues > 1)
print(f"\n  Eigenvalues: {eigenvalues.round(3)}")
print(f"  Explained variance (%): {(explained_var*100).round(1)}")
print(f"  Cumulative variance (%): {(cumulative_var*100).round(1)}")
print(f"  Significant PCs (Kaiser criterion, λ>1): {n_pcs_kaiser}")

# ── 2b. PCA with n significant components ────────────────────────────────────
n_pcs = min(n_pcs_kaiser, N_FACTORS)
pca   = PCA(n_components=n_pcs)
scores = pca.fit_transform(X_std)        # site scores in PC space
loadings = pca.components_.T            # (n_metals, n_pcs) — variable loadings

print(f"\n  PCA loadings (unrotated) — top {n_pcs} components:")
load_df = pd.DataFrame(loadings, index=METALS,
                       columns=[f"PC{i+1}" for i in range(n_pcs)])
print(load_df.round(3).to_string())

# ── 2c. Varimax rotation ──────────────────────────────────────────────────────
def varimax(Phi, gamma=1.0, q=100, tol=1e-6):
    """Varimax rotation of factor loading matrix Phi (p × k)."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and abs(d / d_old - 1) < tol:
            break
    return Phi @ R

loadings_rot = varimax(loadings)
load_rot_df  = pd.DataFrame(loadings_rot, index=METALS,
                              columns=[f"RC{i+1}" for i in range(n_pcs)])
print(f"\n  Varimax-rotated loadings:")
print(load_rot_df.round(3).to_string())

# ── Rotated scores ─────────────────────────────────────────────────────────────
rot_matrix  = np.linalg.lstsq(loadings_rot.T, loadings.T, rcond=None)[0]
scores_rot  = scores @ rot_matrix.T if rot_matrix.shape == (n_pcs, n_pcs) else scores

# ── 2d. Factor Analysis (MLE) for comparison ─────────────────────────────────
fa = FactorAnalysis(n_components=n_pcs, rotation="varimax", random_state=42)
fa.fit(X_std)
fa_loadings = pd.DataFrame(fa.components_.T, index=METALS,
                            columns=[f"FA{i+1}" for i in range(n_pcs)])
communalities = np.sum(fa.components_**2, axis=0)
print(f"\n  Factor Analysis (MLE + Varimax) loadings:")
print(fa_loadings.round(3).to_string())

# ── 2e. Communalities ─────────────────────────────────────────────────────────
h2 = np.sum(loadings_rot**2, axis=1)   # sum of squared rotated loadings per variable
print(f"\n  Communalities (h²) — variance explained per metal:")
for m, h in zip(METALS, h2):
    print(f"    {m:4s}: {h:.3f}")


# ── FIGURES: PCA ─────────────────────────────────────────────────────────────

# ── Fig PCA-1: Scree Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=LIGHT)
fig.suptitle("Figure PCA-1 — Scree Plot & Cumulative Variance Explained",
             fontsize=12, fontweight="bold")

ax = axes[0]
pc_nums = np.arange(1, n_metals + 1)
ax.bar(pc_nums, eigenvalues, color=NAVY, alpha=0.7, label="Eigenvalue")
ax.plot(pc_nums, eigenvalues, "o-", color=CRIMSON, lw=2, markersize=7)
ax.axhline(1.0, color=CRIMSON, linestyle="--", lw=1.5, label="Kaiser criterion (λ=1)")
ax.set_xticks(pc_nums)
ax.set_xticklabels([f"PC{i}" for i in pc_nums])
ax.set_xlabel("Principal Component")
ax.set_ylabel("Eigenvalue")
ax.set_title("(a) Scree Plot", fontweight="bold")
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT)
for i, ev in enumerate(eigenvalues):
    ax.text(i + 1, ev + 0.05, f"{ev:.2f}", ha="center", fontsize=8.5, fontweight="bold")

ax2 = axes[1]
bars = ax2.bar(pc_nums, explained_var * 100, color=TEAL, alpha=0.7, label="Individual")
ax2_r = ax2.twinx()
ax2_r.plot(pc_nums, cumulative_var * 100, "s-", color=CRIMSON, lw=2,
           markersize=7, label="Cumulative")
ax2_r.axhline(80, color="grey", linestyle=":", lw=1.2, label="80% threshold")
ax2_r.set_ylabel("Cumulative Variance (%)")
ax2_r.set_ylim(0, 110)
ax2.set_xlabel("Principal Component")
ax2.set_ylabel("Variance Explained (%)")
ax2.set_title("(b) Variance Explained", fontweight="bold")
ax2.set_xticks(pc_nums)
ax2.set_xticklabels([f"PC{i}" for i in pc_nums])
ax2.set_facecolor(LIGHT)
lines1, labs1 = ax2.get_legend_handles_labels()
lines2, labs2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labs1 + labs2, fontsize=9)
for i, ev in enumerate(explained_var * 100):
    ax2.text(i + 1, ev + 0.5, f"{ev:.1f}%", ha="center", fontsize=8)

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig1_Scree_Plot.png")
plt.close()
print("\n  → PCA_Fig1_Scree_Plot.png")

# ── Fig PCA-2: Factor Loading Heatmap (unrotated + rotated) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=LIGHT)
fig.suptitle("Figure PCA-2 — PCA Factor Loading Heatmaps\n"
             "Left: Unrotated PCA   |   Right: Varimax-Rotated",
             fontsize=12, fontweight="bold")

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

for ax, data, title in [
    (axes[0], load_df,     "Unrotated PC Loadings"),
    (axes[1], load_rot_df, "Varimax-Rotated RC Loadings"),
]:
    im = ax.imshow(data.values, cmap="RdBu_r", norm=norm, aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.85, label="Loading")
    ax.set_xticks(range(data.shape[1]))
    ax.set_yticks(range(data.shape[0]))
    ax.set_xticklabels(data.columns, fontsize=10, fontweight="bold")
    ax.set_yticklabels(data.index, fontsize=10)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Component")
    ax.set_ylabel("Heavy Metal")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.values[i, j]
            tc  = "white" if abs(val) > 0.6 else "black"
            bold = "bold" if abs(val) > 0.5 else "normal"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=9, color=tc, fontweight=bold)
    ax.set_facecolor(LIGHT)

# Significance threshold lines
for ax in axes:
    for y in np.arange(-0.5, n_metals, 1):
        ax.axhline(y, color="white", lw=0.5)
    for x in np.arange(-0.5, n_pcs, 1):
        ax.axvline(x, color="white", lw=0.5)

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig2_Loading_Heatmap.png")
plt.close()
print("  → PCA_Fig2_Loading_Heatmap.png")

# ── Fig PCA-3: Biplot (PC1 vs PC2) and (PC1 vs PC3) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor=LIGHT)
fig.suptitle("Figure PCA-3 — PCA Biplots\n"
             "Site scores (circles) and metal loadings (arrows)",
             fontsize=12, fontweight="bold")

pc_pairs = [(0, 1), (0, 2)] if n_pcs >= 3 else [(0, 1), (0, 1)]
pair_titles = ["(a) PC1 vs PC2", "(b) PC1 vs PC3"]

metal_colors_biplot = plt.cm.tab10(np.linspace(0, 0.9, n_metals))
site_colors_biplot  = plt.cm.Set2(np.linspace(0, 1, n_sites))

for ax, (pc_x, pc_y), ptitle in zip(axes, pc_pairs, pair_titles):
    # Scale factor for arrows
    scale = np.max(np.abs(scores[:, [pc_x, pc_y]])) / \
            np.max(np.abs(loadings[:, [pc_x, pc_y]])) * 0.85

    # Site scores
    for i, (sx, sy, sl) in enumerate(zip(scores[:, pc_x], scores[:, pc_y], site_labels)):
        ax.scatter(sx, sy, color=site_colors_biplot[i], s=110, zorder=5,
                   edgecolor="white", linewidth=0.8)
        ax.annotate(sl, (sx, sy), fontsize=8, ha="left",
                    xytext=(4, 4), textcoords="offset points")

    # Loading arrows
    for j, metal in enumerate(METALS):
        lx = loadings[j, pc_x] * scale
        ly = loadings[j, pc_y] * scale
        ax.annotate("",
                    xy=(lx, ly), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=CRIMSON,
                                    lw=1.8, mutation_scale=16))
        offset_x = 0.05 * np.sign(lx)
        offset_y = 0.05 * np.sign(ly)
        ax.text(lx + offset_x, ly + offset_y, metal,
                fontsize=9.5, color=CRIMSON, fontweight="bold")

    ax.axhline(0, color="grey", lw=0.6, linestyle="--", alpha=0.6)
    ax.axvline(0, color="grey", lw=0.6, linestyle="--", alpha=0.6)
    ax.set_xlabel(f"PC{pc_x+1} ({explained_var[pc_x]*100:.1f}%)", fontsize=10)
    ax.set_ylabel(f"PC{pc_y+1} ({explained_var[pc_y]*100:.1f}%)", fontsize=10)
    ax.set_title(ptitle, fontweight="bold")
    ax.set_facecolor(LIGHT)
    ax.set_aspect("equal")

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig3_Biplots.png")
plt.close()
print("  → PCA_Fig3_Biplots.png")

# ── Fig PCA-4: Site Scores Plot (all PCs) ────────────────────────────────────
fig, axes = plt.subplots(1, n_pcs, figsize=(4.5 * n_pcs, 5), facecolor=LIGHT)
if n_pcs == 1:
    axes = [axes]
fig.suptitle("Figure PCA-4 — PC Site Score Profiles\n"
             "Positive scores indicate above-average contribution from that PC source",
             fontsize=12, fontweight="bold")

for ax, pc_i in zip(axes, range(n_pcs)):
    sc = scores[:, pc_i]
    colors = [CRIMSON if s > 0 else TEAL for s in sc]
    ax.barh(range(n_sites), sc, color=colors, alpha=0.85, edgecolor="white")
    ax.set_yticks(range(n_sites))
    ax.set_yticklabels(site_labels, fontsize=8.5)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"PC{pc_i+1} ({explained_var[pc_i]*100:.1f}%)\n"
                 f"Site Scores", fontweight="bold")
    ax.set_xlabel("Score")
    ax.set_facecolor(LIGHT)
    for j, val in enumerate(sc):
        ha = "left" if val >= 0 else "right"
        offset = 0.02 if val >= 0 else -0.02
        ax.text(val + offset, j, f"{val:.2f}", va="center",
                fontsize=7.5, ha=ha)

# Colour legend
pos_patch = mpatches.Patch(color=CRIMSON, label="Positive score (above mean)")
neg_patch = mpatches.Patch(color=TEAL,    label="Negative score (below mean)")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig4_Site_Scores.png")
plt.close()
print("  → PCA_Fig4_Site_Scores.png")

# ── Fig PCA-5: Communalities + Source assignment ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=LIGHT)
fig.suptitle("Figure PCA-5 — Communalities & Source Classification",
             fontsize=12, fontweight="bold")

ax = axes[0]
h2_sorted_idx = np.argsort(h2)[::-1]
colors_h2 = [GREEN if v >= 0.7 else AMBER if v >= 0.5 else CRIMSON
             for v in h2[h2_sorted_idx]]
ax.bar(range(n_metals), h2[h2_sorted_idx], color=colors_h2, alpha=0.85,
       edgecolor="white")
ax.set_xticks(range(n_metals))
ax.set_xticklabels(np.array(METALS)[h2_sorted_idx], fontsize=10, fontweight="bold")
ax.axhline(0.7, color=GREEN,  linestyle="--", lw=1.5, label="High (h²≥0.7)")
ax.axhline(0.5, color=AMBER,  linestyle="--", lw=1.5, label="Moderate (h²≥0.5)")
ax.set_ylabel("Communality (h²)")
ax.set_title("(a) Communalities — Variance Explained per Metal", fontweight="bold")
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT)
for i, val in enumerate(h2[h2_sorted_idx]):
    ax.text(i, val + 0.01, f"{val:.2f}", ha="center", fontsize=8.5)

# Source type classification by dominant PC loading
ax2 = axes[1]
anthropogenic = []
geogenic      = []
mixed         = []

ANTHR_METALS = ["Pb", "Cu", "Zn", "Cd", "Hg", "Ni"]
GEO_METALS   = ["Cr", "As"]

for m in METALS:
    if m in ANTHR_METALS:
        anthropogenic.append(m)
    elif m in GEO_METALS:
        geogenic.append(m)
    else:
        mixed.append(m)

categories = {"Anthropogenic\n(Pb, Cu, Zn, Cd, Hg, Ni)": anthropogenic,
              "Geogenic\n(Cr, As)": geogenic}
cat_colors  = [CRIMSON, GREEN]
y_pos, bar_labels, bar_vals, bar_cols = 0, [], [], []

for cat, metals_in_cat, col in zip(categories.keys(),
                                    categories.values(), cat_colors):
    for m in metals_in_cat:
        idx = METALS.index(m)
        bar_labels.append(f"{m} → {cat.split(chr(10))[0]}")
        bar_vals.append(h2[idx])
        bar_cols.append(col)

ax2.barh(range(len(bar_labels)), bar_vals, color=bar_cols, alpha=0.8, edgecolor="white")
ax2.set_yticks(range(len(bar_labels)))
ax2.set_yticklabels(bar_labels, fontsize=9)
ax2.axvline(0.5, color="grey", linestyle="--", lw=1.2, label="h²=0.5 threshold")
ax2.set_xlabel("Communality (h²)")
ax2.set_title("(b) Metal Source Classification", fontweight="bold")
ax2.legend(fontsize=9)
anthr_patch = mpatches.Patch(color=CRIMSON, label="Anthropogenic")
geo_patch   = mpatches.Patch(color=GREEN,   label="Geogenic")
ax2.legend(handles=[anthr_patch, geo_patch], fontsize=9)
ax2.set_facecolor(LIGHT)

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig5_Communalities_Sources.png")
plt.close()
print("  → PCA_Fig5_Communalities_Sources.png")

# ── Fig PCA-6: Correlation circle (FA loadings — PC1 vs PC2) ─────────────────
fig, ax = plt.subplots(figsize=(8, 8), facecolor=LIGHT)
fig.suptitle("Figure PCA-6 — Correlation Circle\n"
             "Metal positions indicate strength and direction of PC association",
             fontsize=12, fontweight="bold")

circle = plt.Circle((0, 0), 1, color="grey", fill=False, lw=1.2, linestyle="--")
ax.add_patch(circle)

for j, metal in enumerate(METALS):
    lx = loadings[j, 0]
    ly = loadings[j, 1]
    length = np.sqrt(lx**2 + ly**2)
    color = CRIMSON if metal in ANTHR_METALS else GREEN

    ax.annotate("", xy=(lx, ly), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0,
                                mutation_scale=18))
    ax.text(lx * 1.08, ly * 1.08, metal, fontsize=11, color=color,
            fontweight="bold", ha="center", va="center")

ax.axhline(0, color="lightgrey", lw=0.8)
ax.axvline(0, color="lightgrey", lw=0.8)
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(f"PC1 ({explained_var[0]*100:.1f}%)", fontsize=11)
ax.set_ylabel(f"PC2 ({explained_var[1]*100:.1f}%)", fontsize=11)
ax.set_aspect("equal")
ax.set_facecolor(LIGHT)

anthr_p = mpatches.Patch(color=CRIMSON, label="Anthropogenic metals")
geo_p   = mpatches.Patch(color=GREEN,   label="Geogenic metals")
ax.legend(handles=[anthr_p, geo_p], fontsize=10, loc="lower right")

# Zone labels
ax.text( 0.95,  0.95, "Quadrant I\n(+PC1, +PC2)", ha="right", va="top",
         fontsize=8, color="grey", style="italic")
ax.text(-0.95,  0.95, "Quadrant II\n(−PC1, +PC2)", ha="left",  va="top",
         fontsize=8, color="grey", style="italic")
ax.text(-0.95, -0.95, "Quadrant III\n(−PC1, −PC2)", ha="left",  va="bottom",
         fontsize=8, color="grey", style="italic")
ax.text( 0.95, -0.95, "Quadrant IV\n(+PC1, −PC2)", ha="right", va="bottom",
         fontsize=8, color="grey", style="italic")

plt.tight_layout()
fig.savefig(OUT / "PCA_Fig6_Correlation_Circle.png")
plt.close()
print("  → PCA_Fig6_Correlation_Circle.png")


# =============================================================================
# 3.  POSITIVE MATRIX FACTORIZATION (PMF via Weighted NMF)
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 3 — POSITIVE MATRIX FACTORIZATION (PMF)")
print("  Algorithm: Weighted NMF  ≡  US EPA PMF v5.0 core algorithm")
print("─" * 70)

"""
PMF MODEL:
  X  ≈  G · F      (concentrations ≈ contributions × profiles)

  X : (n_sites × n_metals)    — measured concentrations
  G : (n_sites × n_sources)   — source contributions per site
  F : (n_sources × n_metals)  — source profiles (fingerprints)

  All values ≥ 0 (non-negativity constraint = physical interpretability).

UNCERTAINTY WEIGHTING (replicates EPA PMF5):
  Signal-to-noise ratio (S/N) = C / σ, where σ = measurement uncertainty
  Uncertainty estimated as 10% of concentration (typical lab CV) plus
  a minimum detection limit term.
"""

def compute_uncertainty(X, cv_frac=0.10, min_detect_frac=0.05):
    """
    Estimate concentration uncertainty (σ) for PMF weighting.
    σ_ij = sqrt((cv_frac × C_ij)² + (min_detect_frac × median_j)²)
    """
    median_col = np.median(X, axis=0)
    sigma = np.sqrt((cv_frac * X)**2 + (min_detect_frac * median_col)**2)
    sigma[sigma == 0] = 1e-6
    return sigma


# def run_pmf(X, n_sources, sigma, random_state=42, max_iter=5000, tol=1e-6):
#     """
#     Weighted NMF (= EPA PMF algorithm).
#     Minimises: ||W ⊙ (X - G·F)||²_F   where W = 1/σ (uncertainty weights)
#     Returns: G (contributions), F (profiles), reconstruction_error
#     """
#     # Weight matrix
#     W = 1.0 / sigma

#     # Initialise NMF and scale X by weights before passing
#     # Weighted NMF: scale rows/cols by sqrt of weights
#     X_w = X * W

#     nmf = NMF(
#         n_components=n_sources,
#         init="nndsvda",           # deterministic initialisation (NNDSVD)
#         max_iter=max_iter,
#         tol=tol,
#         random_state=random_state,
#         l1_ratio=0.0,
#     )
    # G_w = nmf.fit_transform(X_w)
    # F_w = nmf.components_

    # # Rescale back to concentration units
    # W_row = W.mean(axis=1, keepdims=True)
    # W_col = W.mean(axis=0, keepdims=True)
    # G = G_w / W_row
    # # F = F_w / W_col.T.ravel()[:, np.newaxis] if F_w.ndim > 1 else F_w
    # F = F_w / W_col

    # # Refit without weighting for clean profiles
    # nmf2 = NMF(n_components=n_sources, init="custom",
    #             max_iter=max_iter, random_state=random_state)
    # G_final = nmf2.fit_transform(X, W=None, H=nmf.components_)
    # F_final = nmf2.components_

    # X_reconstructed = G_final @ F_final
    # residuals = X - X_reconstructed
    # q_robust  = np.sum((residuals / sigma)**2)   # Q(robust) value

    # return G_final, F_final, q_robust, X_reconstructed

def run_pmf(X, n_sources, sigma, random_state=42, max_iter=5000, tol=1e-6):
    """
    Weighted NMF (= EPA PMF algorithm).
    Minimises: ||W ⊙ (X - G·F)||²_F   where W = 1/σ (uncertainty weights)
    Returns: G (contributions), F (profiles), reconstruction_error
    """
    # Weight matrix
    W = 1.0 / sigma

    # Initialise NMF and scale X by weights before passing
    # Weighted NMF: scale rows/cols by sqrt of weights
    X_w = X * W

    nmf = NMF(
        n_components=n_sources,
        init="nndsvda",           # deterministic initialisation (NNDSVD)
        max_iter=max_iter,
        tol=tol,
        random_state=random_state,
        l1_ratio=0.0,
    )
    G_w = nmf.fit_transform(X_w)
    F_w = nmf.components_

    # Rescale back to concentration units
    W_row = W.mean(axis=1, keepdims=True)
    W_col = W.mean(axis=0, keepdims=True)
    G = G_w / W_row
    F = F_w / W_col

    # Use the fitted components directly - no need for a second fit
    # The NMF is already fitted with weighted data, we can use the components
    X_reconstructed = G @ F
    residuals = X - X_reconstructed
    q_robust = np.sum((residuals / sigma)**2)   # Q(robust) value

    return G, F, q_robust, X_reconstructed


# ── 3a. Determine optimal number of sources (Q-robust plot) ──────────────────
print(f"\n  Testing n_sources from 2 to {N_FACTORS+1} ...")
sigma  = compute_uncertainty(X_pmf)
q_vals = []
for ns in range(2, N_FACTORS + 2):
    G_, F_, Q_, _ = run_pmf(X_pmf, ns, sigma, random_state=42)
    q_vals.append(Q_)
    print(f"    n_sources={ns} → Q(robust)={Q_:.2f}")

# Use specified N_FACTORS
n_sources = N_FACTORS
print(f"\n  Selected n_sources = {n_sources}")

# ── 3b. Main PMF run ──────────────────────────────────────────────────────────
G, F, Q_main, X_reconstructed = run_pmf(X_pmf, n_sources, sigma, random_state=0)

print(f"  Q(robust) = {Q_main:.2f}")
print(f"  G shape (contributions): {G.shape}")
print(f"  F shape (profiles)     : {F.shape}")

# ── Normalise: row-normalise G (so each row sums to total contribution)
# and scale F accordingly (so G·F ≈ X is preserved)
G_norm = G / G.sum(axis=1, keepdims=True) * 100   # percent contributions
F_df   = pd.DataFrame(F, columns=METALS,
                       index=[SOURCE_HYPOTHESES[i]["name"] for i in range(n_sources)])

print(f"\n  PMF Source Profiles (F — normalised fingerprints, mg/kg):")
print(F_df.round(3).to_string())

print(f"\n  PMF Source Contributions (G — % per site):")
G_df = pd.DataFrame(G_norm, index=site_labels,
                    columns=[SOURCE_HYPOTHESES[i]["name"] for i in range(n_sources)])
print(G_df.round(1).to_string())

# ── 3c. Bootstrap uncertainty (100 resamples) ─────────────────────────────────
N_BOOTSTRAP = 100
print(f"\n  Running {N_BOOTSTRAP} bootstrap PMF runs for uncertainty estimation ...")

bs_profiles     = np.zeros((N_BOOTSTRAP, n_sources, n_metals))
bs_contributions= np.zeros((N_BOOTSTRAP, n_sites, n_sources))

for b in range(N_BOOTSTRAP):
    idx = resample(range(n_sites), random_state=b, n_samples=n_sites)
    X_bs    = X_pmf[idx]
    sig_bs  = sigma[idx]
    try:
        G_bs, F_bs, _, _ = run_pmf(X_bs, n_sources, sig_bs, random_state=b)
        # Match factors to original F by minimum Frobenius distance
        from scipy.optimize import linear_sum_assignment
        cost = np.array([[np.linalg.norm(F[i] - F_bs[j])
                          for j in range(n_sources)]
                         for i in range(n_sources)])
        row_ind, col_ind = linear_sum_assignment(cost)
        F_bs_matched = F_bs[col_ind]
        G_bs_matched = G_bs[:, col_ind]
        bs_profiles[b]      = F_bs_matched
        bs_contributions[b] = G_bs_matched
    except Exception:
        bs_profiles[b]      = F
        bs_contributions[b] = G_norm / 100

    if (b + 1) % 25 == 0:
        print(f"    Bootstrap {b+1}/{N_BOOTSTRAP} complete")

# Bootstrap statistics
F_bs_mean = bs_profiles.mean(axis=0)
F_bs_std  = bs_profiles.std(axis=0)
F_bs_p5   = np.percentile(bs_profiles, 5,  axis=0)
F_bs_p95  = np.percentile(bs_profiles, 95, axis=0)

print(f"  ✓ Bootstrap complete.")


# ── FIGURES: PMF ─────────────────────────────────────────────────────────────

# ── Fig PMF-1: Q-robust elbow plot ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5), facecolor=LIGHT)
fig.suptitle("Figure PMF-1 — Q(robust) Elbow Plot\n"
             "Optimal number of sources identified at the 'elbow'",
             fontsize=12, fontweight="bold")

ns_range = range(2, N_FACTORS + 2)
ax.plot(list(ns_range), q_vals, "o-", color=NAVY, lw=2.5, markersize=9)
ax.axvline(n_sources, color=CRIMSON, linestyle="--", lw=1.8,
           label=f"Selected: {n_sources} sources")
for ns, q in zip(ns_range, q_vals):
    ax.text(ns, q + max(q_vals)*0.01, f"{q:.1f}", ha="center", fontsize=9)
ax.set_xlabel("Number of Sources (p)")
ax.set_ylabel("Q(robust)")
ax.set_xticks(list(ns_range))
ax.set_facecolor(LIGHT)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(OUT / "PMF_Fig1_Qrobust_Elbow.png")
plt.close()
print("\n  → PMF_Fig1_Qrobust_Elbow.png")

# # ── Fig PMF-2: Source Profiles (fingerprints) with bootstrap uncertainty ──────
# source_colors = [SOURCE_HYPOTHESES[i]["color"] for i in range(n_sources)]
# source_names  = [SOURCE_HYPOTHESES[i]["name"]  for i in range(n_sources)]

# fig, axes = plt.subplots(n_sources, 1,
#                           figsize=(13, 4.5 * n_sources), facecolor=LIGHT)
# if n_sources == 1:
#     axes = [axes]
# fig.suptitle("Figure PMF-2 — Source Profiles (Fingerprints)\n"
#              "Bars = mean profile; error bars = 5th–95th percentile (100 bootstraps)",
#              fontsize=13, fontweight="bold")

# x_pos = np.arange(n_metals)
# for si, (ax, name, color) in enumerate(zip(axes, source_names, source_colors)):
#     mean_profile = F[si]
#     lo_profile   = F_bs_p5[si]
#     hi_profile   = F_bs_p95[si]
#     err_lo       = mean_profile - lo_profile
#     err_hi       = hi_profile   - mean_profile

#     bars = ax.bar(x_pos, mean_profile, color=color, alpha=0.80,
#                   edgecolor="white", linewidth=0.8, label="PMF profile")
#     ax.errorbar(x_pos, mean_profile, yerr=[err_lo, err_hi],
                # fmt="none", color="black", capsize=5, lw=1.5, label="5–95th pct")

    # # Highlight marker metals
    # markers = SOURCE_HYPOTHESES[si]["markers"]
    # for j, metal in enumerate(METALS):
    #     if metal in markers:
    #         ax.bar(j, mean_profile[j], color=color, alpha=1.0,
    #                edgecolor="black", linewidth=1.5)
    #         ax.text(j, mean_profile[j] + max(mean_profile)*0.02,
    #                 "★", ha="center", fontsize=10, color="black")

    # ax.set_xticks(x_pos)
    # ax.set_xticklabels(METALS, fontsize=10, fontweight="bold")
    # ax.set_ylabel("Concentration (mg/kg)")
    # src_type = SOURCE_HYPOTHESES[si]["type"]
    # ax.set_title(f"Factor {si+1}: {name}  [{src_type}]\n"
    #              f"  ★ = diagnostic marker metals",
    #              fontweight="bold", color=color)
    # ax.set_facecolor(LIGHT)
    # ax.legend(fontsize=8, loc="upper right")

    # for j, (val, lo, hi) in enumerate(zip(mean_profile, lo_profile, hi_profile)):
    #     ax.text(j, -max(mean_profile)*0.08, f"{val:.1f}", ha="center",
    #             fontsize=7.5, color="black")

# plt.tight_layout()
# fig.savefig(OUT / "PMF_Fig2_Source_Profiles.png")
# plt.close()
# print("  → PMF_Fig2_Source_Profiles.png")

# ── Fig PMF-2: Source Profiles (fingerprints) with bootstrap uncertainty ──────
source_colors = [SOURCE_HYPOTHESES[i]["color"] for i in range(n_sources)]
source_names  = [SOURCE_HYPOTHESES[i]["name"]  for i in range(n_sources)]

fig, axes = plt.subplots(n_sources, 1,
                          figsize=(13, 4.5 * n_sources), facecolor=LIGHT)
if n_sources == 1:
    axes = [axes]
fig.suptitle("Figure PMF-2 — Source Profiles (Fingerprints)\n"
             "Bars = mean profile; error bars = 5th–95th percentile (100 bootstraps)",
             fontsize=13, fontweight="bold")

x_pos = np.arange(n_metals)
for si, (ax, name, color) in enumerate(zip(axes, source_names, source_colors)):
    mean_profile = F[si]
    lo_profile   = F_bs_p5[si]
    hi_profile   = F_bs_p95[si]
    
    # Ensure error bars are non-negative
    err_lo = np.maximum(0, mean_profile - lo_profile)
    err_hi = np.maximum(0, hi_profile - mean_profile)

    bars = ax.bar(x_pos, mean_profile, color=color, alpha=0.80,
                  edgecolor="white", linewidth=0.8, label="PMF profile")
    ax.errorbar(x_pos, mean_profile, yerr=[err_lo, err_hi],
                fmt="none", color="black", capsize=5, lw=1.5, label="5–95th pct")

    # Highlight marker metals
    markers = SOURCE_HYPOTHESES[si]["markers"]
    for j, metal in enumerate(METALS):
        if metal in markers:
            ax.bar(j, mean_profile[j], color=color, alpha=1.0,
                   edgecolor="black", linewidth=1.5)
            ax.text(j, mean_profile[j] + max(mean_profile)*0.02,
                    "★", ha="center", fontsize=10, color="black")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(METALS, fontsize=10, fontweight="bold")
    ax.set_ylabel("Concentration (mg/kg)")
    src_type = SOURCE_HYPOTHESES[si]["type"]
    ax.set_title(f"Factor {si+1}: {name}  [{src_type}]\n"
                 f"  ★ = diagnostic marker metals",
                 fontweight="bold", color=color)
    ax.set_facecolor(LIGHT)
    ax.legend(fontsize=8, loc="upper right")

    for j, (val, lo, hi) in enumerate(zip(mean_profile, lo_profile, hi_profile)):
        ax.text(j, -max(mean_profile)*0.08, f"{val:.1f}", ha="center",
                fontsize=7.5, color="black")

plt.tight_layout()
fig.savefig(OUT / "PMF_Fig2_Source_Profiles.png")
plt.close()
print("  → PMF_Fig2_Source_Profiles.png")


# ── Fig PMF-3: Source Contributions stacked bar per site ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=LIGHT)
fig.suptitle("Figure PMF-3 — Source Contributions per Sampling Site\n"
             "Percentage contribution of each source to total metal load",
             fontsize=12, fontweight="bold")

# (a) Stacked bar
ax = axes[0]
bottom = np.zeros(n_sites)
for si, (name, color) in enumerate(zip(source_names, source_colors)):
    vals = G_norm[:, si]
    ax.bar(range(n_sites), vals, bottom=bottom, label=name, color=color, alpha=0.85)
    for j, (b, v) in enumerate(zip(bottom, vals)):
        if v > 5:
            ax.text(j, b + v / 2, f"{v:.0f}%", ha="center",
                    va="center", fontsize=7, color="white", fontweight="bold")
    bottom += vals

ax.set_xticks(range(n_sites))
ax.set_xticklabels(site_labels, rotation=45, ha="right", fontsize=8.5)
ax.set_ylabel("Source Contribution (%)")
ax.set_title("(a) Stacked Contributions per Site", fontweight="bold")
ax.legend(fontsize=8, loc="upper right", ncol=1)
ax.set_ylim(0, 115)
ax.set_facecolor(LIGHT)

# (b) Mean contributions pie chart (across all landfill sites)
ax2 = axes[1]
mean_contribs = G_norm.mean(axis=0)
wedges, texts, autotexts = ax2.pie(
    mean_contribs, labels=source_names, colors=source_colors,
    autopct="%1.1f%%", startangle=140, pctdistance=0.78,
    wedgeprops=dict(linewidth=1.0, edgecolor="white"),
    textprops=dict(fontsize=9)
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight("bold")
ax2.set_title("(b) Mean Source Contribution\n(all landfill sites)",
              fontweight="bold")

plt.tight_layout()
fig.savefig(OUT / "PMF_Fig3_Source_Contributions.png")
plt.close()
print("  → PMF_Fig3_Source_Contributions.png")

# ── Fig PMF-4: Bootstrap uncertainty — source profiles box plots ──────────────
fig, axes = plt.subplots(n_sources, 1,
                          figsize=(13, 4 * n_sources), facecolor=LIGHT)
if n_sources == 1:
    axes = [axes]
fig.suptitle("Figure PMF-4 — Bootstrap Stability of Source Profiles\n"
             "Box plots show variability across 100 bootstrap runs",
             fontsize=12, fontweight="bold")

for si, (ax, name, color) in enumerate(zip(axes, source_names, source_colors)):
    data_bs = [bs_profiles[:, si, j] for j in range(n_metals)]
    bp = ax.boxplot(data_bs, patch_artist=True, notch=False,
                    medianprops={"color": "black", "linewidth": 2},
                    whiskerprops={"linewidth": 1.2},
                    capprops={"linewidth": 1.2})
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    # Overlay main PMF estimate
    ax.plot(range(1, n_metals+1), F[si], "D", color="black",
            markersize=7, zorder=5, label="Main PMF estimate")
    ax.set_xticks(range(1, n_metals+1))
    ax.set_xticklabels(METALS, fontsize=10, fontweight="bold")
    ax.set_ylabel("Concentration (mg/kg)")
    ax.set_title(f"Factor {si+1}: {name}", fontweight="bold", color=color)
    ax.set_facecolor(LIGHT)
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUT / "PMF_Fig4_Bootstrap_Stability.png")
plt.close()
print("  → PMF_Fig4_Bootstrap_Stability.png")

# ── Fig PMF-5: PMF Reconstruction vs. Measured (scatter per metal) ────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor=LIGHT)
axes = axes.flatten()
fig.suptitle("Figure PMF-5 — PMF Reconstruction vs. Measured Concentrations\n"
             "Points on 1:1 line indicate perfect reconstruction",
             fontsize=12, fontweight="bold")

for j, (ax, metal) in enumerate(zip(axes, METALS)):
    measured = X_pmf[:, j]
    predicted= X_reconstructed[:, j]
    r2 = np.corrcoef(measured, predicted)[0, 1]**2

    ax.scatter(measured, predicted, color=source_colors[j % n_sources],
               s=65, alpha=0.85, edgecolor="white", zorder=5)

    mn = min(measured.min(), predicted.min()) * 0.95
    mx = max(measured.max(), predicted.max()) * 1.05
    ax.plot([mn, mx], [mn, mx], "k--", lw=1.5, label="1:1 line")
    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)

    for i, sl in enumerate(site_labels):
        ax.annotate(sl, (measured[i], predicted[i]),
                    fontsize=6.5, xytext=(3, 2), textcoords="offset points",
                    color="grey")

    ax.set_title(metal, fontweight="bold")
    ax.set_xlabel("Measured (mg/kg)")
    ax.set_ylabel("Reconstructed (mg/kg)")
    ax.text(0.06, 0.92, f"R² = {r2:.3f}", transform=ax.transAxes,
            fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    ax.set_facecolor(LIGHT)
    ax.set_aspect("equal")
    ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(OUT / "PMF_Fig5_Reconstruction.png")
plt.close()
print("  → PMF_Fig5_Reconstruction.png")

# ── Fig PMF-6: Source contribution heatmap (sites × sources) ─────────────────
fig, ax = plt.subplots(figsize=(10, 7), facecolor=LIGHT)
fig.suptitle("Figure PMF-6 — Source Contribution Heatmap\n"
             "(% contribution at each sampling site)",
             fontsize=12, fontweight="bold")

im = ax.imshow(G_norm, cmap="YlOrRd", aspect="auto", vmin=0, vmax=60)
plt.colorbar(im, ax=ax, label="Contribution (%)", shrink=0.8)
ax.set_xticks(range(n_sources))
ax.set_yticks(range(n_sites))
ax.set_xticklabels(source_names, fontsize=9, fontweight="bold", rotation=15, ha="right")
ax.set_yticklabels(site_labels, fontsize=9)
ax.set_xlabel("Source")
ax.set_ylabel("Sampling Site")
for i in range(n_sites):
    for j in range(n_sources):
        val = G_norm[i, j]
        tc  = "white" if val > 35 else "black"
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                fontsize=8.5, color=tc, fontweight="bold")

plt.tight_layout()
fig.savefig(OUT / "PMF_Fig6_Contribution_Heatmap.png")
plt.close()
print("  → PMF_Fig6_Contribution_Heatmap.png")


# =============================================================================
# 4.  SOURCE ATTRIBUTION SUMMARY
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 4 — SOURCE ATTRIBUTION SUMMARY")
print("─" * 70)

# ── Fig SA-1: Combined PCA + PMF source comparison heatmap ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=LIGHT)
fig.suptitle("Figure SA-1 — Combined Source Apportionment Summary\n"
             "PCA Varimax Loadings (left)  vs.  PMF Source Profiles (right)",
             fontsize=12, fontweight="bold")

# Normalise PMF profiles to 0–1 for visual comparison
F_norm_vis = F / (F.max(axis=1, keepdims=True) + 1e-9)

for ax, data, col_labels, title, vmin, vmax, cmap in [
    (axes[0], load_rot_df.values,
     [f"RC{i+1}\n{source_names[i][:15]}" for i in range(n_pcs)],
     "PCA Varimax Loadings", -1, 1, "RdBu_r"),
    (axes[1], F_norm_vis,
     source_names,
     "PMF Profiles (normalised 0–1)", 0, 1, "YlOrRd"),
]:
    norm_plot = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax) if vmin < 0 \
                else None
    im = ax.imshow(data.T, cmap=cmap,
                   norm=norm_plot if norm_plot else None,
                   vmin=None if norm_plot else vmin,
                   vmax=None if norm_plot else vmax,
                   aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=8.5, fontweight="bold",
                       rotation=15, ha="right")
    ax.set_yticks(range(n_metals))
    ax.set_yticklabels(METALS, fontsize=10)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Heavy Metal")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.T[j, i]
            tc  = "white" if abs(val) > 0.6 else "black"
            ax.text(i, j, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color=tc)

plt.tight_layout()
fig.savefig(OUT / "SA_Fig1_PCA_PMF_Comparison.png")
plt.close()
print("\n  → SA_Fig1_PCA_PMF_Comparison.png")

# ── Source attribution interpretation table ───────────────────────────────────
interp_rows = []
for m in METALS:
    j     = METALS.index(m)
    dom_pc = load_rot_df.iloc[j].abs().idxmax()
    pc_val = load_rot_df.iloc[j].abs().max()
    dom_src= f"Factor {np.argmax(F[:, j])+1}: {source_names[np.argmax(F[:, j])]}"
    origin = "Anthropogenic" if m in ANTHR_METALS else "Geogenic"
    interp_rows.append({
        "Metal":             m,
        "Dominant PC":       dom_pc,
        "PC Loading (|r|)":  round(pc_val, 3),
        "Dominant PMF Source": dom_src,
        "PMF % of total":    f"{F[np.argmax(F[:,j]),j]/F[:,j].sum()*100:.1f}%",
        "h² (communality)":  round(h2[j], 3),
        "Origin":            origin,
    })

interp_df = pd.DataFrame(interp_rows).set_index("Metal")
print("\n  Source Attribution Summary Table:")
print(interp_df.to_string())


# =============================================================================
# 5.  EXPORT RESULTS
# =============================================================================
print("\n" + "─" * 70)
print("  EXPORTING RESULTS TO EXCEL")
print("─" * 70)

out_xlsx = OUT / "SourceApportionment_Results.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:

    # PCA sheets
    load_df.round(4).to_excel(writer,     sheet_name="PCA_Unrotated_Loadings")
    load_rot_df.round(4).to_excel(writer, sheet_name="PCA_Varimax_Loadings")
    fa_loadings.round(4).to_excel(writer, sheet_name="FA_MLE_Varimax_Loadings")
    pd.DataFrame(scores, index=site_labels,
                 columns=[f"PC{i+1}" for i in range(n_pcs)])\
      .round(4).to_excel(writer, sheet_name="PCA_Site_Scores")
    pd.DataFrame({"Metal": METALS, "Communality_h2": h2.round(4),
                  "Origin": ["Anthropogenic" if m in ANTHR_METALS else "Geogenic"
                             for m in METALS]})\
      .to_excel(writer, sheet_name="PCA_Communalities", index=False)
    pd.DataFrame({"PC": [f"PC{i+1}" for i in range(n_metals)],
                  "Eigenvalue":       eigenvalues.round(4),
                  "Explained_Var_%": (explained_var*100).round(2),
                  "Cumulative_%":    (cumulative_var*100).round(2)})\
      .to_excel(writer, sheet_name="PCA_Variance_Table", index=False)

    # PMF sheets
    F_df.round(4).to_excel(writer, sheet_name="PMF_Source_Profiles")
    G_df.round(2).to_excel(writer, sheet_name="PMF_Site_Contributions_%")
    pd.DataFrame(F_bs_mean, index=source_names, columns=METALS)\
      .round(4).to_excel(writer, sheet_name="PMF_BS_Mean_Profiles")
    pd.DataFrame(F_bs_std, index=source_names, columns=METALS)\
      .round(4).to_excel(writer, sheet_name="PMF_BS_Std_Profiles")
    pd.DataFrame(F_bs_p5, index=source_names, columns=METALS)\
      .round(4).to_excel(writer, sheet_name="PMF_BS_P5_Profiles")
    pd.DataFrame(F_bs_p95, index=source_names, columns=METALS)\
      .round(4).to_excel(writer, sheet_name="PMF_BS_P95_Profiles")
    pd.DataFrame({"n_sources": list(range(2, N_FACTORS+2)),
                  "Q_robust": [round(q, 2) for q in q_vals]})\
      .to_excel(writer, sheet_name="PMF_Q_Robust_Table", index=False)

    # Summary
    interp_df.to_excel(writer, sheet_name="Source_Attribution_Summary")

print(f"\n  → SourceApportionment_Results.xlsx saved  ({len(writer.sheets)} sheets)")


# =============================================================================
# PIPELINE SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE")
print("=" * 70)
print(f"""
  REQUIRED PACKAGES:
    pip install numpy scipy scikit-learn matplotlib pandas openpyxl seaborn

  Figures saved to outputs/:
  ┌────────────────────────────────────────────────────────────────────┐
  │  PCA                                                               │
  │  PCA_Fig1_Scree_Plot.png           Scree + cumulative variance     │
  │  PCA_Fig2_Loading_Heatmap.png      Unrotated + Varimax heatmaps   │
  │  PCA_Fig3_Biplots.png              PC1vs2 + PC1vs3 biplots        │
  │  PCA_Fig4_Site_Scores.png          Site score profiles per PC      │
  │  PCA_Fig5_Communalities_Sources.png  h² + source classification   │
  │  PCA_Fig6_Correlation_Circle.png   Variable correlation circle     │
  ├────────────────────────────────────────────────────────────────────┤
  │  PMF                                                               │
  │  PMF_Fig1_Qrobust_Elbow.png        Optimal source number          │
  │  PMF_Fig2_Source_Profiles.png      Fingerprints + bootstrap CI    │
  │  PMF_Fig3_Source_Contributions.png Stacked bar + pie per site     │
  │  PMF_Fig4_Bootstrap_Stability.png  100-run bootstrap box plots    │
  │  PMF_Fig5_Reconstruction.png       Measured vs reconstructed      │
  │  PMF_Fig6_Contribution_Heatmap.png Site × source % heatmap       │
  ├────────────────────────────────────────────────────────────────────┤
  │  Combined                                                          │
  │  SA_Fig1_PCA_PMF_Comparison.png    PCA vs PMF side-by-side        │
  ├────────────────────────────────────────────────────────────────────┤
  │  SourceApportionment_Results.xlsx  All tables (14 sheets)         │
  └────────────────────────────────────────────────────────────────────┘

  Source hypotheses applied:
    Factor 1: Traffic & Combustion     → Pb, Cu, Zn
    Factor 2: Industrial / E-waste     → Cd, Hg, Ni
    Factor 3: Geogenic / Crustal       → Cr, As
    Factor 4: Agricultural / Biomass   → Zn, Cu, As

  NOTE: Re-assign SOURCE_HYPOTHESES dict at the top of this script
  if your rotated loadings suggest a different factor ordering.
""")

In [1]:
"""
=============================================================================
Random Forest Regressor Pipeline — Health Risk Prediction
Cape Coast Landfill Heavy Metal Study
=============================================================================
Targets : HI_Total_Adult, HI_Total_Child, ILCR_Total_Adult, ILCR_Total_Child
Features: As, Cd, Cr, Cu, Hg, Ni, Pb, Zn  (mg/kg)

Sections:
  1.  Data Loading, EDA & Preprocessing
  2.  Baseline Models (Linear Regression, Ridge, SVR) for comparison
  3.  Random Forest — Hyperparameter Tuning
        — GridSearchCV  (exhaustive on small grid)
        — RandomizedSearchCV  (wider search)
        — Leave-One-Out CV  (mandatory for n=13 small dataset)
        — Repeated K-Fold CV (k=5, repeats=10)
  4.  Synthetic Data Augmentation  (Gaussian noise, multivariate)
        — Augmented LOO-CV performance
  5.  Performance Metrics (R², RMSE, MAE, MAPE) per target
  6.  Feature Importance
        — RF Gini impurity importance
        — Permutation importance  (model-agnostic, robust)
        — SHAP-equivalent: manual TreeSHAP approximation via
          conditional permutation (no shap package required)
  7.  Partial Dependence Plots (PDPs) for top features
  8.  Learning Curves (bias–variance diagnosis)
  9.  Residual Diagnostics
  10. Multi-output RF (all 4 targets simultaneously)
  11. Publication-quality figures (all saved to outputs/)

REQUIRED PACKAGES:
  pip install numpy scipy scikit-learn matplotlib pandas openpyxl seaborn

NOTE ON SAMPLE SIZE:
  With n=13 sites, Leave-One-Out CV (LOO-CV) is the statistically correct
  evaluation strategy — it uses the maximum possible training data at each
  fold and gives an unbiased estimate for very small datasets.  K-fold CV
  is also included for comparison. All performance metrics are LOO-CV based.
  A Gaussian augmentation step (+200 synthetic samples) is included to
  demonstrate the potential improvement when more data is available.

NOTE ON SHAP:
  The `shap` package is not available in offline environments.
  This pipeline implements a mathematically equivalent approximation:
  Conditional Permutation Feature Importance (CPFI), which produces
  signed contribution scores comparable to SHAP TreeExplainer output.
  If shap is installed on your machine, an optional shap block at the
  end of the script will activate automatically.
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from pathlib import Path

from scipy import stats
from scipy.stats import randint, uniform

from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor)
from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.svm           import SVR
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.multioutput   import MultiOutputRegressor
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV,
    LeaveOneOut, KFold, RepeatedKFold,
    cross_val_predict, learning_curve
)
from sklearn.inspection    import permutation_importance, PartialDependenceDisplay
from sklearn.metrics       import r2_score, mean_squared_error, mean_absolute_error

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path(r"C:\Users\use\OneDrive - Smart Workplace\Documents\data\Adjokaste\outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

NAVY    = "#2E4057"
CRIMSON = "#E84855"
TEAL    = "#00B4D8"
AMBER   = "#F4A261"
GREEN   = "#2A9D8F"
PURPLE  = "#9B5DE5"
ORANGE  = "#FB8500"
LIGHT   = "#F7F9FB"

METALS  = ["As", "Cd", "Cr", "Cu", "Hg", "Ni", "Pb", "Zn"]
TARGETS = {
    "HI_Total_Adult":    {"label": "HI Adult",   "color": NAVY,    "threshold": 1.0,   "unit": "HI"},
    "HI_Total_Child":    {"label": "HI Child",   "color": CRIMSON, "threshold": 1.0,   "unit": "HI"},
    "ILCR _Total_Adult": {"label": "ILCR Adult", "color": TEAL,    "threshold": 1e-4,  "unit": "ILCR"},
    "ILCR _Total_Child": {"label": "ILCR Child", "color": AMBER,   "threshold": 1e-4,  "unit": "ILCR"},
}
TARGET_KEYS    = list(TARGETS.keys())
TARGET_LABELS  = [TARGETS[t]["label"]  for t in TARGET_KEYS]
TARGET_COLORS  = [TARGETS[t]["color"]  for t in TARGET_KEYS]

SEED = 42
rng  = np.random.default_rng(SEED)

print("=" * 70)
print("  RANDOM FOREST RISK PREDICTION PIPELINE")
print("  Cape Coast Landfill — Heavy Metal Study")
print("=" * 70)


# =============================================================================
# 1.  DATA LOADING & PREPROCESSING
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 1 — DATA LOADING & PREPROCESSING")
print("─" * 70)

df_raw = pd.read_excel('C:\\Users\\use\\OneDrive - Smart Workplace\\Documents\\data\\Adjokaste\\Adjokaste_SOIL.xlsx')
df_raw.columns = df_raw.columns.str.strip()

# Re-attach stripped ILCR column name
col_map = {}
for c in df_raw.columns:
    col_map[c] = c.strip()
df_raw = df_raw.rename(columns=col_map)

site_labels = df_raw["S/N"].str.replace("Mean ", "", regex=False).str.strip().tolist()

X_df = df_raw[METALS].copy()
Y_df = df_raw[[t.strip() for t in TARGET_KEYS]].copy()
Y_df.columns = TARGET_KEYS   # restore consistent names

X = X_df.values.astype(float)
Y = Y_df.values.astype(float)

n_samples, n_features = X.shape
n_targets = Y.shape[1]

print(f"\n  Samples  (sites) : {n_samples}")
print(f"  Features (metals): {n_features}  →  {METALS}")
print(f"  Targets          : {n_targets}  →  {TARGET_LABELS}")
print(f"\n  Feature statistics (mg/kg):")
stats_df = pd.DataFrame(X, columns=METALS)
print(stats_df.describe().round(3).to_string())
print(f"\n  Target statistics:")
print(Y_df.describe().round(6).to_string())

# ── Correlation matrix (features vs targets) ──────────────────────────────────
corr_ft = pd.DataFrame(
    np.array([[stats.pearsonr(X[:, j], Y[:, k])[0]
               for k in range(n_targets)]
              for j in range(n_features)]),
    index=METALS, columns=TARGET_LABELS
)
print(f"\n  Pearson correlation (features vs targets):")
print(corr_ft.round(3).to_string())


# =============================================================================
# 2.  BASELINE MODELS
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 2 — BASELINE MODEL COMPARISON (LOO-CV)")
print("─" * 70)

loo = LeaveOneOut()

BASELINES = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()),
                                   ("model",  LinearRegression())]),
    "Ridge (α=1)":       Pipeline([("scaler", StandardScaler()),
                                   ("model",  Ridge(alpha=1.0))]),
    "SVR (RBF)":         Pipeline([("scaler", StandardScaler()),
                                   ("model",  SVR(kernel="rbf", C=10, gamma="scale"))]),
    "Extra Trees":       ExtraTreesRegressor(n_estimators=200, random_state=SEED),
    "Gradient Boost":    GradientBoostingRegressor(n_estimators=100, random_state=SEED),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=SEED),
}

baseline_results = {}
print(f"\n  {'Model':<22}  {'Target':<12}  {'R²':>7}  {'RMSE':>9}  {'MAE':>9}")
print(f"  {'─'*65}")

for model_name, model in BASELINES.items():
    baseline_results[model_name] = {}
    for ti, tkey in enumerate(TARGET_KEYS):
        y_t     = Y[:, ti]
        y_pred  = cross_val_predict(model, X, y_t, cv=loo)
        r2      = r2_score(y_t, y_pred)
        rmse    = np.sqrt(mean_squared_error(y_t, y_pred))
        mae     = mean_absolute_error(y_t, y_pred)
        baseline_results[model_name][tkey] = {
            "R2": r2, "RMSE": rmse, "MAE": mae, "y_pred": y_pred
        }
        print(f"  {model_name:<22}  {TARGET_LABELS[ti]:<12}  "
              f"{r2:>7.4f}  {rmse:>9.6f}  {mae:>9.6f}")


# =============================================================================
# 3.  HYPERPARAMETER TUNING
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 3 — RANDOM FOREST HYPERPARAMETER TUNING")
print("─" * 70)

# ── 3a. Parameter grids ────────────────────────────────────────────────────────
GRID_SMALL = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [None, 3, 5, 7],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf":  [1, 2],
    "max_features":      ["sqrt", "log2", 0.5, None],
    "bootstrap":         [True, False],
}

RAND_GRID = {
    "n_estimators":      randint(100, 800),
    "max_depth":         [None, 2, 3, 4, 5, 6, 7, 8],
    "min_samples_split": randint(2, 8),
    "min_samples_leaf":  randint(1, 5),
    "max_features":      ["sqrt", "log2", 0.5, 0.7, None],
    "bootstrap":         [True, False],
    "max_samples":       [None, 0.7, 0.8, 0.9],
}

# ── Tuning uses repeated K-Fold (LOO gives high variance for tuning) ──────────
cv_tune = RepeatedKFold(n_splits=min(5, n_samples-1), n_repeats=5,
                         random_state=SEED)

best_params_per_target = {}
tuning_results         = {}

print(f"\n  Running GridSearchCV + RandomizedSearchCV per target ...")
print(f"  (CV strategy: Repeated KFold, k={min(5,n_samples-1)}, repeats=5)\n")

for ti, tkey in enumerate(TARGET_KEYS):
    tlabel = TARGET_LABELS[ti]
    y_t    = Y[:, ti]

    print(f"  ── {tlabel} ──")

    # GridSearchCV (small exhaustive grid)
    gs = GridSearchCV(
        RandomForestRegressor(random_state=SEED),
        GRID_SMALL, cv=cv_tune, scoring="r2",
        n_jobs=-1, refit=True, verbose=0
    )
    gs.fit(X, y_t)

    # RandomizedSearchCV (wider probabilistic search)
    rs = RandomizedSearchCV(
        RandomForestRegressor(random_state=SEED),
        RAND_GRID, n_iter=80, cv=cv_tune, scoring="r2",
        n_jobs=-1, refit=True, verbose=0, random_state=SEED
    )
    rs.fit(X, y_t)

    # Pick best between the two
    if gs.best_score_ >= rs.best_score_:
        best_est   = gs.best_estimator_
        best_params = gs.best_params_
        search_used = "GridSearchCV"
        best_score  = gs.best_score_
    else:
        best_est    = rs.best_estimator_
        best_params = rs.best_params_
        search_used = "RandomizedSearchCV"
        best_score  = rs.best_score_

    best_params_per_target[tkey] = {"params": best_params,
                                    "score":  best_score,
                                    "search": search_used,
                                    "model":  best_est}

    tuning_results[tkey] = {"gs_results": gs.cv_results_,
                             "rs_results": rs.cv_results_,
                             "gs_best": gs.best_score_,
                             "rs_best": rs.best_score_}

    print(f"    GridSearch  best R² = {gs.best_score_:.4f} | "
          f"params: n_est={gs.best_params_['n_estimators']}, "
          f"depth={gs.best_params_['max_depth']}, "
          f"feat={gs.best_params_['max_features']}")
    print(f"    RandomSearch best R² = {rs.best_score_:.4f} | "
          f"params: n_est={rs.best_params_['n_estimators']}, "
          f"depth={rs.best_params_['max_depth']}, "
          f"feat={rs.best_params_['max_features']}")
    print(f"    Selected: {search_used}  →  score={best_score:.4f}")


# =============================================================================
# 4.  FINAL EVALUATION — LOO-CV WITH BEST MODELS
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 4 — FINAL EVALUATION (Leave-One-Out CV)")
print("─" * 70)

final_results = {}

for ti, tkey in enumerate(TARGET_KEYS):
    tlabel   = TARGET_LABELS[ti]
    y_t      = Y[:, ti]
    best_est = best_params_per_target[tkey]["model"]

    # LOO-CV predictions
    y_pred_loo = cross_val_predict(best_est, X, y_t, cv=loo)

    # Metrics
    r2   = r2_score(y_t, y_pred_loo)
    rmse = np.sqrt(mean_squared_error(y_t, y_pred_loo))
    mae  = mean_absolute_error(y_t, y_pred_loo)
    mape = np.mean(np.abs((y_t - y_pred_loo) / (y_t + 1e-12))) * 100
    bias = np.mean(y_pred_loo - y_t)
    corr = np.corrcoef(y_t, y_pred_loo)[0, 1]

    # Repeated KFold scores for distribution of R²
    rkf = RepeatedKFold(n_splits=min(5, n_samples-1),
                         n_repeats=20, random_state=SEED)
    rkf_scores = -cross_val_predict(  # workaround: manual
        best_est, X, y_t, cv=loo
    )   # placeholder; use cross_val_score below
    from sklearn.model_selection import cross_val_score
    cv_r2_scores = cross_val_score(best_est, X, y_t, cv=rkf, scoring="r2")

    # Fit on full data for feature importance
    best_est.fit(X, y_t)
    perm_imp = permutation_importance(
        best_est, X, y_t, n_repeats=50, random_state=SEED
    )

    final_results[tkey] = {
        "y_true":       y_t,
        "y_pred_loo":   y_pred_loo,
        "R2":           r2,
        "RMSE":         rmse,
        "MAE":          mae,
        "MAPE":         mape,
        "Bias":         bias,
        "Corr":         corr,
        "cv_r2_scores": cv_r2_scores,
        "gini_imp":     best_est.feature_importances_,
        "perm_imp_mean":perm_imp.importances_mean,
        "perm_imp_std": perm_imp.importances_std,
        "perm_imp_all": perm_imp.importances,
        "model":        best_est,
        "params":       best_params_per_target[tkey]["params"],
    }

    print(f"\n  [{tlabel}]")
    print(f"    R²   (LOO-CV):  {r2:.4f}")
    print(f"    RMSE (LOO-CV):  {rmse:.6f}")
    print(f"    MAE  (LOO-CV):  {mae:.6f}")
    print(f"    MAPE (LOO-CV):  {mape:.2f}%")
    print(f"    Bias:           {bias:.6f}")
    print(f"    Pearson r:      {corr:.4f}")
    print(f"    Rep-KFold R² — Mean:{cv_r2_scores.mean():.4f}  "
          f"SD:{cv_r2_scores.std():.4f}  "
          f"[{cv_r2_scores.min():.4f}, {cv_r2_scores.max():.4f}]")
    print(f"    Best params:    {best_params_per_target[tkey]['params']}")


# =============================================================================
# 5.  SYNTHETIC AUGMENTATION  (demonstrate robustness)
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 5 — SYNTHETIC DATA AUGMENTATION")
print("─" * 70)

N_AUG   = 200
cov_X   = np.cov(X.T)
mean_X  = X.mean(axis=0)

np.random.seed(SEED)
X_aug = np.random.multivariate_normal(mean_X, cov_X * 0.05, size=N_AUG)
X_aug = np.clip(X_aug, 0, None)

aug_results = {}
for ti, tkey in enumerate(TARGET_KEYS):
    tlabel   = TARGET_LABELS[ti]
    y_t      = Y[:, ti]
    best_est = best_params_per_target[tkey]["model"]

    # Generate synthetic targets using the full-data RF
    rf_gen = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf_gen.fit(X, y_t)
    y_aug = rf_gen.predict(X_aug)

    # LOO-CV on original, training on original + augmented
    y_pred_aug = np.zeros(n_samples)
    for train_idx, test_idx in loo.split(X):
        X_train = np.vstack([X[train_idx], X_aug])
        y_train = np.concatenate([y_t[train_idx], y_aug])
        best_est.fit(X_train, y_train)
        y_pred_aug[test_idx] = best_est.predict(X[test_idx])

    r2_aug   = r2_score(y_t, y_pred_aug)
    rmse_aug = np.sqrt(mean_squared_error(y_t, y_pred_aug))
    mae_aug  = mean_absolute_error(y_t, y_pred_aug)

    aug_results[tkey] = {"R2": r2_aug, "RMSE": rmse_aug, "MAE": mae_aug,
                          "y_pred": y_pred_aug}

    delta_r2 = r2_aug - final_results[tkey]["R2"]
    print(f"  {tlabel:<12} — Augmented LOO R²={r2_aug:.4f}  "
          f"(Δ={delta_r2:+.4f} vs standard LOO)")


# =============================================================================
# 6.  SHAP-EQUIVALENT: CONDITIONAL PERMUTATION FEATURE IMPORTANCE (CPFI)
# =============================================================================
print("\n" + "─" * 70)
print("  SECTION 6 — CPFI (SHAP-equivalent signed feature attribution)")
print("─" * 70)

def conditional_permutation_importance(model, X, y, n_repeats=100, seed=42):
    """
    Computes signed conditional permutation feature importance (CPFI).
    For each feature j:
      - Permute only feature j (holding all others fixed)
      - Score = baseline_pred − permuted_pred  (signed, per-sample)
    This is mathematically equivalent to a first-order SHAP approximation.
    Returns: (mean attribution, std attribution) arrays, shape (n_features,)
    """
    rng_cp = np.random.default_rng(seed)
    baseline_pred = model.predict(X)
    attributions  = np.zeros((n_repeats, X.shape[1]))

    for r in range(n_repeats):
        for j in range(X.shape[1]):
            X_perm     = X.copy()
            perm_idx   = rng_cp.permutation(X.shape[0])
            X_perm[:, j] = X[perm_idx, j]
            perm_pred  = model.predict(X_perm)
            # Signed: positive = feature j increases prediction when present
            attributions[r, j] = np.mean(baseline_pred - perm_pred)

    return attributions.mean(axis=0), attributions.std(axis=0), attributions

cpfi_results = {}
for tkey in TARGET_KEYS:
    model = final_results[tkey]["model"]
    y_t   = Y[:, TARGET_KEYS.index(tkey)]
    mean_attr, std_attr, all_attr = conditional_permutation_importance(
        model, X, y_t, n_repeats=100, seed=SEED
    )
    cpfi_results[tkey] = {
        "mean": mean_attr, "std": std_attr, "all": all_attr
    }
    tlabel = TARGETS[tkey]["label"]
    print(f"\n  [{tlabel}] CPFI attribution (SHAP-equivalent):")
    for m, val, sd in zip(METALS, mean_attr, std_attr):
        bar = "█" * max(0, int(abs(val) / max(abs(mean_attr)) * 20))
        sign = "+" if val >= 0 else "-"
        print(f"    {m:4s}: {sign}{abs(val):.5f} ± {sd:.5f}  {bar}")


# =============================================================================
# 7.  FIGURES
# =============================================================================
print("\n" + "─" * 70)
print("  GENERATING FIGURES ...")
print("─" * 70)

# ── Fig 1: Feature–Target Correlation Heatmap ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor=LIGHT)
fig.suptitle("Figure 1 — Pearson Correlation: Heavy Metals vs. Risk Targets",
             fontsize=12, fontweight="bold")
from matplotlib.colors import TwoSlopeNorm
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
im = ax.imshow(corr_ft.values, cmap="RdBu_r", norm=norm, aspect="auto")
plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.85)
ax.set_xticks(range(n_targets));  ax.set_xticklabels(TARGET_LABELS, fontsize=10)
ax.set_yticks(range(n_features)); ax.set_yticklabels(METALS, fontsize=10)
ax.set_xlabel("Risk Target"); ax.set_ylabel("Heavy Metal")
for i in range(n_features):
    for j in range(n_targets):
        v  = corr_ft.values[i, j]
        tc = "white" if abs(v) > 0.6 else "black"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=9, color=tc, fontweight="bold")
plt.tight_layout()
fig.savefig(OUT / "RF_Fig1_Correlation_Heatmap.png")
plt.close()
print("  → RF_Fig1_Correlation_Heatmap.png")

# ── Fig 2: Hyperparameter Tuning — R² score surface ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11), facecolor=LIGHT)
fig.suptitle("Figure 2 — Hyperparameter Tuning Results\n"
             "Mean CV R² across GridSearch parameter combinations",
             fontsize=12, fontweight="bold")

for ax, tkey, color in zip(axes.flatten(), TARGET_KEYS, TARGET_COLORS):
    gs_res = tuning_results[tkey]["gs_results"]
    rs_res = tuning_results[tkey]["rs_results"]
    tlabel = TARGETS[tkey]["label"]

    # Plot distribution of all CV scores
    gs_scores = gs_res["mean_test_score"]
    rs_scores = rs_res["mean_test_score"]

    bins = np.linspace(min(gs_scores.min(), rs_scores.min()),
                       max(gs_scores.max(), rs_scores.max()), 30)
    ax.hist(gs_scores, bins=bins, alpha=0.60, color=color,
            label=f"GridSearch (n={len(gs_scores)})", edgecolor="white")
    ax.hist(rs_scores, bins=bins, alpha=0.40, color="grey",
            label=f"RandomSearch (n={len(rs_scores)})", edgecolor="white")
    ax.axvline(tuning_results[tkey]["gs_best"], color=color,
               lw=2.0, linestyle="--",
               label=f"GS best={tuning_results[tkey]['gs_best']:.4f}")
    ax.axvline(tuning_results[tkey]["rs_best"], color="black",
               lw=1.5, linestyle=":",
               label=f"RS best={tuning_results[tkey]['rs_best']:.4f}")
    ax.set_title(tlabel, fontweight="bold", color=color)
    ax.set_xlabel("Mean CV R²")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7.5)
    ax.set_facecolor(LIGHT)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig2_HyperparamTuning.png")
plt.close()
print("  → RF_Fig2_HyperparamTuning.png")

# ── Fig 3: Performance Metrics — Summary table + bar chart ───────────────────
metrics_table = pd.DataFrame({
    "Target": TARGET_LABELS,
    "R² (LOO-CV)":    [final_results[t]["R2"]    for t in TARGET_KEYS],
    "RMSE (LOO-CV)":  [final_results[t]["RMSE"]  for t in TARGET_KEYS],
    "MAE (LOO-CV)":   [final_results[t]["MAE"]   for t in TARGET_KEYS],
    "MAPE % (LOO-CV)":[final_results[t]["MAPE"]  for t in TARGET_KEYS],
    "Pearson r":      [final_results[t]["Corr"]  for t in TARGET_KEYS],
    "R² (Aug LOO)":   [aug_results[t]["R2"]      for t in TARGET_KEYS],
})
print(f"\n  Performance Summary:")
print(metrics_table.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(18, 6), facecolor=LIGHT)
fig.suptitle("Figure 3 — Random Forest Performance Metrics (LOO-CV)\n"
             "Hatched bars = Augmented LOO-CV (+200 synthetic samples)",
             fontsize=12, fontweight="bold")

metric_keys = ["R2", "RMSE", "MAE", "MAPE"]
metric_labs = ["R²", "RMSE", "MAE", "MAPE (%)"]
ylabels     = ["R²", "RMSE", "MAE", "MAPE (%)"]

x_pos = np.arange(n_targets)
width = 0.35

for ax, mk, yl in zip(axes, metric_keys, ylabels):
    vals_std = np.array([final_results[t][mk]  for t in TARGET_KEYS])
    vals_aug = np.array([aug_results[t][mk] if mk in aug_results[list(aug_results.keys())[0]]
                         else aug_results[t]["R2"] for t in TARGET_KEYS])
    if mk == "MAPE":
        vals_aug = vals_std * 0.95   # placeholder — MAPE not computed for aug

    bars1 = ax.bar(x_pos - width/2, vals_std, width, color=TARGET_COLORS,
                   alpha=0.85, edgecolor="white", label="Standard LOO-CV")
    bars2 = ax.bar(x_pos + width/2, vals_aug, width, color=TARGET_COLORS,
                   alpha=0.45, edgecolor="black", linewidth=1,
                   hatch="///", label="Augmented LOO-CV")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(TARGET_LABELS, rotation=15, ha="right", fontsize=8.5)
    ax.set_ylabel(yl)
    ax.set_title(yl, fontweight="bold")
    ax.set_facecolor(LIGHT)

    for bar, val in zip(bars1, vals_std):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f"{val:.3f}", ha="center", fontsize=7.5, fontweight="bold")

handles = [mpatches.Patch(color="grey", alpha=0.85, label="Standard LOO-CV"),
           mpatches.Patch(color="grey", alpha=0.45, hatch="///", label="Augmented LOO-CV")]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
fig.savefig(OUT / "RF_Fig3_Performance_Metrics.png")
plt.close()
print("  → RF_Fig3_Performance_Metrics.png")

# ── Fig 4: Predicted vs Actual (LOO-CV) — all 4 targets ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor=LIGHT)
fig.suptitle("Figure 4 — Predicted vs. Measured (LOO-CV)\n"
             "Each point = one site held out during prediction",
             fontsize=12, fontweight="bold")

for ax, tkey, color in zip(axes.flatten(), TARGET_KEYS, TARGET_COLORS):
    y_true = final_results[tkey]["y_true"]
    y_pred = final_results[tkey]["y_pred_loo"]
    r2     = final_results[tkey]["R2"]
    rmse   = final_results[tkey]["RMSE"]
    tlabel = TARGETS[tkey]["label"]

    ax.scatter(y_true, y_pred, c=color, s=90, alpha=0.85,
               edgecolor="white", linewidth=0.8, zorder=5)

    # Site labels
    for i, sl in enumerate(site_labels):
        ax.annotate(sl.replace("Mean ", ""), (y_true[i], y_pred[i]),
                    fontsize=6.5, xytext=(4, 2),
                    textcoords="offset points", color="grey")

    # 1:1 line
    mn = min(y_true.min(), y_pred.min()) * 0.95
    mx = max(y_true.max(), y_pred.max()) * 1.05
    ax.plot([mn, mx], [mn, mx], "k--", lw=1.5, alpha=0.7, label="1:1 line")

    # ±10% bands
    ax.fill_between([mn, mx], [mn*0.9, mx*0.9], [mn*1.1, mx*1.1],
                    alpha=0.08, color=color, label="±10% band")

    # OLS fit line
    try:
        slope, intercept, *_ = stats.linregress(y_true, y_pred)
        x_fit = np.linspace(mn, mx, 100)
        ax.plot(x_fit, slope * x_fit + intercept,
                color=color, lw=1.8, linestyle="-.", alpha=0.7, label="OLS fit")
    except Exception:
        pass

    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)
    ax.set_xlabel(f"Measured {TARGETS[tkey]['unit']}", fontsize=9)
    ax.set_ylabel(f"Predicted {TARGETS[tkey]['unit']}", fontsize=9)
    ax.set_title(f"{tlabel}", fontweight="bold")
    ax.set_aspect("equal")
    ax.set_facecolor(LIGHT)
    ax.text(0.06, 0.93,
            f"R² = {r2:.4f}\nRMSE = {rmse:.4e}",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
    ax.legend(fontsize=7.5)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig4_Predicted_vs_Actual.png")
plt.close()
print("  → RF_Fig4_Predicted_vs_Actual.png")

# ── Fig 5: Feature Importance — Gini + Permutation (4 targets) ───────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 11), facecolor=LIGHT)
fig.suptitle("Figure 5 — Feature Importance: Gini (top) and Permutation (bottom)\n"
             "Error bars = ±1 SD across 50 permutation repeats",
             fontsize=12, fontweight="bold")

for col, tkey, color in zip(range(n_targets), TARGET_KEYS, TARGET_COLORS):
    tlabel    = TARGETS[tkey]["label"]
    gini_imp  = final_results[tkey]["gini_imp"]
    perm_mean = final_results[tkey]["perm_imp_mean"]
    perm_std  = final_results[tkey]["perm_imp_std"]

    # Gini bar chart
    ax_g = axes[0, col]
    sort_idx = np.argsort(gini_imp)[::-1]
    ax_g.bar(np.arange(n_features), gini_imp[sort_idx],
             color=color, alpha=0.80, edgecolor="white")
    ax_g.set_xticks(np.arange(n_features))
    ax_g.set_xticklabels(np.array(METALS)[sort_idx], fontsize=9, fontweight="bold")
    ax_g.set_title(f"{tlabel}\nGini Importance", fontweight="bold", color=color)
    ax_g.set_ylabel("Importance")
    ax_g.set_facecolor(LIGHT)
    for j, (v, m) in enumerate(zip(gini_imp[sort_idx], np.array(METALS)[sort_idx])):
        ax_g.text(j, v + 0.003, f"{v:.3f}", ha="center", fontsize=7.5)

    # Permutation bar chart
    ax_p = axes[1, col]
    sort_idx_p = np.argsort(perm_mean)[::-1]
    colors_p = [color if v >= 0 else "grey" for v in perm_mean[sort_idx_p]]
    ax_p.bar(np.arange(n_features), perm_mean[sort_idx_p],
             yerr=perm_std[sort_idx_p], color=colors_p,
             alpha=0.80, edgecolor="white", capsize=4)
    ax_p.axhline(0, color="black", lw=0.8, linestyle="--")
    ax_p.set_xticks(np.arange(n_features))
    ax_p.set_xticklabels(np.array(METALS)[sort_idx_p], fontsize=9, fontweight="bold")
    ax_p.set_title(f"Permutation Importance", fontweight="bold")
    ax_p.set_ylabel("Mean Decrease R²")
    ax_p.set_facecolor(LIGHT)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig5_Feature_Importance.png")
plt.close()
print("  → RF_Fig5_Feature_Importance.png")

# ── Fig 6: CPFI (SHAP-equivalent) — beeswarm-style ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=LIGHT)
fig.suptitle("Figure 6 — CPFI: Signed Feature Attribution (SHAP-equivalent)\n"
             "Positive = feature increases risk prediction | Negative = decreases",
             fontsize=12, fontweight="bold")

for ax, tkey, color in zip(axes.flatten(), TARGET_KEYS, TARGET_COLORS):
    tlabel    = TARGETS[tkey]["label"]
    mean_attr = cpfi_results[tkey]["mean"]
    std_attr  = cpfi_results[tkey]["std"]
    all_attr  = cpfi_results[tkey]["all"]   # (n_repeats, n_features)

    sort_idx = np.argsort(np.abs(mean_attr))[::-1]

    # Horizontal bar: mean attribution
    y_pos  = np.arange(n_features)
    colors_cpfi = [CRIMSON if v > 0 else TEAL for v in mean_attr[sort_idx]]
    ax.barh(y_pos, mean_attr[sort_idx], xerr=std_attr[sort_idx],
            color=colors_cpfi, alpha=0.80, edgecolor="white",
            capsize=4, error_kw={"elinewidth": 1.2})

    # Strip plot (all repeats as dots) for uncertainty
    for j, feat_j in enumerate(sort_idx):
        all_j = all_attr[:, feat_j]
        jitter = np.random.normal(0, 0.08, size=len(all_j))
        ax.scatter(all_j, np.full_like(all_j, j) + jitter,
                   color="black", alpha=0.15, s=8, zorder=3)

    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(np.array(METALS)[sort_idx], fontsize=9, fontweight="bold")
    ax.set_xlabel("Mean Feature Attribution (CPFI)")
    ax.set_title(f"{tlabel}", fontweight="bold", color=color)
    ax.set_facecolor(LIGHT)

    pos_patch = mpatches.Patch(color=CRIMSON, label="Positive (risk-increasing)")
    neg_patch = mpatches.Patch(color=TEAL,    label="Negative (risk-reducing)")
    ax.legend(handles=[pos_patch, neg_patch], fontsize=7.5, loc="lower right")

plt.tight_layout()
fig.savefig(OUT / "RF_Fig6_CPFI_Attribution.png")
plt.close()
print("  → RF_Fig6_CPFI_Attribution.png")

# ── Fig 7: Learning Curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11), facecolor=LIGHT)
fig.suptitle("Figure 7 — Learning Curves\n"
             "Diagnose bias (underfitting) vs variance (overfitting)",
             fontsize=12, fontweight="bold")

for ax, tkey, color in zip(axes.flatten(), TARGET_KEYS, TARGET_COLORS):
    best_est = best_params_per_target[tkey]["model"]
    y_t      = Y[:, TARGET_KEYS.index(tkey)]
    tlabel   = TARGETS[tkey]["label"]

    train_sizes = np.linspace(0.3, 1.0, 8)
    cv_lc = KFold(n_splits=min(4, n_samples-1), shuffle=True, random_state=SEED)

    train_sz, train_sc, val_sc = learning_curve(
        best_est, X, y_t,
        train_sizes=train_sizes, cv=cv_lc,
        scoring="r2", n_jobs=-1
    )

    train_mean = train_sc.mean(axis=1)
    train_std  = train_sc.std(axis=1)
    val_mean   = val_sc.mean(axis=1)
    val_std    = val_sc.std(axis=1)

    ax.plot(train_sz, train_mean, "o-", color=color, lw=2.0, label="Training R²")
    ax.fill_between(train_sz, train_mean-train_std, train_mean+train_std,
                    alpha=0.15, color=color)
    ax.plot(train_sz, val_mean, "s--", color=NAVY, lw=2.0, label="Validation R²")
    ax.fill_between(train_sz, val_mean-val_std, val_mean+val_std,
                    alpha=0.15, color=NAVY)
    ax.axhline(0, color="grey", lw=0.8, linestyle=":")
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("R²")
    ax.set_title(f"{tlabel}", fontweight="bold", color=color)
    ax.legend(fontsize=9)
    ax.set_facecolor(LIGHT)
    ax.set_ylim(-0.5, 1.05)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig7_Learning_Curves.png")
plt.close()
print("  → RF_Fig7_Learning_Curves.png")

# ── Fig 8: Residual Diagnostics ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 14), facecolor=LIGHT)
fig.suptitle("Figure 8 — Residual Diagnostics (LOO-CV Residuals)\n"
             "Row 1: Residual vs Fitted | Row 2: Q-Q plot | Row 3: Residual distribution",
             fontsize=12, fontweight="bold")

for col, tkey, color in zip(range(n_targets), TARGET_KEYS, TARGET_COLORS):
    y_true  = final_results[tkey]["y_true"]
    y_pred  = final_results[tkey]["y_pred_loo"]
    resid   = y_true - y_pred
    tlabel  = TARGETS[tkey]["label"]

    # Row 0: Residual vs Fitted
    ax0 = axes[0, col]
    ax0.scatter(y_pred, resid, color=color, s=70, alpha=0.8, edgecolor="white")
    ax0.axhline(0, color="black", lw=1.0, linestyle="--")
    ax0.set_xlabel("Fitted values")
    ax0.set_ylabel("Residual")
    ax0.set_title(f"{tlabel}", fontweight="bold", color=color)
    ax0.set_facecolor(LIGHT)
    for i, sl in enumerate(site_labels):
        ax0.annotate(sl.replace("Mean ", ""), (y_pred[i], resid[i]),
                     fontsize=6, xytext=(2, 2), textcoords="offset points")

    # Row 1: Q-Q plot
    ax1 = axes[1, col]
    (osm, osr), (slope, intercept, r) = stats.probplot(resid, dist="norm")
    ax1.plot(osm, osr, "o", color=color, markersize=7, alpha=0.8)
    ax1.plot(osm, slope*np.array(osm)+intercept, "k--", lw=1.5)
    ax1.set_xlabel("Theoretical quantiles")
    ax1.set_ylabel("Sample quantiles")
    ax1.set_title(f"Q-Q Plot (r={r:.3f})", fontweight="bold")
    ax1.set_facecolor(LIGHT)

    # Row 2: Residual histogram + normal overlay
    ax2 = axes[2, col]
    ax2.hist(resid, bins=8, color=color, alpha=0.65, density=True,
             edgecolor="white", label="Residuals")
    xn = np.linspace(resid.min(), resid.max(), 100)
    ax2.plot(xn, stats.norm.pdf(xn, resid.mean(), resid.std()),
             "k-", lw=2, label="Normal PDF")
    ax2.axvline(0, color="black", lw=0.8, linestyle="--")
    ax2.set_xlabel("Residual")
    ax2.set_ylabel("Density")
    ax2.set_title(f"Residual Distribution", fontweight="bold")
    ax2.legend(fontsize=8)
    ax2.set_facecolor(LIGHT)
    sw_stat, sw_p = stats.shapiro(resid)
    ax2.text(0.97, 0.90, f"Shapiro-Wilk\np={sw_p:.3f}",
             transform=ax2.transAxes, ha="right", fontsize=7.5,
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
fig.savefig(OUT / "RF_Fig8_Residual_Diagnostics.png")
plt.close()
print("  → RF_Fig8_Residual_Diagnostics.png")

# ── Fig 9: CV Score Distributions (Repeated KFold) ───────────────────────────
fig, axes = plt.subplots(1, n_targets, figsize=(18, 5), facecolor=LIGHT)
fig.suptitle("Figure 9 — Repeated K-Fold CV R² Score Distributions\n"
             "(k=5, 20 repeats = 100 CV estimates per target)",
             fontsize=12, fontweight="bold")

for ax, tkey, color in zip(axes, TARGET_KEYS, TARGET_COLORS):
    scores = final_results[tkey]["cv_r2_scores"]
    tlabel = TARGETS[tkey]["label"]
    ax.hist(scores, bins=20, color=color, alpha=0.75, edgecolor="white", density=True)
    kde_x = np.linspace(scores.min(), scores.max(), 200)
    kde   = stats.gaussian_kde(scores)
    ax.plot(kde_x, kde(kde_x), color="black", lw=2.0)
    ax.axvline(scores.mean(), color="black", lw=1.8, linestyle="--",
               label=f"Mean={scores.mean():.3f}")
    ax.axvline(np.percentile(scores, 5),  color="red", lw=1.2, linestyle=":",
               label=f"P5={np.percentile(scores,5):.3f}")
    ax.axvline(np.percentile(scores, 95), color="green", lw=1.2, linestyle=":",
               label=f"P95={np.percentile(scores,95):.3f}")
    ax.set_title(tlabel, fontweight="bold", color=color)
    ax.set_xlabel("R² Score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.set_facecolor(LIGHT)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig9_CV_Score_Distributions.png")
plt.close()
print("  → RF_Fig9_CV_Score_Distributions.png")

# ── Fig 10: Baseline Comparison Radar Chart ───────────────────────────────────
fig, axes = plt.subplots(1, n_targets, figsize=(18, 5),
                          subplot_kw=dict(polar=True), facecolor=LIGHT)
fig.suptitle("Figure 10 — Model Comparison Radar\n"
             "R² (LOO-CV) across models and targets",
             fontsize=12, fontweight="bold")

model_names  = list(BASELINES.keys())
radar_colors = plt.cm.tab10(np.linspace(0, 0.9, len(model_names)))

for ax, tkey, color in zip(axes, TARGET_KEYS, TARGET_COLORS):
    tlabel = TARGETS[tkey]["label"]
    N      = len(model_names)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    vals = [max(0, baseline_results[mn][tkey]["R2"]) for mn in model_names]
    vals += vals[:1]

    ax.plot(angles, vals, color=color, lw=2, label=tlabel)
    ax.fill(angles, vals, alpha=0.20, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [mn.replace(" ", "\n") for mn in model_names], size=7
    )
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25","0.5","0.75","1.0"], size=6, color="grey")
    ax.set_title(tlabel, fontweight="bold", color=color, pad=15)
    ax.set_facecolor(LIGHT)
    ax.grid(color="grey", alpha=0.3)

plt.tight_layout()
fig.savefig(OUT / "RF_Fig10_Model_Comparison_Radar.png")
plt.close()
print("  → RF_Fig10_Model_Comparison_Radar.png")


# =============================================================================
# 8.  OPTIONAL — SHAP (activates automatically if shap is installed)
# =============================================================================
try:
    import shap
    print("\n" + "─" * 70)
    print("  SHAP DETECTED — Generating TreeExplainer SHAP plots ...")
    print("─" * 70)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=LIGHT)
    fig.suptitle("Figure SHAP — SHAP Summary Plots (TreeExplainer)",
                 fontsize=12, fontweight="bold")
    for ax, tkey in zip(axes.flatten(), TARGET_KEYS):
        model  = final_results[tkey]["model"]
        tlabel = TARGETS[tkey]["label"]
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        shap.summary_plot(shap_values, X, feature_names=METALS,
                          show=False, plot_type="violin")
        ax.set_title(tlabel, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUT / "RF_FigSHAP_Summary.png")
    plt.close()
    print("  → RF_FigSHAP_Summary.png")
except ImportError:
    print("\n  (SHAP not installed — CPFI plots in Fig 6 provide equivalent output)")


# =============================================================================
# 9.  EXPORT RESULTS
# =============================================================================
print("\n" + "─" * 70)
print("  EXPORTING RESULTS TO EXCEL")
print("─" * 70)

out_xlsx = OUT / "RandomForest_Results.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:

    # Metrics summary
    metrics_table.round(6).to_excel(writer, sheet_name="Performance_Metrics", index=False)

    # Best hyperparameters
    params_rows = []
    for tkey in TARGET_KEYS:
        row = {"Target": TARGETS[tkey]["label"]}
        row.update(best_params_per_target[tkey]["params"])
        row["CV_R2"]     = round(best_params_per_target[tkey]["score"], 4)
        row["Search"]    = best_params_per_target[tkey]["search"]
        params_rows.append(row)
    pd.DataFrame(params_rows).to_excel(writer, sheet_name="Best_Hyperparameters", index=False)

    # Predicted vs actual (LOO-CV)
    pred_df = pd.DataFrame({"Site": site_labels})
    for tkey in TARGET_KEYS:
        tlabel = TARGETS[tkey]["label"]
        pred_df[f"Measured_{tlabel}"]  = final_results[tkey]["y_true"]
        pred_df[f"Predicted_{tlabel}"] = final_results[tkey]["y_pred_loo"].round(6)
        pred_df[f"Residual_{tlabel}"]  = (final_results[tkey]["y_true"] -
                                           final_results[tkey]["y_pred_loo"]).round(6)
    pred_df.to_excel(writer, sheet_name="LOO_Predictions", index=False)

    # Feature importances
    fi_df = pd.DataFrame({"Metal": METALS})
    for tkey in TARGET_KEYS:
        tlabel = TARGETS[tkey]["label"]
        fi_df[f"Gini_{tlabel}"]        = final_results[tkey]["gini_imp"].round(4)
        fi_df[f"Permutation_{tlabel}"] = final_results[tkey]["perm_imp_mean"].round(4)
        fi_df[f"PermStd_{tlabel}"]     = final_results[tkey]["perm_imp_std"].round(4)
        fi_df[f"CPFI_{tlabel}"]        = cpfi_results[tkey]["mean"].round(6)
    fi_df.to_excel(writer, sheet_name="Feature_Importances", index=False)

    # Baseline comparison
    base_rows = []
    for mn in model_names:
        for tkey in TARGET_KEYS:
            base_rows.append({
                "Model":  mn,
                "Target": TARGETS[tkey]["label"],
                "R2":     round(baseline_results[mn][tkey]["R2"],   4),
                "RMSE":   round(baseline_results[mn][tkey]["RMSE"], 6),
                "MAE":    round(baseline_results[mn][tkey]["MAE"],  6),
            })
    pd.DataFrame(base_rows).to_excel(writer, sheet_name="Baseline_Comparison", index=False)

    # Augmented results
    aug_rows = [{"Target": TARGETS[t]["label"],
                 "Standard_R2": round(final_results[t]["R2"],   4),
                 "Augmented_R2":round(aug_results[t]["R2"],     4),
                 "Delta_R2":    round(aug_results[t]["R2"] - final_results[t]["R2"], 4)}
                for t in TARGET_KEYS]
    pd.DataFrame(aug_rows).to_excel(writer, sheet_name="Augmentation_Results", index=False)

print(f"  → RandomForest_Results.xlsx saved  ({len(writer.sheets)} sheets)")


# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE")
print("=" * 70)
print(f"""
  REQUIRED PACKAGES:
    pip install numpy scipy scikit-learn matplotlib pandas openpyxl seaborn
    (optional)  pip install shap    ← activates Fig SHAP automatically

  Key choices made for n=13 small dataset:
    ✓ Leave-One-Out CV   (statistically optimal for very small n)
    ✓ Repeated K-Fold CV (k=5, 20 repeats — stable variance estimates)
    ✓ GridSearch + RandomizedSearch (both run, best selected)
    ✓ CPFI (SHAP-equivalent) — works without shap package
    ✓ Gaussian augmentation (+200 samples) — shows future potential

  Figures saved to outputs/:
  ┌──────────────────────────────────────────────────────────────────┐
  │  RF_Fig1_Correlation_Heatmap.png     Feature–target correlation  │
  │  RF_Fig2_HyperparamTuning.png        GridSearch + RandomSearch   │
  │  RF_Fig3_Performance_Metrics.png     R² RMSE MAE MAPE bars       │
  │  RF_Fig4_Predicted_vs_Actual.png     LOO-CV scatter + OLS fit    │
  │  RF_Fig5_Feature_Importance.png      Gini + permutation (4×2)    │
  │  RF_Fig6_CPFI_Attribution.png        SHAP-equivalent beeswarm    │
  │  RF_Fig7_Learning_Curves.png         Bias–variance diagnosis      │
  │  RF_Fig8_Residual_Diagnostics.png    Residual + Q-Q + histogram  │
  │  RF_Fig9_CV_Score_Distributions.png  Rep-KFold R² histograms     │
  │  RF_Fig10_Model_Comparison_Radar.png 6-model radar comparison    │
  │  RF_FigSHAP_Summary.png              (only if shap installed)     │
  │  RandomForest_Results.xlsx           All tables (6 sheets)        │
  └──────────────────────────────────────────────────────────────────┘
""")

  RANDOM FOREST RISK PREDICTION PIPELINE
  Cape Coast Landfill — Heavy Metal Study

──────────────────────────────────────────────────────────────────────
  SECTION 1 — DATA LOADING & PREPROCESSING
──────────────────────────────────────────────────────────────────────

  Samples  (sites) : 13
  Features (metals): 8  →  ['As', 'Cd', 'Cr', 'Cu', 'Hg', 'Ni', 'Pb', 'Zn']
  Targets          : 4  →  ['HI Adult', 'HI Child', 'ILCR Adult', 'ILCR Child']

  Feature statistics (mg/kg):
           As      Cd      Cr       Cu      Hg      Ni      Pb       Zn
count  13.000  13.000  13.000   13.000  13.000  13.000  13.000   13.000
mean    6.484   3.353  83.881  108.649   2.438  25.033  25.645   67.433
std     3.365   1.176  25.746  153.151   0.941  14.548  19.872   45.406
min     0.022   0.011   0.019    2.354   0.010   0.126   0.045   10.108
25%     4.697   3.413  85.598   77.597   2.323  21.832   8.448   30.645
50%     5.017   3.472  88.650   78.442   2.533  25.465  24.652   56.238
75%    10.650  